In [ ]:
import sys, os, importlib, subprocess, json
# pip installs if missing (Kaggle already has them, kept for safety)
for pkg in ["torch", "pandas", "numpy", "sklearn"]:
    try:
        importlib.import_module(pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import torch, numpy as np, pandas as pd
import os, pathlib, shutil, sys
WROOT = pathlib.Path('/kaggle/working/wearfusion')
if WROOT.exists():
    shutil.rmtree(WROOT)
WROOT.mkdir(parents=True, exist_ok=True)
(WROOT / '__init__.py').write_text('')
SRC = {
    "config": "\"\"\"Central configuration for the WEAR fusion pipeline.\n\nSupports running either inside a Kaggle notebook (data mounted under\n``/kaggle/input/...``) or locally with a copy of the data directory.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n# ---------------------------------------------------------------------------\n# Constants / data schema\n# ---------------------------------------------------------------------------\nCOMPETITION_NAME = \"3rd-wear-dataset-challenge-hasca-2026\"\nCOMPETITION_DIR = Path(\"/kaggle/input/competitions\") / COMPETITION_NAME\n\nSENSOR_LOCATIONS = [\"right_arm\", \"right_leg\", \"left_leg\", \"left_arm\"]\nSENSOR_TO_ID = {s: i for i, s in enumerate(SENSOR_LOCATIONS)}\nN_SENSORS = len(SENSOR_LOCATIONS)\nN_AXES = 3\n\n# Test metadata uses these location strings.\nTEST_SENSOR_LOCATIONS = SENSOR_LOCATIONS\n\nN_CLASSES = 19\nCLASS_NAMES = [\n    \"null\",\n    \"jogging\",\n    \"jogging (rotating arms)\",\n    \"jogging (skipping)\",\n    \"jogging (sidesteps)\",\n    \"jogging (butt-kicks)\",\n    \"stretching (triceps)\",\n    \"stretching (lunging)\",\n    \"stretching (shoulders)\",\n    \"stretching (hamstrings)\",\n    \"stretching (lumbar rotation)\",\n    \"push-ups\",\n    \"push-ups (complex)\",\n    \"sit-ups\",\n    \"sit-ups (complex)\",\n    \"burpees\",\n    \"lunges\",\n    \"lunges (complex)\",\n    \"bench-dips\",\n]\nLABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}\n\n# Modality shapes (per 1-second window).\nINERTIAL_HZ = 50\nVIDEO_FPS = 30\nINERTIAL_WINDOW = 50          # 50 samples @ 50 Hz\nVIDEO_WINDOW = 15             # frames used in test features\nVIDEO_DIM = 768               # VideoMAEv2-Base feature dim\n\n# For training, video features are extracted at 30 fps while inertial at 50 Hz.\n# The 15 usable video frames map to inertial time as (see baseline): the npy\n# row index `i` corresponds to clip centred on frame `i`; we use rows\n# video_start+8 .. video_start+22 (15 rows) to avoid edge leakage.\nVIDEO_FRAME_OFFSET = 8\n\n\n# ---------------------------------------------------------------------------\n# Runtime environment helpers\n# ---------------------------------------------------------------------------\ndef is_kaggle() -> bool:\n    return os.path.exists(\"/kaggle/input\")\n\n\n@dataclass\nclass Paths:\n    \"\"\"Resolve data/output paths for the current runtime.\"\"\"\n\n    competition_dir: Path = field(\n        default_factory=lambda: COMPETITION_DIR if is_kaggle() else Path(\"data\")\n    )\n    output_dir: Path = field(\n        default_factory=lambda: Path(\"/kaggle/working\" if is_kaggle() else \"runs\")\n    )\n\n    @property\n    def train_inertial_dir(self) -> Path:\n        return self.competition_dir / \"train\" / \"inertial_feat\"\n\n    @property\n    def train_video_dir(self) -> Path:\n        return self.competition_dir / \"train\" / \"videomae_feat\"\n\n    @property\n    def test_dir(self) -> Path:\n        return self.competition_dir / \"test\"\n\n    @property\n    def test_inertial(self) -> Path:\n        return self.test_dir / \"test_inertial_data.npy\"\n\n    @property\n    def test_video(self) -> Path:\n        return self.test_dir / \"test_videomae_data.npy\"\n\n    @property\n    def test_meta(self) -> Path:\n        return self.test_dir / \"test_meta_data.csv\"\n\n    @property\n    def sample_submission(self) -> Path:\n        return self.competition_dir / \"sample_submission.csv\"\n\n\n# ---------------------------------------------------------------------------\n# Training configuration\n# ---------------------------------------------------------------------------\n@dataclass\nclass DataConfig:\n    \"\"\"Windowing + sampling configuration.\"\"\"\n\n    window_stride: int = 25          # inertial-sample stride between windows\n    max_windows_per_class_per_subject: int = 600  # cap for compute (None = all)\n    seed: int = 42\n    # Training-time augmentation knobs.\n    augment_scale: float = 0.15      # uniform(1-s, 1+s) amplitude scaling\n    augment_noise: float = 0.02      # gaussian noise std (relative)\n    augment_shift: int = 2           # max random time shift (samples)\n    single_sensor_prob: float = 0.6  # prob of presenting a single random sensor\n                                     # during training (mirrors the 1-sensor test)\n\n\n@dataclass\nclass ModelConfig:\n    \"\"\"Architecture knobs for the fusion model.\"\"\"\n\n    name: str = \"wear_fusion\"\n    inertial_hidden: int = 192\n    inertial_blocks: int = 3            # Inception blocks in the inertial encoder\n    inertial_channels: int = 128\n    video_hidden: int = 192\n    video_transformer_layers: int = 2\n    video_heads: int = 4\n    sensor_embed_dim: int = 16\n    classifier_hidden: int = 256\n    dropout: float = 0.2\n    modality_dropout: float = 0.1\n    scale: float = 1.0                 # global width multiplier (applied in model)\n\n\n@dataclass\nclass TrainConfig:\n    \"\"\"Training loop hyper-parameters.\"\"\"\n\n    epochs: int = 6\n    batch_size: int = 256\n    lr: float = 8e-4\n    weight_decay: float = 1e-3\n    label_smoothing: float = 0.05\n    warmup_epochs: int = 1\n    num_workers: int = 4\n    seed: int = 42\n    amp: bool = True\n    accumulate_grad_steps: int = 1\n    class_weighting: str = \"none\"   # \"none\" | \"inverse\" | \"sqrt_inverse\"\n    max_grad_norm: float = 5.0\n\n\n@dataclass\nclass EvalConfig:\n    \"\"\"Ensemble / threshold tuning knobs.\"\"\"\n\n    null_bias: float = 0.75            # multiplicative boost on null logit at inference\n    n_val_subjects: int = 4            # subjects held out per grouped CV fold\n    blend_weight: float = 1.0          # reserved\n\n\n@dataclass\nclass ReservoirSpec:\n    \"\"\"One echo-state reservoir configuration.\"\"\"\n\n    state_size: int = 384\n    spectral_radius: float = 0.85\n    leak: float = 0.35\n    sparsity: float = 0.03\n    washout: int = 5\n\n\n@dataclass\nclass ReservoirConfig:\n    \"\"\"Reservoir / CNN / readout / video / fusion / TTA configuration.\"\"\"\n\n    # CNN front-end\n    cnn_channels: int = 128\n    cnn_blocks: int = 3\n    cnn_proj: int = 64\n    cnn_epochs: int = 5\n    cnn_seeds: int = 2\n    # Reservoirs\n    reservoirs: tuple = (\n        (\"r1\", 384, 0.70, 0.20, 0.03),\n        (\"r2\", 384, 0.85, 0.35, 0.03),\n        (\"r3\", 512, 0.95, 0.50, 0.02),\n        (\"r4\", 512, 1.05, 0.70, 0.02),\n    )\n    washout: int = 5\n    # Readouts\n    ridge_alphas: tuple = (0.01, 0.1, 1.0, 10.0)\n    readout_boots: int = 3\n    readout_fit_limit: int = 40000   # max sensor-views used to fit ridge readouts\n    reservoir_batch: int = 2048      # batch size for reservoir feature extraction\n    # Video branch\n    video_hidden: int = 256\n    video_layers: int = 4\n    video_heads: int = 8\n    video_epochs: int = 5\n    video_seeds: int = 2\n    # TTA\n    tta_aug: tuple = (\"none\", \"x\", \"y\", \"z\", \"all\")\n    # Late fusion\n    fusion_alpha: float = 0.7          # weight on inertial branch\n    sensor_fusion: bool = False        # per-sensor alpha (only if OOF stable)\n    max_per_class: int = 300\n    stride: int = 25\n    seed: int = 42\n\n\n@dataclass\nclass Config:\n    data: DataConfig = field(default_factory=DataConfig)\n    model: ModelConfig = field(default_factory=ModelConfig)\n    train: TrainConfig = field(default_factory=TrainConfig)\n    eval: EvalConfig = field(default_factory=EvalConfig)\n    reservoir: ReservoirConfig = field(default_factory=ReservoirConfig)\n    paths: Paths = field(default_factory=Paths)\n\n    def save(self, path: Path) -> None:\n        import json\n\n        path.write_text(\n            json.dumps(\n                {\n                    \"data\": _asdict(self.data),\n                    \"model\": _asdict(self.model),\n                    \"train\": _asdict(self.train),\n                    \"eval\": _asdict(self.eval),\n                    \"reservoir\": _asdict(self.reservoir),\n                },\n                indent=2,\n            )\n        )\n\n\ndef _asdict(obj):\n    from dataclasses import asdict\n\n    return asdict(obj)",
    "data": "\"\"\"Data loading, windowing and LOSO-style grouped splitting for the WEAR data.\n\nModality alignment\n------------------\n* Inertial is recorded at 50 Hz -> a 1-second window is 50 samples.\n* Video (VideoMAE) features are 30 fps -> 15 usable frames per second.\n* We map an inertial window start ``i`` to the 15 video rows\n  ``j = i * 3 // 5`` (rows ``j .. j+15`` offset by ``VIDEO_FRAME_OFFSET``),\n  matching the reference \"TS/Emb\" pipeline so frames near window edges don't\n  leak context.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import (\n    CLASS_NAMES,\n    INERTIAL_WINDOW,\n    LABEL_TO_ID,\n    N_SENSORS,\n    SENSOR_LOCATIONS,\n    SENSOR_TO_ID,\n    VIDEO_DIM,\n    VIDEO_FRAME_OFFSET,\n    VIDEO_WINDOW,\n    Paths,\n)\n\n# Inertial CSV columns.\nINERTIAL_COLS = [f\"{loc}_acc_{ax}\" for loc in SENSOR_LOCATIONS for ax in \"xyz\"]\n# Reshape order: each sensor -> 3 axes (xyz).\nSENSOR_AXES = np.array(\n    [[f\"{loc}_acc_{ax}\" for ax in \"xyz\"] for loc in SENSOR_LOCATIONS]\n)\n\n\ndef _read_label_ids(labels: pd.Series) -> np.ndarray:\n    \"\"\"Map raw label strings/NaN to class ids (NaN -> 0 / 'null').\"\"\"\n    out = np.zeros(len(labels), dtype=np.int64)\n    seen = labels.to_numpy()\n    for i, name in enumerate(seen):\n        if isinstance(name, str) and name in LABEL_TO_ID:\n            out[i] = LABEL_TO_ID[name]\n        else:\n            out[i] = 0  # NaN / unknown -> null\n    return out\n\n\nclass Recording:\n    \"\"\"Prepared single participant recording (inertial + video + labels).\"\"\"\n\n    def __init__(self, sbj_id: int, inertial: np.ndarray, video: np.ndarray,\n                 labels: np.ndarray):\n        self.sbj_id = sbj_id\n        # inertial: (T, N_SENSORS, 3)\n        self.inertial = inertial\n        # video: (T_v, VIDEO_DIM) where T_v = T * 30 // 50 (approx)\n        self.video = video\n        # labels: (T,)\n        self.labels = labels\n\n    @property\n    def n_samples(self) -> int:\n        return self.inertial.shape[0]\n\n\ndef _load_inertial_csv(path: Path) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Load a train inertial CSV -> (inertial (T,4,3), labels (T,)).\"\"\"\n    df = pd.read_csv(path, low_memory=False, dtype={\"label\": \"string\"})\n    inertial = df[INERTIAL_COLS].to_numpy(np.float64).reshape(\n        len(df), N_SENSORS, 3\n    )\n    labels = _read_label_ids(df[\"label\"])\n    return inertial, labels\n\n\ndef _load_video_npy(path: Path) -> np.ndarray:\n    return np.load(path, mmap_mode=\"r\")\n\n\ndef load_train_recording(paths: Paths, stem: str) -> Recording:\n    \"\"\"Load one recording by filename stem, e.g. ``sbj_0`` or ``sbj_0_2``.\"\"\"\n    inertial_path = paths.train_inertial_dir / f\"{stem}.csv\"\n    video_path = paths.train_video_dir / f\"{stem}.npy\"\n\n    inertial, labels = _load_inertial_csv(inertial_path)\n    video = _load_video_npy(video_path)\n\n    sbj_id = _sbj_id_from_stem(stem)\n    return Recording(sbj_id, inertial, video, labels)\n\n\ndef _sbj_id_from_stem(stem: str) -> int:\n    import re\n\n    m = re.match(r\"sbj_(\\d+)\", stem)\n    return int(m.group(1))\n\n\ndef train_stems(paths: Paths) -> list[str]:\n    \"\"\"Return sorted train recording stems (e.g. ['sbj_0', 'sbj_0_2', ...]).\"\"\"\n    stems = [p.name[:-4] for p in paths.train_inertial_dir.glob(\"*.csv\")]\n    return sorted(stems)\n\n\ndef subject_ids_for_stems(stems: list[str]) -> dict[int, list[str]]:\n    \"\"\"Map subject id -> list of recording stems for that subject.\"\"\"\n    subj: dict[int, list[str]] = {}\n    for s in stems:\n        subj.setdefault(_sbj_id_from_stem(s), []).append(s)\n    return subj\n\n\ndef build_windows(rec: Recording, stride: int) -> pd.DataFrame:\n    \"\"\"Generate window metadata for a recording (matches the reference).\n\n    Windows are drawn *within contiguous label segments* (so a window never\n    straddles two different activities), aligned to the segment start.  This\n    mirrors the proven reference pipeline.\n\n    Returns a DataFrame with columns:\n        rec, sbj_id, target, inertial_start, video_start\n    \"\"\"\n    T = rec.n_samples\n    labels = rec.labels\n\n    # Segment boundaries: where the label changes.\n    bounds = np.flatnonzero(labels[1:] != labels[:-1]) + 1\n    seg_starts = np.concatenate(([0], bounds))\n    seg_ends = np.concatenate((bounds, [T]))\n\n    starts_all, targets_all, video_all = [], [], []\n    for s, e in zip(seg_starts, seg_ends):\n        lab = labels[s]\n        if e - s < INERTIAL_WINDOW:\n            continue\n        first = -(-s // stride) * stride          # ceil(s / stride) * stride\n        if first + INERTIAL_WINDOW > e:\n            continue\n        seg_starts_idx = np.arange(first, e - INERTIAL_WINDOW + 1, stride)\n        starts_all.append(seg_starts_idx)\n        targets_all.append(np.full(len(seg_starts_idx), lab, dtype=np.int64))\n        video_all.append((seg_starts_idx * 3 // 5 + VIDEO_FRAME_OFFSET).astype(np.int64))\n\n    if not starts_all:\n        return pd.DataFrame(columns=[\"rec\", \"sbj_id\", \"target\",\n                                     \"inertial_start\", \"video_start\"])\n\n    starts = np.concatenate(starts_all)\n    return pd.DataFrame(\n        {\n            \"rec\": np.repeat(\"\", len(starts)),\n            \"sbj_id\": np.repeat(rec.sbj_id, len(starts)),\n            \"target\": np.concatenate(targets_all),\n            \"inertial_start\": starts,\n            \"video_start\": np.concatenate(video_all),\n        }\n    )\n\n\ndef build_dataset_windows(paths: Paths, stride: int) -> pd.DataFrame:\n    \"\"\"Build the full window metadata table for all train recordings.\"\"\"\n    frames = []\n    for stem in train_stems(paths):\n        rec = load_train_recording(paths, stem)\n        w = build_windows(rec, stride)\n        w[\"rec\"] = stem\n        frames.append(w)\n    return pd.concat(frames, ignore_index=True)\n\n\nclass InertialNormalizer:\n    \"\"\"Per-(sensor, axis) z-score computed from training data.\n\n    Applied identically to train and test windows so that scaling is\n    consistent across the model's inputs (critical for cross-subject\n    generalization with unseen test subjects).\n    \"\"\"\n\n    def __init__(self, mean: np.ndarray | None = None,\n                 std: np.ndarray | None = None):\n        # mean/std shape (N_SENSORS, 3)\n        self.mean = mean\n        self.std = std\n\n    @classmethod\n    def fit(cls, paths: Paths, max_frames: int = 3_000_000,\n            stems: list[str] | None = None) -> \"InertialNormalizer\":\n        \"\"\"Compute per-(sensor,axis) mean/std from finite train samples.\n\n        If ``stems`` is given, only those recording stems are used (fold-only\n        fitting avoids normalizer leakage across grouped CV folds).\n        \"\"\"\n        if stems is None:\n            stems = train_stems(paths)\n        sums = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        sumsq = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        count = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        for stem in stems:\n            rec = load_train_recording(paths, stem)\n            data = rec.inertial.reshape(-1, N_SENSORS, 3)  # (T,4,3)\n            valid = np.isfinite(data)\n            v = np.where(valid, data, 0.0)\n            cnt = valid.sum(axis=0)\n            sums += v.sum(axis=0)\n            sumsq += (v * v).sum(axis=0)\n            count += cnt\n            if count.min() > max_frames:\n                break\n        count = np.maximum(count, 1e-8)\n        mean = sums / count\n        var = np.maximum(sumsq / count - mean * mean, 1e-8)\n        std = np.sqrt(var)\n        return cls(mean.astype(np.float32), std.astype(np.float32))\n\n    def transform(self, inertial: np.ndarray) -> np.ndarray:\n        \"\"\"inertial: (..., N_SENSORS, 3) -> standardized (in-place copy).\"\"\"\n        out = inertial.astype(np.float32, copy=True)\n        out = np.where(np.isfinite(out), out, self.mean)\n        out = (out - self.mean) / self.std\n        return out\n\n    def save(self, path: Path) -> None:\n        np.savez(path, mean=self.mean, std=self.std)\n\n    @classmethod\n    def load(cls, path: Path) -> \"InertialNormalizer\":\n        d = np.load(path)\n        return cls(d[\"mean\"], d[\"std\"])\n\n\nclass PerWindowNormalizer:\n    \"\"\"Normalise each window independently per (sensor, axis).\n\n    For every 1-second window we subtract the window's own per-axis mean and\n    divide by its per-axis std (across the 50 time samples).  This is robust to\n    arbitrary per-subject sensor scale/offset (calibration differences), which\n    is likely the source of the train/test inertial shift for the unseen test\n    subjects.  Needs no statistics from the unseen subjects.\n    \"\"\"\n\n    def __init__(self, eps: float = 1e-4):\n        self.eps = eps\n\n    def transform(self, inertial: np.ndarray) -> np.ndarray:\n        \"\"\"inertial: (..., T, N_SENSORS, 3) or (..., T, 3).\"\"\"\n        out = inertial.astype(np.float32, copy=True)\n        out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)\n        axes = tuple(range(out.ndim - 2))       # all but (time, feat)\n        mean = out.mean(axis=-2, keepdims=True)\n        std = out.std(axis=-2, keepdims=True)\n        out = (out - mean) / (std + self.eps)\n        return out\n\n\ndef subsample_windows(windows: pd.DataFrame,\n                      max_per_class_per_subject: int | None,\n                      seed: int) -> pd.DataFrame:\n    \"\"\"Balance-ish subsample: cap windows per (sbj, class) group.\"\"\"\n    if max_per_class_per_subject is None:\n        return windows\n    df = windows.copy()\n    # Shuffle deterministically, then keep the first `max` per (sbj, class).\n    rng = np.random.RandomState(seed)\n    df[\"_r\"] = rng.rand(len(df))\n    df = df.sort_values(\"_r\")\n    df[\"_n\"] = df.groupby([\"sbj_id\", \"target\"])[\"_r\"].cumcount()\n    df = df[df[\"_n\"] < max_per_class_per_subject]\n    df = df.drop(columns=[\"_r\", \"_n\"])\n    return df.reset_index(drop=True)\n\n\nclass WearTrainDataset:\n    \"\"\"PyTorch dataset yielding (inertial (50,4,3), video (15,768), target, valid).\"\"\"\n\n    def __init__(self, windows: pd.DataFrame, paths: Paths,\n                 normalizer: InertialNormalizer | None = None,\n                 augment: bool = False, data_cfg=None):\n        self.windows = windows.reset_index(drop=True)\n        self.paths = paths\n        self.normalizer = normalizer\n        self.augment = augment\n        self.dcfg = data_cfg\n        self._rec_cache: dict[str, Recording] = {}\n\n    def _get_rec(self, rec: str) -> Recording:\n        if rec not in self._rec_cache:\n            self._rec_cache[rec] = load_train_recording(self.paths, rec)\n        return self._rec_cache[rec]\n\n    def __len__(self) -> int:\n        return len(self.windows)\n\n    def __getitem__(self, idx: int):\n        row = self.windows.iloc[idx]\n        rec = self._get_rec(row[\"rec\"])\n\n        i0 = int(row[\"inertial_start\"])\n        # Optional random time shift (keeps the window within the recording).\n        if self.augment and self.dcfg is not None and self.dcfg.augment_shift:\n            shift = int(np.random.randint(-self.dcfg.augment_shift,\n                                          self.dcfg.augment_shift + 1))\n            lo = max(0, i0 + shift)\n            hi = lo + INERTIAL_WINDOW\n            if hi > rec.n_samples:\n                hi = rec.n_samples\n                lo = hi - INERTIAL_WINDOW\n            inertial = np.asarray(rec.inertial[lo:hi],\n                                  dtype=np.float32).copy()\n            # Re-sync video start with the (possibly shifted) inertial start.\n            i0 = lo\n        else:\n            inertial = np.asarray(rec.inertial[i0:i0 + INERTIAL_WINDOW],\n                                  dtype=np.float32).copy()\n\n        # Compute sensor validity from the raw signal (before any fill).\n        valid = np.isfinite(inertial).all(axis=(0, 2))  # (4,) which sensors valid\n\n        if self.normalizer is not None:\n            inertial = self.normalizer.transform(inertial)\n\n        if self.augment and self.dcfg is not None:\n            s = 1.0 + float(np.random.uniform(-self.dcfg.augment_scale,\n                                              self.dcfg.augment_scale))\n            inertial = inertial * s\n            if self.dcfg.augment_noise:\n                inertial = inertial + np.random.randn(*inertial.shape).astype(\n                    np.float32) * self.dcfg.augment_noise\n\n        # Fill non-finite values (missing sensors) with 0 as the reference does,\n        # so conv/pool layers never receive NaN.\n        inertial = np.nan_to_num(inertial, nan=0.0, posinf=0.0, neginf=0.0)\n\n        v0 = int(row[\"video_start\"])\n        video = np.asarray(rec.video[v0:v0 + VIDEO_WINDOW],\n                           dtype=np.float32).copy()  # (15,768) writable\n\n        return {\n            \"inertial\": inertial,\n            \"video\": video,\n            \"target\": int(row[\"target\"]),\n            \"valid\": valid,\n        }\n\n\ndef build_test_dataset(paths: Paths):\n    \"\"\"Return test tensors + metadata: (inertial (N,50,3), video (N,15,768), sensor_ids (N,), ids (N,), sbj (N,)).\"\"\"\n    inertial = np.load(paths.test_inertial, mmap_mode=\"r\")  # (N,50,3) or (N,3,50)\n    # Transpose inertial if stored as (N, 3, 50) per window.\n    if inertial.ndim == 3 and inertial.shape[1] == 3 and inertial.shape[2] == 50:\n        inertial = np.transpose(inertial, (0, 2, 1))        # (N,50,3)\n    # The raw test video is stored per-window as (N, 768, 15); transpose to\n    # (N, 15, 768) to match the model's expected frame-first layout.\n    video = np.load(paths.test_video, mmap_mode=\"r\")        # (N,768,15)\n    if video.ndim == 3 and video.shape[2] == 15 and video.shape[1] == 768:\n        video = np.transpose(video, (0, 2, 1))              # (N,15,768)\n    meta = pd.read_csv(paths.test_meta)\n\n    sensor_ids = meta[\"sensor_location\"].map(SENSOR_TO_ID).to_numpy(np.int64)\n    ids = meta[\"id\"].to_numpy(np.int64)\n    sbj = meta[\"sbj_id\"].to_numpy(np.int64)\n    return inertial, video, sensor_ids, ids, sbj",
    "models": "\"\"\"Fusion model for inertial + VideoMAE features.\n\nDesign notes\n------------\n* Test windows carry a *single* inertial sensor location while training\n  carries all four.  We therefore use a *shared* per-sensor inertial encoder\n  and condition predictions on a learned sensor-location embedding, so the\n  model can make a prediction from one or many sensors.\n\n* The inertial encoder is a multi-scale 1D-Inception CNN (parallel kernels)\n  with channel attention and temporal global pooling, applied to\n  (4 channels = 3 axes + magnitude).\n\n* The video encoder projects the 768-d VideoMAE frames to a small hidden dim,\n  passes them through a lightweight Transformer, and attention-pools across\n  the 15 frames.\n\n* Fusion is a learned gated blend of inertial and video embeddings plus an\n  additive interaction term, followed by an MLP classifier (19 classes).\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\n# ---------------------------------------------------------------------------\n# Building blocks\n# ---------------------------------------------------------------------------\nclass ConvNormAct(nn.Module):\n    def __init__(self, cin: int, cout: int, kernel: int, groups: int = 1,\n                 stride: int = 1):\n        super().__init__()\n        self.conv = nn.Conv1d(cin, cout, kernel, stride=stride,\n                              padding=kernel // 2, groups=groups, bias=False)\n        self.norm = nn.GroupNorm(min(8, cout), cout)\n        self.act = nn.GELU()\n\n    def forward(self, x):\n        return self.act(self.norm(self.conv(x)))\n\n\nclass InceptionBlock(nn.Module):\n    \"\"\"Multi-scale conv block (kernels 5/11/21) with a 1x1 pooling branch.\"\"\"\n\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        # Branch width chosen so concatenation == out.\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)          # (B, bottleneck*4, T)\n        out = out + self.skip(x)\n        return self.dropout(F.gelu(self.norm(out)))\n\n\nclass ChannelAttention(nn.Module):\n    \"\"\"Squeeze-excite over channels.\"\"\"\n\n    def __init__(self, channels: int, reduction: int = 8):\n        super().__init__()\n        self.fc = nn.Sequential(\n            nn.Linear(channels, max(8, channels // reduction)),\n            nn.ReLU(inplace=True),\n            nn.Linear(max(8, channels // reduction), channels),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):  # (B,C,T)\n        w = x.mean(dim=2)  # (B,C)\n        w = self.fc(w).unsqueeze(-1)\n        return x * w\n\n\nclass InertialEncoder(nn.Module):\n    \"\"\"Shared per-sensor encoder: (B, 4, 50) -> (B, hidden).\"\"\"\n\n    def __init__(self, hidden: int, blocks: int, channels: int,\n                 dropout: float):\n        super().__init__()\n        # Input: 4 channels (x,y,z,magnitude)\n        layers = [InceptionBlock(4, channels)]\n        for _ in range(blocks - 1):\n            layers.append(InceptionBlock(channels, channels))\n        self.blocks = nn.Sequential(*layers)\n        self.attention = ChannelAttention(channels)\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B, 4, 50)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)  # (B, 4, 50)\n        h = self.blocks(x)              # (B, C, 50)\n        h = self.attention(h)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)  # (B, 2C)\n        return self.head(pooled)\n\n\nclass VideoEncoder(nn.Module):\n    \"\"\"(B, 15, 768) -> (B, hidden) with Transformer + attention pooling.\"\"\"\n\n    def __init__(self, hidden: int, layers: int, heads: int, dropout: float):\n        super().__init__()\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM),\n            nn.Linear(VIDEO_DIM, hidden),\n            nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            hidden, heads, dim_feedforward=hidden * 4, dropout=dropout,\n            activation=\"gelu\", batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(\n            layer, layers, enable_nested_tensor=False\n        )\n        self.attn = nn.Sequential(\n            nn.Linear(hidden, 64), nn.Tanh(), nn.Linear(64, 1)\n        )\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B, 15, 768)\n        h = self.projection(x) + self.position\n        h = self.transformer(h)\n        scores = self.attn(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)       # (B, hidden)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\nclass WearFusionNet(nn.Module):\n    \"\"\"Single multi-sensor fusion model.\"\"\"\n\n    def __init__(self, cfg, n_classes: int = N_CLASSES):\n        super().__init__()\n        m = cfg\n        # Apply the global width multiplier to the hidden dims.\n        s = getattr(m, \"scale\", 1.0)\n        i_hidden = int(m.inertial_hidden * s)\n        i_channels = int(m.inertial_channels * s)\n        v_hidden = int(m.video_hidden * s)\n        c_hidden = int(m.classifier_hidden * s)\n\n        self.inertial_encoder = InertialEncoder(\n            i_hidden, m.inertial_blocks, i_channels, m.dropout\n        )\n        self.video_encoder = VideoEncoder(\n            v_hidden, m.video_transformer_layers, m.video_heads, m.dropout\n        )\n        self.sensor_embed = nn.Embedding(N_SENSORS, m.sensor_embed_dim)\n\n        gate_in = i_hidden + v_hidden + m.sensor_embed_dim\n        self.gate = nn.Sequential(\n            nn.Linear(gate_in, i_hidden), nn.Sigmoid()\n        )\n        cls_in = i_hidden + i_hidden + m.sensor_embed_dim\n        self.classifier = nn.Sequential(\n            nn.Linear(cls_in, c_hidden),\n            nn.LayerNorm(c_hidden),\n            nn.GELU(),\n            nn.Dropout(m.dropout),\n            nn.Linear(c_hidden, n_classes),\n        )\n        self.modality_dropout = m.modality_dropout\n        self.sensor_dropout = 0.2   # probability of zeroing a sensor in training\n\n    def forward(self, inertial, video, active_mask=None):\n        \"\"\"inertial: (B, 50, 4, 3) | video: (B, 15, 768).\n\n        ``active_mask``: optional (B, 4) float tensor in {0,1} indicating which\n        sensor slots are present (used to mimic the single-sensor test setup).\n        Returns per-sensor logits (B, 4, n_classes).\n        \"\"\"\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 3, 1)         # (B, 4, 50, 3)\n\n        if active_mask is not None:\n            inertial = inertial * active_mask[:, :, None, None].float()\n        elif self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inertial.device) >= self.sensor_dropout\n            keep = keep[:, :, None, None]\n            inertial = inertial * keep.float()\n\n        inertial = inertial.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        inertial_feat = self.inertial_encoder(inertial)   # (B*4, hidden)\n        inertial_feat = inertial_feat.reshape(B, N_SENSORS, -1)\n\n        # Video: shared across sensors, expanded.\n        video_feat = self.video_encoder(video)           # (B, hidden)\n        video_feat = video_feat[:, None].expand(B, N_SENSORS, -1)\n\n        # Fixed per-slot sensor position embedding (discriminative location).\n        pos = self.sensor_embed.weight[None, :, :].expand(B, N_SENSORS, -1)\n\n        if self.training:\n            inertial_feat, video_feat = self._modality_dropout(\n                inertial_feat, video_feat, self.modality_dropout\n            )\n\n        gate_in = torch.cat([inertial_feat, video_feat, pos], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * inertial_feat + (1 - gate) * video_feat\n        interaction = inertial_feat * video_feat\n        features = torch.cat([fused, interaction, pos], dim=-1)  # (B,4,cls_in)\n\n        logits = self.classifier(features)               # (B,4,19)\n        return logits\n\n    @staticmethod\n    def _modality_dropout(i_feat, v_feat, p: float):\n        \"\"\"Zero out an entire modality during training with prob p.\"\"\"\n        if p <= 0:\n            return i_feat, v_feat\n        scale = 1.0 / (1.0 - p)\n        # (B,4,1)\n        drop_i = (torch.rand(i_feat.shape[:2], device=i_feat.device) < p)[..., None]\n        drop_v = (torch.rand(v_feat.shape[:2], device=v_feat.device) < p)[..., None]\n        i_feat = torch.where(drop_i, torch.zeros_like(i_feat), i_feat * scale)\n        v_feat = torch.where(drop_v, torch.zeros_like(v_feat), v_feat * scale)\n        return i_feat, v_feat\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)",
    "train": "\"\"\"Training loop with grouped (participant-independent) CV and AMP.\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_SENSORS, N_CLASSES\nfrom .data import (\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .models import WearFusionNet, count_parameters\n\n\n@dataclass\nclass FoldResult:\n    fold: int\n    oof_targets: np.ndarray\n    oof_logits: np.ndarray       # (n, n_sensors, n_classes) or (n, n_classes)\n    macro_f1: float\n    seed: int\n\n\ndef _worker_init(seed: int):\n    def _init(worker_id):\n        np.random.seed(seed + worker_id)\n    return _init\n\n\ndef _random_single_sensor_mask(valid, device=None):\n    \"\"\"Return a (B, 4) bool mask picking a random valid sensor/window.\"\"\"\n    B, S = valid.shape\n    vf = valid.to(torch.float32)\n    noise = torch.rand(B, S, device=vf.device)\n    cand = (vf - 1.0) * 1e9 + noise\n    idx = cand.argmax(dim=1)                      # (B,)\n    mask = torch.zeros(B, S, dtype=torch.bool, device=vf.device)\n    mask[torch.arange(B, device=vf.device), idx] = True\n    return mask\n\n\ndef make_dataloaders(cfg: Config, train_idx, val_idx, windows, paths,\n                     seed: int, normalizer=None):\n    train_ws = windows.iloc[train_idx].reset_index(drop=True)\n    val_ws = windows.iloc[val_idx].reset_index(drop=True)\n\n    train_ds = WearTrainDataset(train_ws, paths, normalizer=normalizer,\n                                augment=True, data_cfg=cfg.data)\n    val_ds = WearTrainDataset(val_ws, paths, normalizer=normalizer,\n                              augment=False, data_cfg=cfg.data)\n\n    train_dl = DataLoader(\n        train_ds, batch_size=cfg.train.batch_size, shuffle=True,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n    val_dl = DataLoader(\n        val_ds, batch_size=cfg.train.batch_size * 2, shuffle=False,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n    return train_dl, val_dl\n\n\ndef grouped_split(subject_ids: np.ndarray, n_val_subjects: int = 4,\n                  seed: int = 42):\n    \"\"\"Leave-out a disjoint set of subjects -> (train_idx, val_idx).\"\"\"\n    subjects = np.unique(subject_ids)\n    rng = np.random.RandomState(seed)\n    val_subjects = set(rng.choice(subjects, size=n_val_subjects, replace=False))\n    val_idx = np.where(np.isin(subject_ids, list(val_subjects)))[0]\n    train_idx = np.where(~np.isin(subject_ids, list(val_subjects)))[0]\n    return train_idx, val_idx, val_subjects\n\n\ndef train_one_fold(cfg: Config, windows, paths, train_idx, val_idx,\n                   fold: int, seed: int, device: torch.device,\n                   out_dir: Path, normalizer=None) -> FoldResult:\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n\n    model = WearFusionNet(cfg.model).to(device)\n    n_params = count_parameters(model)\n    print(f\"[fold {fold}] params={n_params/1e6:.3f}M\")\n\n    train_dl, val_dl = make_dataloaders(cfg, train_idx, val_idx, windows,\n                                        paths, seed, normalizer=normalizer)\n\n    # Class weights for macro-F1 oriented training.\n    train_targets = windows.iloc[train_idx][\"target\"].to_numpy()\n    class_w = class_weights_from_counts(count_class_labels(train_targets),\n                                        cfg.train.class_weighting,\n                                        device=device)\n    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing,\n                                    weight=class_w)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay\n    )\n    total_steps = cfg.train.epochs * len(train_dl)\n    warmup_steps = cfg.train.warmup_epochs * len(train_dl)\n\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return step / max(1, warmup_steps)\n        # cosine decay\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        return 0.5 * (1 + math_cos(progress * math_pi()))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=cfg.train.amp and device.type == \"cuda\")\n    use_amp = cfg.train.amp and device.type == \"cuda\"\n\n    model.train()\n    global_step = 0\n    t0 = time.time()\n    for epoch in range(cfg.train.epochs):\n        running = 0.0\n        nb = 0\n        for batch in train_dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)          # (B, 4)\n\n            # Single-sensor conditioning: with probability p, present only one\n            # random (valid) sensor so the model learns to predict from a\n            # single sensor, matching the 1-sensor test windows.\n            active_mask = None\n            if cfg.data.single_sensor_prob > 0 and \\\n                    torch.rand(1).item() < cfg.data.single_sensor_prob:\n                active_mask = _random_single_sensor_mask(\n                    valid, device=device)\n                inertial = inertial * active_mask[:, None, :, None].float()\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n                # Per-sensor loss restricted to the active/present sensors.\n                tgt = target[:, None].expand(-1, N_SENSORS).reshape(-1)\n                logits = logits.reshape(-1, logits.shape[-1])\n                if active_mask is not None:\n                    valid_flat = active_mask.reshape(-1)\n                else:\n                    valid_flat = valid.reshape(-1)\n                if valid_flat.any():\n                    loss = criterion(logits[valid_flat], tgt[valid_flat])\n                else:\n                    loss = torch.tensor(0.0, device=device)\n\n            scaler.scale(loss).backward()\n            if cfg.train.max_grad_norm:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(),\n                                               cfg.train.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            global_step += 1\n            running += loss.item() * (1.0 if valid_flat.any() else 1.0)\n            nb += 1\n\n        print(f\"  [fold {fold}] epoch {epoch+1}/{cfg.train.epochs} \"\n              f\"loss={running/max(1,nb):.4f} lr={scheduler.get_last_lr()[0]:.2e} \"\n              f\"time={time.time()-t0:.0f}s\")\n\n    # ---- Validation OOF (single-sensor, matching test) ----\n    model.eval()\n    all_logits = []\n    all_targets = []\n    with torch.inference_mode():\n        for batch in val_dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].numpy()\n            valid = batch[\"valid\"].to(device)\n            # Present a random single valid sensor per window (as in test).\n            active_mask = _random_single_sensor_mask(valid, device=device)\n            inertial = inertial * active_mask[:, None, :, None].float()\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n                # Take only the active sensor's logits.\n                active_idx = active_mask.long().argmax(dim=1)\n                logits = logits[torch.arange(logits.shape[0], device=device),\n                                active_idx, :]\n            all_logits.append(logits.float().cpu().numpy())\n            all_targets.append(target)\n\n    logits = np.concatenate(all_logits, axis=0)   # (N,19) single-sensor logits\n    targets = np.concatenate(all_targets, axis=0)  # (N,)\n\n    probs = softmax(logits, axis=-1)              # (N,19)\n    preds = probs.argmax(axis=1)\n    f1 = macro_f1(targets, preds, n_classes=N_CLASSES)\n\n    return FoldResult(\n        fold=fold,\n        oof_targets=targets,\n        oof_logits=probs,\n        macro_f1=f1,\n        seed=seed,\n    )\n\n\n# ---------------------------------------------------------------------------\n# Small helpers\n# ---------------------------------------------------------------------------\ndef class_weights_from_counts(counts: np.ndarray, mode: str,\n                              n_classes: int = N_CLASSES,\n                              device=None) -> torch.Tensor | None:\n    \"\"\"Compute per-class loss weights for macro-F1 oriented training.\"\"\"\n    if mode == \"none\":\n        return None\n    counts = counts.astype(np.float64) + 1.0\n    w = 1.0 / counts\n    if mode == \"sqrt_inverse\":\n        w = 1.0 / np.sqrt(counts)\n    # Normalize so mean weight is 1.\n    w = w / w.mean()\n    t = torch.as_tensor(w, dtype=torch.float32)\n    return t.to(device) if device is not None else t\n\n\ndef count_class_labels(targets: np.ndarray, n_classes: int = N_CLASSES) -> np.ndarray:\n    counts = np.bincount(targets, minlength=n_classes).astype(np.float64)\n    return counts\n\n\ndef math_cos(x):\n    return float(np.cos(x))\n\n\ndef math_pi():\n    return float(np.pi)\n\n\ndef softmax(x, axis=-1):\n    e = np.exp(x - x.max(axis=axis, keepdims=True))\n    return e / e.sum(axis=axis, keepdims=True)\n\n\ndef macro_f1(y_true, y_pred, n_classes=None):\n    from sklearn.metrics import f1_score\n\n    if n_classes is None:\n        n_classes = max(y_true.max(), y_pred.max()) + 1\n    return float(f1_score(y_true, y_pred, average=\"macro\", labels=list(range(n_classes))))\n\n\ndef run_cv(cfg: Config, n_folds: int = 1, n_val_subjects: int = 4,\n           device=None, out_dir: Path | None = None):\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"device={device}\")\n\n    out_dir = out_dir or (cfg.paths.output_dir / \"cv\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building window table...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"windows total = {len(windows)}\")\n\n    # Fit a global inertial normalizer once from training data.\n    from .data import InertialNormalizer\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    print(f\"normalizer mean/ (sensor,axis): {normalizer.mean.shape}\")\n\n    subject_ids = windows[\"sbj_id\"].to_numpy()\n    results = []\n    for fold in range(n_folds):\n        train_idx, val_idx, val_subjects = grouped_split(\n            subject_ids, n_val_subjects=n_val_subjects, seed=cfg.data.seed + fold\n        )\n        print(f\"\\n=== fold {fold}: val subjects = {sorted(val_subjects)} \"\n              f\"train={len(train_idx)} val={len(val_idx)} ===\")\n        res = train_one_fold(cfg, windows, cfg.paths, train_idx, val_idx,\n                             fold, cfg.train.seed + fold, device, out_dir,\n                             normalizer=normalizer)\n        print(f\"fold {fold} macro-F1 = {res.macro_f1:.4f}\")\n        results.append(res)\n\n    mean_f1 = float(np.mean([r.macro_f1 for r in results]))\n    print(f\"\\nmean macro-F1 over {n_folds} folds = {mean_f1:.4f}\")\n\n    # Optimise per-class logit offsets on the pooled OOF set.\n    from .tuning import optimize_class_offsets, apply_offsets, argmax_preds\n    if n_folds > 0 and results:\n        all_probs = np.concatenate([r.oof_logits for r in results], axis=0)\n        all_targets = np.concatenate([r.oof_targets for r in results], axis=0)\n        offsets = optimize_class_offsets(all_probs, all_targets,\n                                         n_classes=N_CLASSES)\n        tuned_f1 = macro_f1(all_targets,\n                            argmax_preds(apply_offsets(all_probs, offsets)))\n        print(f\"\\n[OOF tuning] raw macro-F1={mean_f1:.4f} | \"\n              f\"offset-tuned macro-F1={tuned_f1:.4f}\")\n        print(\"offsets:\", np.round(offsets, 3).tolist())\n        json_summary = {\n            \"folds\": [{\"fold\": r.fold, \"macro_f1\": r.macro_f1}\n                      for r in results],\n            \"mean_macro_f1\": mean_f1,\n            \"offset_tuned_macro_f1\": tuned_f1,\n            \"class_offsets\": offsets.tolist(),\n        }\n    else:\n        json_summary = {\"folds\": [], \"mean_macro_f1\": mean_f1}\n\n    cfg.save(out_dir / \"config.json\")\n    (out_dir / \"cv_summary.json\").write_text(\n        json.dumps(json_summary, indent=2)\n    )\n    return results, mean_f1\n\n\ndef train_full(cfg: Config, device=None, out_path: Path | None = None):\n    \"\"\"Train on the entire training set (no holdout) and return the model.\n\n    Used for the final submission model once hyper-parameters are fixed.\n    \"\"\"\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"full-data windows = {len(windows)}\")\n\n    from .data import InertialNormalizer\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer,\n                          augment=True, data_cfg=cfg.data)\n    dl = DataLoader(\n        ds, batch_size=cfg.train.batch_size, shuffle=True,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(cfg.train.seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n\n    torch.manual_seed(cfg.train.seed)\n    model = WearFusionNet(cfg.model).to(device)\n    class_w = class_weights_from_counts(count_class_labels(windows[\"target\"].to_numpy()),\n                                        cfg.train.class_weighting, device=device)\n    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing,\n                                    weight=class_w)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay\n    )\n    total_steps = cfg.train.epochs * len(dl)\n    warmup_steps = cfg.train.warmup_epochs * len(dl)\n\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return step / max(1, warmup_steps)\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        return 0.5 * (1 + math_cos(progress * math_pi()))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n    scaler = torch.amp.GradScaler(\"cuda\",\n                                  enabled=cfg.train.amp and device.type == \"cuda\")\n    use_amp = cfg.train.amp and device.type == \"cuda\"\n\n    model.train()\n    t0 = time.time()\n    for epoch in range(cfg.train.epochs):\n        running = 0.0\n        nb = 0\n        for batch in dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n\n            active_mask = None\n            if cfg.data.single_sensor_prob > 0 and \\\n                    torch.rand(1).item() < cfg.data.single_sensor_prob:\n                active_mask = _random_single_sensor_mask(valid, device=device)\n                inertial = inertial * active_mask[:, None, :, None].float()\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)\n                tgt = target[:, None].expand(-1, N_SENSORS).reshape(-1)\n                logits = logits.reshape(-1, logits.shape[-1])\n                valid_flat = (active_mask if active_mask is not None else valid).reshape(-1)\n                loss = criterion(logits[valid_flat], tgt[valid_flat]) if valid_flat.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if cfg.train.max_grad_norm:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(),\n                                               cfg.train.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            running += loss.item()\n            nb += 1\n        print(f\"[full] epoch {epoch+1}/{cfg.train.epochs} \"\n              f\"loss={running/max(1,nb):.4f} time={time.time()-t0:.0f}s\")\n\n    if out_path is not None:\n        out_path = Path(out_path)\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        # Save model config as a plain dict so the checkpoint stays\n        # picklable with torch's default weights_only=True.\n        from dataclasses import asdict\n        torch.save({\"model\": model.state_dict(),\n                    \"cfg\": asdict(cfg.model),\n                    \"normalizer_mean\": normalizer.mean.tolist(),\n                    \"normalizer_std\": normalizer.std.tolist()},\n                   out_path)\n        print(f\"saved model -> {out_path}\")\n    return model\n\n\ndef load_model_for_inference(model_path: Path, device) -> WearFusionNet:\n    \"\"\"Load a saved full-data model.\"\"\"\n    ckpt = torch.load(model_path, map_location=device, weights_only=True)\n    from .config import Config as _C, ModelConfig\n    model_cfg = ModelConfig(**ckpt[\"cfg\"])\n    model = WearFusionNet(model_cfg).to(device)\n    model.load_state_dict(ckpt[\"model\"])\n    model.eval()\n    model._normalizer_mean = np.asarray(ckpt.get(\"normalizer_mean\"), dtype=np.float32)\n    model._normalizer_std = np.asarray(ckpt.get(\"normalizer_std\"), dtype=np.float32)\n    return model",
    "inference": "\"\"\"Test-time inference and submission generation.\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .config import (\n    INERTIAL_WINDOW,\n    N_CLASSES,\n    N_SENSORS,\n    VIDEO_DIM,\n    VIDEO_WINDOW,\n    Config,\n)\nfrom .data import build_test_dataset\nfrom .models import WearFusionNet\n\n\nclass WearTestDataset(Dataset):\n    def __init__(self, paths, null_video: bool = False, normalizer=None):\n        inertial, video, sensor_ids, ids, sbj = build_test_dataset(paths)\n        self.sensor_ids = sensor_ids\n        self.ids = ids\n        self.inertial = np.nan_to_num(np.asarray(inertial, dtype=np.float32).copy())\n        if normalizer is not None:\n            if hasattr(normalizer, \"mean\"):\n                # Global per-(sensor,axis) normalizer: use the window's sensor.\n                mean = normalizer.mean[sensor_ids]   # (N, 3)\n                std = normalizer.std[sensor_ids]     # (N, 3)\n                self.inertial = (self.inertial - mean[:, None, :]) / std[:, None, :]\n            else:\n                # Generic per-window normalizer (e.g. PerWindowNormalizer).\n                self.inertial = normalizer.transform(self.inertial)\n        self.video = np.nan_to_num(np.asarray(video, dtype=np.float32).copy())\n        if null_video:\n            self.video = np.zeros_like(self.video)\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, idx):\n        return {\n            \"inertial\": self.inertial[idx],      # (50,3)\n            \"video\": self.video[idx],            # (15,768)\n            \"sensor_id\": self.sensor_ids[idx],\n            \"id\": self.ids[idx],\n        }\n\n\ndef _collate(batch):\n    inertial = torch.as_tensor(\n        np.stack([b[\"inertial\"] for b in batch])\n    )                       # (B,50,3)\n    video = torch.as_tensor(np.stack([b[\"video\"] for b in batch]))\n    sensor_id = torch.as_tensor([b[\"sensor_id\"] for b in batch])\n    ids = torch.as_tensor([b[\"id\"] for b in batch])\n    return inertial, video, sensor_id, ids\n\n\ndef predict_test(model: WearFusionNet, paths, device, batch_size=512,\n                 null_bias: float = 0.0, video_weight: float = 1.0,\n                 amp=True, normalizer=None):\n    \"\"\"Return (ids, probs (N,19)). Applies null bias to class 0.\"\"\"\n    ds = WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,\n                    collate_fn=_collate)\n    model.eval()\n    use_amp = amp and device.type == \"cuda\"\n\n    all_probs = []\n    all_ids = []\n    with torch.inference_mode():\n        for inertial, video, sensor_id, ids in dl:\n            inertial = inertial.to(device)\n            video = video.to(device)\n            sensor_id = sensor_id.to(device)\n            # reshape single sensor -> (B,50,4,3); NaN out non-present sensors\n            inertial = inertial.unsqueeze(2).expand(\n                -1, -1, N_SENSORS, -1)  # (B,50,4,3)\n            present = torch.zeros(inertial.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inertial = inertial * present[:, None, :, None]\n\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n            probs = torch.softmax(logits.float(), dim=-1)\n\n            # Use only the prediction from the window's present sensor slot.\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]           # (B,19)\n\n            # Video-weighting (if we ever want to down-weight video).\n            if video_weight != 1.0:\n                # Simple heuristic: blend toward uniform along video axis.\n                pass\n\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n\n    if null_bias:\n        probs = probs.copy()\n        probs[:, 0] *= np.exp(null_bias)\n\n    order = np.argsort(ids)\n    return ids[order], probs[order]\n\n\ndef write_submission(ids, probs, sample_submission_path, out_path):\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    if probs.ndim == 2:\n        preds = probs.argmax(axis=1)\n    else:\n        preds = probs\n    df = pd.DataFrame({\"id\": ids, \"target_feature\": preds})\n    df.to_csv(out_path, index=False)\n    return df\n\n\ndef predict_ensemble(models, paths, device, batch_size=512, null_bias=0.0,\n                     amp=True, normalizer=None):\n    \"\"\"Average probabilities across an ensemble of model objects.\"\"\"\n    ds = WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,\n                    collate_fn=_collate)\n    use_amp = amp and device.type == \"cuda\"\n    acc = None\n    all_ids = None\n    for m in models:\n        m.eval()\n        m_acc = None\n        ids_out = None\n        with torch.inference_mode():\n            for inertial, video, sensor_id, ids in dl:\n                inertial = inertial.to(device)\n                video = video.to(device)\n                sensor_id = sensor_id.to(device)\n                inertial = inertial.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n                present = torch.zeros(inertial.shape[0], N_SENSORS, device=device)\n                present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n                inertial = inertial * present[:, None, :, None]\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = m(inertial, video)\n                probs = torch.softmax(logits.float(), dim=-1)\n                probs = probs[torch.arange(probs.shape[0], device=device),\n                              sensor_id, :].cpu().numpy()\n                m_acc = probs if m_acc is None else np.concatenate([m_acc, probs], 0)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        m_acc = m_acc[np.argsort(ids_out)]\n        acc = m_acc if acc is None else acc + m_acc\n        all_ids = ids_out if all_ids is None else all_ids\n\n    acc = acc / len(models)\n    if null_bias:\n        acc = acc.copy()\n        acc[:, 0] *= np.exp(null_bias)\n    return all_ids, acc",
    "tuning": "\"\"\"OOF-based ensemble blending and per-class threshold optimisation.\n\nThese tools operate on out-of-fold (OOF) probabilities.  Because macro-F1\ntreats all classes equally, a plain argmax is often sub-optimal; a small\nper-class logit offset can meaningfully improve the metric, especially for\nthe dominant ``null`` class.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .train import macro_f1, softmax\n\n\ndef argmax_preds(probs: np.ndarray) -> np.ndarray:\n    return probs.argmax(axis=1)\n\n\ndef apply_offsets(probs: np.ndarray, offsets: np.ndarray) -> np.ndarray:\n    \"\"\"Add a per-class logit offset (i.e. multiply prob by exp(offset)).\"\"\"\n    p = probs * np.exp(offsets[None, :])\n    p = p / p.sum(axis=1, keepdims=True)\n    return p\n\n\ndef optimize_class_offsets(probs: np.ndarray, targets: np.ndarray,\n                           n_classes: int, iters: int = 3,\n                           grid: np.ndarray | None = None) -> np.ndarray:\n    \"\"\"Coordinate-ascent per-class logit offsets to maximise macro-F1.\n\n    Returns ``offsets`` shape (n_classes,).  Works on OOF probabilities.\n    The search is intentionally conservative to limit overfitting: each class\n    is nudged by at most one grid step per iteration and never away from 0\n    unless it strictly improves the metric.\n    \"\"\"\n    if grid is None:\n        grid = np.linspace(-1.0, 1.0, 13)\n    offsets = np.zeros(n_classes, dtype=np.float64)\n    best = macro_f1(targets, argmax_preds(apply_offsets(probs, offsets)))\n    for _ in range(iters):\n        for c in range(n_classes):\n            cur = offsets[c]\n            best_off, best_val = cur, best\n            for g in grid:\n                cand = cur + g\n                offsets[c] = cand\n                p = apply_offsets(probs, offsets)\n                v = macro_f1(targets, argmax_preds(p))\n                if v > best_val + 1e-6:\n                    best_val, best_off = v, cand\n            offsets[c] = best_off\n            best = best_val\n    return offsets\n\n\ndef optimize_ensemble_weights(model_probs: list[np.ndarray],\n                              targets: np.ndarray,\n                              n_classes: int,\n                              iters: int = 50) -> np.ndarray:\n    \"\"\"Iterative weight search for a softmax-probability ensemble.\n\n    ``model_probs``: list of (N, n_classes) probability matrices.\n    Returns non-negative weights summing to 1 that maximise macro-F1.\n    \"\"\"\n    K = len(model_probs)\n    # Start with a probability-averaging init (weights 1/K).\n    w = np.ones(K, dtype=np.float64) / K\n    best = _ensemble_f1(model_probs, w, targets)\n    rng = np.random.RandomState(0)\n    for _ in range(iters):\n        cand = w + rng.normal(0, 0.15, K)\n        cand = np.clip(cand, 0, None)\n        if cand.sum() <= 0:\n            continue\n        cand = cand / cand.sum()\n        v = _ensemble_f1(model_probs, cand, targets)\n        if v > best:\n            best, w = v, cand\n    return w\n\n\ndef _ensemble_f1(model_probs, weights, targets):\n    blended = sum(wi * p for wi, p in zip(weights, model_probs))\n    return macro_f1(targets, argmax_preds(blended))\n\n\ndef combine_models(probs_list: list[np.ndarray],\n                   weights: np.ndarray | None = None) -> np.ndarray:\n    \"\"\"Weighted sum of probability matrices (soft-voting).\"\"\"\n    if weights is None:\n        weights = np.ones(len(probs_list)) / len(probs_list)\n    return sum(w * p for w, p in zip(weights, probs_list))",
    "train_ensemble": "\"\"\"Phase 2: train an ensemble of diverse models, blend OOF, and predict test.\n\nRun on Kaggle (2x T4) after the single-model CV.  Trains ``n_models`` models\n(each on its own grouped fold or the full data), collects OOF probabilities,\noptimises ensemble weights + per-class offsets, and writes a submission.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .config import Config, N_CLASSES\nfrom .data import (\n    InertialNormalizer,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .train import (\n    count_class_labels,\n    grouped_split,\n    make_dataloaders,\n    train_one_fold,\n    macro_f1,\n    softmax,\n)\nfrom .tuning import (\n    optimize_class_offsets,\n    optimize_ensemble_weights,\n    apply_offsets,\n    argmax_preds,\n    combine_models,\n)\nfrom .inference import predict_test, predict_ensemble, write_submission\n\n\ndef collect_oof(cfg: Config, windows, paths, normalizer, n_folds, device,\n                seeds, out_dir) -> tuple[np.ndarray, np.ndarray, list]:\n    \"\"\"Run grouped CV per seed and return pooled OOF probs/targets + models list.\n\n    Returns (all_probs (N,19), all_targets (N,), model_list).\n    OOF entries from different folds/seeds are concatenated.\n    \"\"\"\n    subject_ids = windows[\"sbj_id\"].to_numpy()\n    all_probs, all_targets = [], []\n    # Per-seed OOF alignment: we keep each (fold,seed) OOF separately.\n    n_val = cfg.eval.n_val_subjects\n    for seed in seeds:\n        train_idx, val_idx, val_subjects = grouped_split(\n            subject_ids, n_val_subjects=n_val, seed=seed\n        )\n        res = train_one_fold(cfg, windows, paths, train_idx, val_idx,\n                             fold=f\"seed{seed}\", seed=seed, device=device,\n                             out_dir=out_dir, normalizer=normalizer)\n        all_probs.append(res.oof_logits)\n        all_targets.append(res.oof_targets)\n    return (np.concatenate(all_probs, axis=0),\n            np.concatenate(all_targets, axis=0))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--models\", type=int, default=4, help=\"number of seeds\")\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--max-per-class\", type=int, default=1000)\n    ap.add_argument(\"--class-weighting\", type=str, default=\"sqrt_inverse\")\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--single-sensor-prob\", type=float, default=0.6)\n    ap.add_argument(\"--scale\", type=float, default=1.0)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.train.epochs = args.epochs\n    cfg.train.batch_size = args.batch_size\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    cfg.train.class_weighting = args.class_weighting\n    cfg.model.scale = args.scale\n    cfg.data.single_sensor_prob = args.single_sensor_prob\n    cfg.eval.null_bias = args.null_bias\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    out_dir = cfg.paths.output_dir / \"ensemble\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    seeds = [cfg.train.seed + i for i in range(args.models)]\n    all_probs, all_targets = collect_oof(cfg, windows, cfg.paths, normalizer,\n                                         args.models, device, seeds, out_dir)\n    print(f\"pooled OOF probs {all_probs.shape}, targets {all_targets.shape}\")\n\n    # Per-seed OOF splits for ensemble-weight optimisation.\n    # (We optimise weights over the concatenated OOF; for a proper estimate we\n    # treat each seed's OOF as a \"model\".)\n    raw_f1 = macro_f1(all_targets, argmax_preds(all_probs))\n    print(f\"single best OOF macro-F1 = {raw_f1:.4f}\")\n\n    offsets = optimize_class_offsets(all_probs, all_targets, N_CLASSES)\n    tuned = macro_f1(all_targets, argmax_preds(apply_offsets(all_probs, offsets)))\n    print(f\"offset-tuned OOF macro-F1 = {tuned:.4f}\")\n    # Per-class offsets tuned on OOF frequently overfit the held-out subjects\n    # and hurt the real test set. For the submission we rely on the safe\n    # null-bias lever only, and merely report the tuned figure for reference.\n    apply_offs = False\n    print(\"per-class offsets applied to test? False (safe null-bias only)\")\n\n    # Tune the null-class multiplicative bias on the (now single-sensor) OOF.\n    best_bias, best_bias_f1 = 0.0, raw_f1\n    for b in np.linspace(0.0, 2.0, 21):\n        p = all_probs.copy()\n        p[:, 0] *= np.exp(b)\n        p = p / p.sum(1, keepdims=True)\n        f = macro_f1(all_targets, argmax_preds(p))\n        if f > best_bias_f1:\n            best_bias_f1, best_bias = f, b\n    print(f\"null-bias tuned on OOF: {best_bias:.2f} -> macro-F1 {best_bias_f1:.4f}\")\n    null_bias = best_bias\n\n    # --- full-data training + test prediction for each seed ---\n    from .train import train_full\n    models = []\n    for i, seed in enumerate(seeds):\n        cfg.train.seed = seed\n        cfg.data.seed = seed\n        mpath = out_dir / f\"full_seed{seed}.pt\"\n        m = train_full(cfg, device=device, out_path=mpath)\n        models.append(m)\n        print(f\"trained full model seed {seed}\")\n\n    # Ensemble prediction: average probs over models, apply tuned null bias.\n    all_ids, ens_probs = predict_ensemble(models, cfg.paths, device,\n                                          null_bias=0.0, normalizer=normalizer)\n    if null_bias:\n        ens_probs = ens_probs.copy()\n        ens_probs[:, 0] *= np.exp(null_bias)\n        ens_probs = ens_probs / ens_probs.sum(1, keepdims=True)\n    write_submission(all_ids, ens_probs, cfg.paths.sample_submission,\n                     out_dir / \"submission.csv\")\n    print(f\"submission written with {len(all_ids)} rows\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "reference_baseline": "\"\"\"Faithful reproduction of the proven reference baseline (scored 0.61278).\n\nThis ports the \"TS/Emb 3WDC | Temporal Fusion Ensemble\" notebook by Nomannic\ninto our framework so we can (a) validate our data/inference pipeline against a\nknown-good public-leaderboard score, and (b) establish a correct floor to\nimprove on.\n\nReference architecture (from the notebook):\n  * PooledFusionModel  -- per-sensor statistical pooling + video stats\n  * TemporalFusionModel-- Inception inertial CNN + Transformer video + gated fusion\n  * Blend: 0.60 * temporal + 0.40 * pooled, null_bias = exp(0.75) on class 0.\n\nTraining details (reference):\n  * Trains on ALL 4 sensors per window (per-sensor predictions + valid mask).\n  * 150 windows / subject / class; pooled 3 epochs, temporal 4 epochs.\n  * CrossEntropyLoss(label_smoothing=0.05) over valid sensors.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nfrom .config import (\n    INERTIAL_WINDOW,\n    N_CLASSES,\n    N_SENSORS,\n    VIDEO_DIM,\n    VIDEO_WINDOW,\n)\n\n\n# ---------------------------------------------------------------------------\n# Reference building blocks\n# ---------------------------------------------------------------------------\nclass RefInceptionBlock(nn.Module):\n    \"\"\"Reference multi-scale conv block (kernels 5/11/21).\"\"\"\n\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(8, out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)\n        out = out + self.skip(x)\n        return self.dropout(torch.nn.functional.gelu(self.norm(out)))\n\n\n# ---------------------------------------------------------------------------\n# PooledFusionModel\n# ---------------------------------------------------------------------------\nclass PooledFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES, scale: float = 1.0):\n        super().__init__()\n        ih = int(64 * scale)\n        vh = int(192 * scale)\n        ch = int(128 * scale)\n        self.inertial_encoder = nn.Sequential(\n            nn.Linear(16, ih), nn.LayerNorm(ih), nn.GELU(), nn.Dropout(0.15),\n        )\n        self.video_encoder = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM * 2),\n            nn.Linear(VIDEO_DIM * 2, vh),\n            nn.GELU(),\n            nn.Dropout(0.2),\n        )\n        self.sensor_embedding = nn.Embedding(N_SENSORS, 8)\n        self.classifier = nn.Sequential(\n            nn.Linear(ih + vh + 8, ch),\n            nn.GELU(),\n            nn.Dropout(0.2),\n            nn.Linear(ch, n_classes),\n        )\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        # Per-sensor statistical features (16 per sensor).\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 1, 3)                   # (B,4,50,3)\n        mag = torch.linalg.vector_norm(inertial, dim=-1)          # (B,4,50)\n        mean = inertial.mean(dim=2)                                # (B,4,3)\n        std = inertial.std(dim=2)                                  # (B,4,3)\n        amin = inertial.amin(dim=2)\n        amax = inertial.amax(dim=2)\n        feats = torch.cat([\n            mean, std, amin, amax,                                  # 12\n            mag.mean(dim=2, keepdim=True),\n            mag.std(dim=2, keepdim=True),\n            mag.amin(dim=2, keepdim=True),\n            mag.amax(dim=2, keepdim=True),                          # 4\n        ], dim=-1)                                                 # (B,4,16)\n\n        i_feat = self.inertial_encoder(feats)                      # (B,4,64)\n\n        v_mean = video.mean(dim=1)                                 # (B,768)\n        v_std = video.std(dim=1)\n        v_feat = self.video_encoder(torch.cat([v_mean, v_std], dim=-1))  # (B,192)\n        v_feat = v_feat[:, None].expand(B, N_SENSORS, -1)\n\n        sens = self.sensor_embedding.weight[None].expand(B, N_SENSORS, -1)\n\n        cls_in = torch.cat([i_feat, v_feat, sens], dim=-1)         # (B,4,264)\n        logits = self.classifier(cls_in)                           # (B,4,19)\n        return logits\n\n\n# ---------------------------------------------------------------------------\n# TemporalFusionModel\n# ---------------------------------------------------------------------------\nclass RefInertialEncoder(nn.Module):\n    def __init__(self, dropout: float = 0.2, scale: float = 1.0):\n        super().__init__()\n        c = int(128 * scale)\n        h = int(192 * scale)\n        self.blocks = nn.Sequential(\n            RefInceptionBlock(4, c),\n            RefInceptionBlock(c, c),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(2 * c, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,3,50) -> (B,h); adds magnitude channel\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)  # (B,1,50)\n        x = torch.cat([x, mag], dim=1)                          # (B,4,50)\n        h = self.blocks(x)          # (B,c,50)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)  # (B,2c)\n        return self.head(pooled)\n\n\nclass RefVideoEncoder(nn.Module):\n    def __init__(self, dropout: float = 0.2, scale: float = 1.0):\n        super().__init__()\n        h = int(192 * scale)\n        ff = int(384 * scale)\n        heads = 4\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM), nn.Linear(VIDEO_DIM, h), nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, h)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            h, heads, dim_feedforward=ff, dropout=0.15, activation=\"gelu\",\n            batch_first=True, norm_first=True,\n        )\n        self.temporal = nn.TransformerEncoder(layer, 1,\n                                              enable_nested_tensor=False)\n        self.attention = nn.Sequential(nn.Linear(h, int(64 * scale)), nn.Tanh(),\n                                       nn.Linear(int(64 * scale), 1))\n        self.head = nn.Sequential(\n            nn.Linear(2 * h, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,15,768)\n        h = self.projection(x) + self.position\n        h = self.temporal(h)\n        scores = self.attention(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\nclass TemporalFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES, scale: float = 1.0):\n        super().__init__()\n        h = int(192 * scale)\n        self.inertial_encoder = RefInertialEncoder(scale=scale)\n        self.video_encoder = RefVideoEncoder(scale=scale)\n        self.sensor_embedding = nn.Embedding(N_SENSORS, int(16 * scale))\n        se = int(16 * scale)\n        self.gate = nn.Sequential(nn.Linear(2 * h + se, h), nn.Sigmoid())\n        self.classifier = nn.Sequential(\n            nn.Linear(2 * h + se, h),\n            nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.3),\n            nn.Linear(h, n_classes),\n        )\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        h = self.inertial_encoder.head[0].out_features\n        # Per-sensor inertial (B*4, 3, 50) -> (B,4,h)\n        inert = inertial.permute(0, 2, 3, 1).reshape(B * N_SENSORS, 3,\n                                                     INERTIAL_WINDOW)\n        i_feat = self.inertial_encoder(inert).reshape(B, N_SENSORS, h)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        sens = self.sensor_embedding.weight[None].expand(B, N_SENSORS, -1)\n\n        gate_in = torch.cat([i_feat, v_feat, sens], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * i_feat + (1 - gate) * v_feat\n        features = torch.cat([fused, i_feat * v_feat, sens], dim=-1)\n        logits = self.classifier(features)   # (B,4,19)\n        return logits\n\n\n# ---------------------------------------------------------------------------\n# Train / eval helpers matching the reference\n# ---------------------------------------------------------------------------\ndef blend_probabilities(pooled_logits, temporal_logits, temporal_weight=0.60,\n                        null_bias=0.75):\n    \"\"\"Blend per-sensor logits -> (N,19) probabilities with null bias.\"\"\"\n    p_pooled = torch.softmax(pooled_logits, dim=-1)\n    p_temporal = torch.softmax(temporal_logits, dim=-1)\n    p = temporal_weight * p_temporal + (1 - temporal_weight) * p_pooled\n    # (N,4,19) -> average over sensors -> (N,19)\n    p = p.mean(dim=1)\n    if null_bias:\n        p[:, 0] *= math.exp(null_bias)\n    return p / p.sum(dim=1, keepdim=True)",
    "run_reference": "\"\"\"Run the proven reference baseline (reproduce ~0.61278 on the leaderboard).\n\nTrains the Pooled + Temporal fusion models exactly as the reference notebook,\nthen blends them and writes a submission.  Use this to validate that our data /\ninference pipeline reproduces the known-good score.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .reference_baseline import PooledFusionModel, TemporalFusionModel\n\n\nclass _NullVideo:\n    \"\"\"Wrap a dataset and return zeroed video (inertial-only mode).\"\"\"\n\n    def __init__(self, inner):\n        self.inner = inner\n\n    def __len__(self):\n        return len(self.inner)\n\n    def __getitem__(self, idx):\n        b = dict(self.inner[idx])\n        b[\"video\"] = np.zeros_like(b[\"video\"])\n        return b\n\n\ndef _train_model(model, dl, epochs, lr, wd, device, use_amp, label_smooth=0.05,\n                 warmup_epochs=1, grad_norm=5.0, tag=\"\", seed=0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer,\n                  inertial_only=False):\n    from .inference import WearTestDataset\n    from .data import build_test_dataset\n    from . import inference as inf\n\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device)\n            vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            if inertial_only:\n                vid = torch.zeros_like(vid)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)   # (B,4,19)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]       # (B,19)\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs-pooled\", type=int, default=3)\n    ap.add_argument(\"--epochs-temporal\", type=int, default=4)\n    ap.add_argument(\"--temporal-weight\", type=float, default=0.60)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--normalize\", type=int, default=0,\n                    help=\"0=none, 1=global z-score, 2=per-window z-score\")\n    ap.add_argument(\"--scale\", type=float, default=1.0,\n                    help=\"model width multiplier (1.0 = original)\")\n    ap.add_argument(\"--seeds\", type=int, default=1,\n                    help=\"train this many seeds and average their blended probs\")\n    ap.add_argument(\"--inertial-only\", type=int, default=0,\n                    help=\"1 to zero the video modality (inertial-only model)\")\n    ap.add_argument(\"--smooth-window\", type=int, default=0,\n                    help=\"temporal majority-vote smoothing half-window k (0=off)\")\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"reference\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = None\n    if args.normalize == 1:\n        normalizer = InertialNormalizer.fit(cfg.paths)\n        print(\"using global inertial normalisation\")\n    elif args.normalize == 2:\n        from .data import PerWindowNormalizer\n        normalizer = PerWindowNormalizer()\n        print(\"using per-window inertial normalisation\")\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    if args.inertial_only:\n        ds = _NullVideo(ds)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc_blend = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        pooled = PooledFusionModel(scale=args.scale).to(device)\n        temporal = TemporalFusionModel(scale=args.scale).to(device)\n        print(f\"seed {seed}: Pooled={sum(p.numel() for p in pooled.parameters())/1e6:.3f}M \"\n              f\"Temporal={sum(p.numel() for p in temporal.parameters())/1e6:.3f}M\")\n\n        _train_model(pooled, dl, args.epochs_pooled, lr=2e-3, wd=1e-4,\n                     device=device, use_amp=use_amp, tag=f\"pooled-{seed}\", seed=seed)\n        _train_model(temporal, dl, args.epochs_temporal, lr=8e-4, wd=1e-3,\n                     device=device, use_amp=use_amp, tag=f\"temporal-{seed}\", seed=seed)\n\n        pooled_ids, pooled_probs = _predict_test(pooled, cfg.paths, device,\n                                                 use_amp, normalizer,\n                                                 args.inertial_only)\n        temp_ids, temp_probs = _predict_test(temporal, cfg.paths, device,\n                                             use_amp, normalizer,\n                                             args.inertial_only)\n        assert (pooled_ids == temp_ids).all()\n        blend = args.temporal_weight * temp_probs + (1 - args.temporal_weight) * pooled_probs\n        ref_ids = pooled_ids\n        acc_blend = blend if acc_blend is None else acc_blend + blend\n\n    blended = acc_blend / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    # Optional temporal majority-vote smoothing (per subject, in id order).\n    if args.smooth_window > 0:\n        import pandas as pd\n        from .temporal_smooth import smooth_majority_vote\n        meta = pd.read_csv(cfg.paths.test_meta)\n        # probs are aligned to meta rows sorted by id == meta row order (ids 0..N-1)\n        smoothed = smooth_majority_vote(blended, meta, k=args.smooth_window)\n        preds = smoothed\n        print(f\"temporal smoothing applied (k={args.smooth_window})\")\n    else:\n        preds = blended.argmax(axis=1)\n\n    # Write to the standard Kaggle submission location so it's easy to submit.\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, preds, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose": "\"\"\"Diagnostic: which modality carries the signal?\n\nRuns a fast grouped CV (held-out subjects) with the reference architecture in\nthree configurations:\n  * inertial-only  (video replaced by zeros)\n  * video-only     (inertial replaced by zeros)\n  * fusion         (both)\n\nReports macro-F1 per config.  This tells us whether video is adding signal or\nif there's an alignment problem, guiding where to invest.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import build_dataset_windows, subsample_windows\nfrom .train import grouped_split, make_dataloaders, macro_f1, softmax\nfrom .reference_baseline import TemporalFusionModel\n\n\nclass NullVideoDataset:\n    \"\"\"Wraps WearTrainDataset and returns zero video.\"\"\"\n    def __init__(self, ds):\n        self.ds = ds\n\n    def __len__(self):\n        return len(self.ds)\n\n    def __getitem__(self, idx):\n        b = self.ds[idx]\n        b = dict(b)\n        b[\"video\"] = np.zeros_like(b[\"video\"])\n        return b\n\n\ndef _run(cfg, windows, train_idx, val_idx, normalizer, device, use_amp,\n         null_video, null_inertial, epochs=3, tag=\"\"):\n    from .data import WearTrainDataset\n\n    def make(idx, nv, ni):\n        ws = windows.iloc[idx].reset_index(drop=True)\n        ds = WearTrainDataset(ws, cfg.paths, normalizer=normalizer)\n        if nv:\n            ds = NullVideoDataset(ds)\n        if ni:\n            # zero inertial\n            class NID:\n                def __init__(s, inner): s.inner = inner\n                def __len__(s): return len(s.inner)\n                def __getitem__(s, i):\n                    b = dict(s.inner[i]); b[\"inertial\"] = np.zeros_like(b[\"inertial\"]); return b\n            ds = NID(ds)\n        return ds\n\n    tr_ds = make(train_idx, null_video, null_inertial)\n    va_ds = make(val_idx, null_video, null_inertial)\n    tr_dl = DataLoader(tr_ds, batch_size=cfg.train.batch_size, shuffle=True,\n                       num_workers=0, pin_memory=True)\n    va_dl = DataLoader(va_ds, batch_size=cfg.train.batch_size * 2, shuffle=False,\n                       num_workers=0, pin_memory=True)\n\n    torch.manual_seed(0)\n    model = TemporalFusionModel().to(device)\n    crit = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing)\n    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        for b in tr_dl:\n            inert = b[\"inertial\"].to(device); vid = b[\"video\"].to(device)\n            tgt = b[\"target\"].to(device); valid = b[\"valid\"].to(device)\n            opt.zero_grad()\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = crit(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()\n        # simple LR step (diagnostic)\n        for g in opt.param_groups:\n            g[\"lr\"] *= 0.7\n\n    model.eval()\n    yt, yp = [], []\n    with torch.no_grad():\n        for b in va_dl:\n            inert = b[\"inertial\"].to(device); vid = b[\"video\"].to(device)\n            logits = model(inert, vid).float()\n            pr = softmax(logits.cpu().numpy(), -1).mean(axis=1).argmax(1)\n            yt.append(b[\"target\"].numpy()); yp.append(pr)\n    f1 = macro_f1(np.concatenate(yt), np.concatenate(yp), n_classes=N_CLASSES)\n    print(f\"[{tag}] macro-F1 = {f1:.4f}\")\n    return f1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=3)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n\n    from .data import InertialNormalizer\n    normalizer = None  # no normalization (match reference)\n\n    subj = windows[\"sbj_id\"].to_numpy()\n    tr, va, vs = grouped_split(subj, n_val_subjects=4, seed=cfg.data.seed)\n    print(f\"val subjects {sorted(vs)} | train {len(tr)} val {len(va)}\")\n\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=True, null_inertial=False, epochs=args.epochs, tag=\"inertial-only\")\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=False, null_inertial=True, epochs=args.epochs, tag=\"video-only\")\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=False, null_inertial=False, epochs=args.epochs, tag=\"fusion\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "temporal_smooth": "\"\"\"Temporal majority-vote smoothing of test predictions.\n\nThe test windows come from continuous recordings of 4 unseen subjects.  Sorting\neach subject's windows by ``id`` recovers its 1-second time axis; adjacent\nwindows are the same activity ~90%+ of the time.  We reassign each window to the\nmajority class of its +/-k neighborhood (per subject, in time order), correcting\nthe scattered single-window misclassifications that hurt macro-F1 most.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef temporal_order(test_meta: pd.DataFrame):\n    \"\"\"Return an array of ``test_meta`` row indices in per-subject time order.\n\n    Sorts by (sbj_id, id).  Returns (sorted_ids, subject_boundaries) where\n    ``sorted_ids`` are meta row positions in temporal order and\n    ``subject_boundaries`` are the end indices of each subject block.\n    \"\"\"\n    meta = test_meta.reset_index(drop=True)\n    order = meta.sort_values([\"sbj_id\", \"id\"]).index.to_numpy()\n    subj = meta.loc[order, \"sbj_id\"].to_numpy()\n    boundaries = np.flatnonzero(subj[1:] != subj[:-1]) + 1\n    return order, boundaries\n\n\ndef smooth_majority_vote(probs: np.ndarray, meta: pd.DataFrame,\n                         k: int = 2) -> np.ndarray:\n    \"\"\"Majority-vote over +/-k neighbors per subject (in time order).\n\n    ``probs``: (N, n_classes) in the same order as ``meta`` rows.\n    Returns smoothed class ids (N,).\n    \"\"\"\n    preds = probs.argmax(axis=1)\n    order, boundaries = temporal_order(meta)\n\n    # Place predictions in temporal order, per subject block.\n    blocks = np.split(order, boundaries)\n    out = preds.copy()\n    for blk in blocks:\n        seq = preds[blk]                       # temporal class sequence (L,)\n        L = len(seq)\n        if L <= 1:\n            continue\n        smoothed = np.empty(L, dtype=seq.dtype)\n        for i in range(L):\n            lo, hi = max(0, i - k), min(L, i + k + 1)\n            # majority (ties broken by center / lowest class)\n            counts = np.bincount(seq[lo:hi], minlength=seq.max() + 1)\n            smoothed[i] = counts.argmax()\n        out[blk] = smoothed\n    return out\n\n\ndef smooth_by_probability_average(probs: np.ndarray, meta: pd.DataFrame,\n                                  k: int = 2) -> np.ndarray:\n    \"\"\"Average probabilities over +/-k neighbors per subject, then argmax.\"\"\"\n    order, boundaries = temporal_order(meta)\n    blocks = np.split(order, boundaries)\n    out = np.empty(probs.shape[0], dtype=np.int64)\n    for blk in blocks:\n        seq = probs[blk]                       # (L, C)\n        L = len(seq)\n        for i in range(L):\n            lo, hi = max(0, i - k), min(L, i + k + 1)\n            out[blk[i]] = seq[lo:hi].mean(axis=0).argmax()\n    return out",
    "strong_model": "\"\"\"Stronger fusion model for the WEAR challenge.\n\nDesign rationale (based on measured results):\n  * Video carries transferable signal on the real test set (fusion > inertial-only\n    on the leaderboard), so we invest heavily in a deep VideoMAE encoder.\n  * The inertial branch stays lean (scaling inertial capacity overfit the test),\n    using the proven per-sensor multi-scale Inception encoder + normalisation.\n  * Fusion is a learned gated blend + interaction, with a larger classifier.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\n# ---------------------------------------------------------------------------\n# Inception building block (proven, kept lean)\n# ---------------------------------------------------------------------------\nclass SInceptionBlock(nn.Module):\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)\n        out = out + self.skip(x)\n        return self.dropout(F.gelu(self.norm(out)))\n\n\nclass SInertialEncoder(nn.Module):\n    \"\"\"Per-sensor lean Inception encoder: (B,3,50) -> (B,h). Adds magnitude.\"\"\"\n\n    def __init__(self, hidden: int = 192, blocks: int = 3,\n                 channels: int = 128, dropout: float = 0.2):\n        super().__init__()\n        layers = [SInceptionBlock(4, channels)]\n        for _ in range(blocks - 1):\n            layers.append(SInceptionBlock(channels, channels))\n        self.blocks = nn.Sequential(*layers)\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B,3,50)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)          # (B,4,50)\n        h = self.blocks(x)                      # (B,C,50)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)\n        return self.head(pooled)\n\n\n# ---------------------------------------------------------------------------\n# Strong VideoMAE encoder (deep transformer over the 15x768 features)\n# ---------------------------------------------------------------------------\nclass SVideoEncoder(nn.Module):\n    def __init__(self, hidden: int = 256, layers: int = 4, heads: int = 8,\n                 ff_mult: int = 4, dropout: float = 0.15):\n        super().__init__()\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM),\n            nn.Linear(VIDEO_DIM, hidden),\n            nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            hidden, heads, dim_feedforward=hidden * ff_mult, dropout=dropout,\n            activation=\"gelu\", batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(\n            layer, layers, enable_nested_tensor=False\n        )\n        self.attn = nn.Sequential(\n            nn.Linear(hidden, hidden // 2), nn.Tanh(),\n            nn.Linear(hidden // 2, 1),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,15,768)\n        h = self.projection(x) + self.position\n        h = self.transformer(h)                # (B,15,H)\n        scores = self.attn(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\n# ---------------------------------------------------------------------------\n# Strong fusion model\n# ---------------------------------------------------------------------------\nclass StrongFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES,\n                 video_hidden: int = 256, video_layers: int = 4,\n                 video_heads: int = 8, inertial_hidden: int = 192,\n                 inertial_blocks: int = 3, inertial_channels: int = 128,\n                 cls_hidden: int = 384, sensor_dim: int = 16,\n                 dropout: float = 0.2, modality_dropout: float = 0.1):\n        super().__init__()\n        self.inertial_encoder = SInertialEncoder(\n            inertial_hidden, inertial_blocks, inertial_channels, dropout)\n        self.video_encoder = SVideoEncoder(\n            video_hidden, video_layers, video_heads, dropout=dropout)\n        self.sensor_embed = nn.Embedding(N_SENSORS, sensor_dim)\n\n        # Project both modalities to a common fusion dim for gating.\n        fusion_dim = inertial_hidden\n        self.i_proj = nn.Linear(inertial_hidden, fusion_dim)\n        self.v_proj = nn.Linear(video_hidden, fusion_dim)\n        self.sensor_proj = nn.Linear(sensor_dim, sensor_dim)\n\n        gate_in = fusion_dim * 2 + sensor_dim\n        self.gate = nn.Sequential(nn.Linear(gate_in, fusion_dim),\n                                  nn.Sigmoid())\n        cls_in = fusion_dim * 2 + sensor_dim\n        self.classifier = nn.Sequential(\n            nn.Linear(cls_in, cls_hidden),\n            nn.LayerNorm(cls_hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(cls_hidden, n_classes),\n        )\n        self.modality_dropout = modality_dropout\n        self.sensor_dropout = 0.2\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 3, 1)          # (B,4,50,3)\n\n        # Sensor dropout (robust to single-sensor test).\n        if self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inertial.device) >= self.sensor_dropout\n            inertial = inertial * keep[:, :, None, None].float()\n\n        i_feat = self.inertial_encoder(\n            inertial.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        ).reshape(B, N_SENSORS, -1)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        pos = self.sensor_embed.weight[None, :, :].expand(B, N_SENSORS, -1)\n\n        # Project to common fusion dim.\n        i_feat = self.i_proj(i_feat)\n        v_feat = self.v_proj(v_feat)\n        pos = self.sensor_proj(pos)\n\n        if self.training:\n            i_feat, v_feat = self._modality_dropout(\n                i_feat, v_feat, self.modality_dropout)\n\n        gate_in = torch.cat([i_feat, v_feat, pos], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * i_feat + (1 - gate) * v_feat\n        interaction = i_feat * v_feat\n        features = torch.cat([fused, interaction, pos], dim=-1)\n        return self.classifier(features)                   # (B,4,n_classes)\n\n    @staticmethod\n    def _modality_dropout(i_feat, v_feat, p):\n        if p <= 0:\n            return i_feat, v_feat\n        scale = 1.0 / (1.0 - p)\n        drop_i = (torch.rand(i_feat.shape[:2], device=i_feat.device) < p)[..., None]\n        drop_v = (torch.rand(v_feat.shape[:2], device=v_feat.device) < p)[..., None]\n        i_feat = torch.where(drop_i, torch.zeros_like(i_feat), i_feat * scale)\n        v_feat = torch.where(drop_v, torch.zeros_like(v_feat), v_feat * scale)\n        return i_feat, v_feat",
    "run_strong": "\"\"\"Train the strong fusion model and produce a test submission.\n\nUses our verified data pipeline (segment windowing, per-sensor NaN handling)\nwith global z-score normalisation.  The video branch is deep (the transferable\nmodality on the real test set); the inertial branch stays lean (scaling it\noverfit the test).  Trains ``seeds`` models and averages their probabilities.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .strong_model import StrongFusionModel\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag=\"\", seed=0,\n           label_smooth=0.05, warmup_epochs=1, grad_norm=5.0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer):\n    from . import inference as inf\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device)\n            vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--lr\", type=float, default=6e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=4)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-layers\", type=int, default=4)\n    ap.add_argument(\"--video-hidden\", type=int, default=256)\n    ap.add_argument(\"--video-heads\", type=int, default=8)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"strong\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    print(\"global normalisation ON\")\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        model = StrongFusionModel(\n            video_hidden=args.video_hidden, video_layers=args.video_layers,\n            video_heads=args.video_heads).to(device)\n        nparam = sum(p.numel() for p in model.parameters())\n        print(f\"seed {seed}: params={nparam/1e6:.2f}M\")\n        _train(model, dl, args.epochs, args.lr, 1e-3, device, use_amp,\n               tag=f\"strong-{seed}\", seed=seed)\n        ids, probs = _predict_test(model, cfg.paths, device, use_amp, normalizer)\n        ref_ids = ids\n        acc = probs if acc is None else acc + probs\n\n    blended = acc / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, blended, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "features": "\"\"\"Fast, vectorised feature engineering for the WEAR challenge.\n\nBased on the prior WEAR winners (FAME: frequency-domain features; 1st winner:\nrich features + gradient boosting).  All feature extraction is vectorised over\nthe batch so it runs in seconds, not hours.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nN_ACC_FEATS = 19  # per axis/magnitude: 13 time + 6 frequency\n\n\ndef _freq_features_vect(x: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"x: (..., N) zero-mean signals -> (..., 8) freq features. Vectorised.\"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    x = x - x.mean(axis=-1, keepdims=True)\n    n = x.shape[-1]\n    mag = np.abs(np.fft.rfft(x, axis=-1))          # (..., n//2+1)\n    freqs = np.fft.rfftfreq(n, 1.0 / fs)\n    power = mag ** 2\n    total = power.sum(axis=-1, keepdims=True)\n    total = np.where(total < 1e-12, 1.0, total)\n    p = power / total\n\n    dom_freq = freqs[power.argmax(axis=-1)]\n    centroid = (freqs[None, :] * power).sum(axis=-1) / total[..., 0]\n    entropy = -np.sum(p * np.log(p + 1e-12), axis=-1)\n\n    low = power[..., freqs < 2].sum(axis=-1) / total[..., 0]\n    mid = power[..., (freqs >= 2) & (freqs < 8)].sum(axis=-1) / total[..., 0]\n    high = power[..., freqs >= 8].sum(axis=-1) / total[..., 0]\n\n    # spectral rolloff (95% energy) and spectral flux\n    cum = np.cumsum(p, axis=-1)\n    rolloff_idx = np.argmax(cum >= 0.95, axis=-1)\n    rolloff = freqs[rolloff_idx]\n    # spectral flux (first-order difference of magnitude)\n    flux = np.mean(np.abs(np.diff(p, axis=-1)), axis=-1)\n\n    return np.stack([dom_freq, centroid, entropy, low, mid, high,\n                     rolloff, flux], axis=-1)\n\n\ndef _time_features_vect(x: np.ndarray) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 13) time features. Vectorised, NaN-safe.\"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    n = x.shape[-1]\n    m = x.mean(axis=-1)\n    # stable std\n    s = np.sqrt(np.mean((x - m[..., None]) ** 2, axis=-1) + 1e-12)\n    rms = np.sqrt(np.mean(x ** 2, axis=-1) + 1e-12)\n    xmin = x.min(axis=-1)\n    xmax = x.max(axis=-1)\n    ptp = xmax - xmin\n    median = np.median(x, axis=-1)\n    # skew / kurtosis (central moments, NaN-safe)\n    zm = x - m[..., None]\n    z2 = np.mean(zm ** 2, axis=-1) + 1e-12\n    skew = np.mean(zm ** 3, axis=-1) / (z2 ** 1.5)\n    kurt = np.mean(zm ** 4, axis=-1) / (z2 ** 2) - 3.0\n    # zero crossings & mean crossings (per sample)\n    zc = (np.sign(zm[..., 1:]) * np.sign(zm[..., :-1]) < 0).sum(axis=-1) / max(1, n - 1)\n    mcr = (np.sign(x[..., 1:] - m[..., None]) * np.sign(x[..., :-1] - m[..., None]) < 0).sum(axis=-1) / max(1, n - 1)\n    # autocorr lag-1\n    xm = x - m[..., None]\n    num = np.mean(xm[..., :-1] * xm[..., 1:], axis=-1)\n    ar1 = num / (np.mean(xm[..., :-1] ** 2, axis=-1) + 1e-12)\n    # jerk proxy (std of first difference)\n    jerk = np.std(np.diff(x, axis=-1), axis=-1)\n    td = np.stack([m, s, rms, xmin, xmax, ptp, median, skew, kurt,\n                   zc, mcr, ar1, jerk], axis=-1)\n    return np.nan_to_num(td, nan=0.0, posinf=0.0, neginf=0.0)\n\n\ndef _axis_features_vect(x: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 21) features. Vectorised (13 time + 8 freq).\"\"\"\n    td = _time_features_vect(x)\n    fr = _freq_features_vect(x, fs)\n    return np.concatenate([td, fr], axis=-1)\n\n\ndef _subwindow_features_vect(x: np.ndarray) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 4) sub-window trend features.\n\n    Splits the window in half and compares summary stats between the two\n    halves to capture temporal structure/drift.\n    \"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    n = x.shape[-1]\n    half = max(1, n // 2)\n    x0 = x[..., :half]\n    x1 = x[..., half:2 * half]\n    m0 = x0.mean(axis=-1)\n    m1 = x1.mean(axis=-1)\n    s0 = x0.std(axis=-1)\n    s1 = x1.std(axis=-1)\n    # energy ratio first/second half, and trend of mean/std\n    e0 = np.mean(x0 ** 2, axis=-1)\n    e1 = np.mean(x1 ** 2, axis=-1)\n    trend_mean = m1 - m0\n    trend_std = s1 - s0\n    energy_ratio = e1 / (e0 + 1e-9)\n    # slope of linear fit (normalized)\n    t = np.linspace(0, 1, n)\n    denom = np.sum((t - t.mean()) ** 2)\n    slope = np.sum((x - x.mean(axis=-1, keepdims=True)) * (t - t.mean()), axis=-1) / denom\n    return np.stack([trend_mean, trend_std, energy_ratio, slope], axis=-1)\n\n\ndef extract_sensor_correlations(windows: np.ndarray) -> np.ndarray:\n    \"\"\"windows: (N, 50, 4, 3) -> (N, 12) pairwise sensor magnitude correlations.\n\n    For each pair of sensors, correlation of their magnitude time-series.\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    N, T, S, A = w.shape\n    mag = np.linalg.norm(w, axis=-1)              # (N,T,S)\n    out = np.zeros((N, S * (S - 1) // 2), dtype=np.float64)\n    k = 0\n    for i in range(S):\n        for j in range(i + 1, S):\n            a = mag[:, :, i] - mag[:, :, i].mean(axis=-1, keepdims=True)\n            b = mag[:, :, j] - mag[:, :, j].mean(axis=-1, keepdims=True)\n            denom = np.sqrt((a ** 2).sum(-1) * (b ** 2).sum(-1)) + 1e-9\n            out[:, k] = (a * b).sum(-1) / denom\n            k += 1\n    return out\n\n\ndef fit_inertial_pca(F_base: np.ndarray, n_components: int = 32,\n                     seed: int = 42):\n    \"\"\"Fit PCA on base inertial feature matrix for feature augmentation.\"\"\"\n    Xc = F_base - F_base.mean(axis=0)\n    from sklearn.decomposition import TruncatedSVD\n    svd = TruncatedSVD(n_components=min(n_components, Xc.shape[1] - 1),\n                       random_state=seed)\n    svd.fit(Xc)\n    return Xc.mean(axis=0), svd.components_.T\n\n\ndef apply_inertial_pca(F_base: np.ndarray, center, components) -> np.ndarray:\n    return (F_base - center) @ components\n\n\ndef _lowpass(x, alpha=0.1):\n    \"\"\"Exponential moving average along last axis (approx gravity).\"\"\"\n    out = np.empty_like(x)\n    acc = x[..., 0].copy()\n    out[..., 0] = acc\n    for t in range(1, x.shape[-1]):\n        acc = alpha * x[..., t] + (1 - alpha) * acc\n        out[..., t] = acc\n    return out\n\n\ndef _gravity_body_decompose(a):\n    \"\"\"a: (..., T, 3) -> a_parallel (...,T,1), a_perp (...,T,1), gravity_norm (...,T,1).\"\"\"\n    a = np.asarray(a, dtype=np.float64)\n    g = _lowpass(a, alpha=0.15)                      # gravity estimate\n    gn = np.linalg.norm(g, axis=-1, keepdims=True) + 1e-9\n    ghat = g / gn\n    a_par = (a * ghat).sum(axis=-1, keepdims=True)   # projection along gravity\n    a_perp = np.linalg.norm(a - a_par * ghat, axis=-1, keepdims=True)\n    return a_par, a_perp, gn\n\n\ndef _rotation_invariant_stats(a):\n    \"\"\"a: (..., T, 3) -> rotation-invariant per-sample stats (..., T, 6).\"\"\"\n    x = a[..., 0]; y = a[..., 1]; z = a[..., 2]\n    return np.stack([x * x + y * y, y * y + z * z, x * x + z * z,\n                     x * y, x * z, y * z], axis=-1)\n\n\ndef extract_orientation_invariant(windows: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"Orientation-invariant features per sensor (vectorised).\n\n    windows: (N,T,3) test or (N,T,4,3) train. Returns (N, F) of added features.\n    Includes: magnitude (already partly in base, expanded here), gravity/body-frame\n    decomposition, rotation-invariant products, covariance eigenvalues.\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    if w.ndim == 4:                       # (N,T,4,3)\n        N, T, S, A = w.shape\n        cols = []\n        for s in range(S):\n            cols.append(_orientation_invariant_one(w[:, :, s], fs))\n        return np.concatenate(cols, axis=-1)\n    else:                                 # (N,T,3)\n        return _orientation_invariant_one(w, fs)\n\n\ndef _orientation_invariant_one(a, fs=50.0):\n    \"\"\"a: (N,T,3) -> (N, F) orientation-invariant features for one sensor.\"\"\"\n    mag = np.linalg.norm(a, axis=-1)                       # (N,T)\n    # magnitude derivatives\n    dm = np.gradient(mag, axis=-1)\n    d2m = np.gradient(dm, axis=-1)\n    a_par, a_perp, gn = _gravity_body_decompose(a)         # each (N,T,1)\n    ri = _rotation_invariant_stats(a)                      # (N,T,6)\n    # covariance eigenvalues (per window) of raw 3 axes\n    N = a.shape[0]\n    eig = np.zeros((N, 3), dtype=np.float64)\n    for i in range(N):\n        c = np.cov(a[i].T)                                  # 3x3\n        ev = np.linalg.eigvalsh(c)\n        eig[i] = ev[::-1]\n    # stack per-sample feature streams, then time-features each\n    streams = np.concatenate([\n        mag[:, :, None], dm[:, :, None], d2m[:, :, None],\n        a_par, a_perp, gn, ri], axis=-1)                    # (N,T,12)\n    feats = []\n    for k in range(streams.shape[-1]):\n        feats.append(_time_features_vect(streams[:, :, k]))\n    feats.append(eig)                                        # 3 eigenvalues\n    return np.concatenate(feats, axis=-1)\n\n\ndef extract_acc_features_vect(windows: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"Vectorised feature extraction.\n\n    windows: (N, T, 3) single-sensor (test) or (N, T, 4, 3) all-sensors (train).\n    Returns (N, F).  For (N,T,4,3) we concatenate per-sensor features (each\n    sensor = 3 axes + magnitude).\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    if w.ndim == 4:  # (N,T,4,3)\n        N, T, S, A = w.shape\n        cols = []\n        for s in range(S):\n            for a in range(A):\n                cols.append(_axis_features_vect(w[:, :, s, a], fs))\n                cols.append(_subwindow_features_vect(w[:, :, s, a]))\n            mag = np.linalg.norm(w[:, :, s], axis=-1)\n            cols.append(_axis_features_vect(mag, fs))\n            cols.append(_subwindow_features_vect(mag))\n        return np.concatenate(cols, axis=-1)\n    else:  # (N,T,3)\n        N, T, A = w.shape\n        cols = []\n        for a in range(A):\n            cols.append(_axis_features_vect(w[:, :, a], fs))\n            cols.append(_subwindow_features_vect(w[:, :, a]))\n        mag = np.linalg.norm(w, axis=-1)\n        cols.append(_axis_features_vect(mag, fs))\n        cols.append(_subwindow_features_vect(mag))\n        return np.concatenate(cols, axis=-1)\n\n\ndef extract_video_features_vect(videos: np.ndarray,\n                                n_components: int = 64) -> np.ndarray:\n    \"\"\"videos: (N, 15, 768) -> (N, 2*n_components) pooled features.\"\"\"\n    v = np.asarray(videos, dtype=np.float64)\n    mean = v.mean(axis=1)                # (N,768)\n    std = v.std(axis=1)\n    feats = np.concatenate([mean, std], axis=-1)  # (N,1536)\n    rng = np.random.RandomState(0)\n    proj = rng.randn(1536, n_components) / np.sqrt(1536)\n    return feats @ proj",
    "feature_data": "\"\"\"Build train/test feature matrices for the boosting model.\n\nTrain windows have all 4 sensors; test windows have a single sensor location.\nTo keep feature dimensions consistent, we treat each sensor as a \"view\": every\nwindow yields one feature vector per sensor, tagged with a one-hot sensor\nlocation.  At test we use the present sensor's vector (matching its location).\n\nFeature extraction is fully vectorised (see features.py).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import N_SENSORS, Paths\nfrom .data import (\n    build_dataset_windows,\n    load_train_recording,\n    subsample_windows,\n    train_stems,\n    INERTIAL_WINDOW,\n    VIDEO_WINDOW,\n    VIDEO_FRAME_OFFSET,\n)\nfrom .features import (\n    extract_acc_features_vect,\n    extract_orientation_invariant,\n    extract_video_features_vect,\n)\n\n# Base per-sensor features (3 axes + magnitude, 25 each = 100) + orientation-invariant (159).\n_BASE_PER_SENSOR = 4 * (21 + 4)      # 100\n_ORIENT_PER_SENSOR = 159\nPER_SENSOR_FEATS = _BASE_PER_SENSOR + _ORIENT_PER_SENSOR  # 259\nSENSOR_FEAT_DIM = PER_SENSOR_FEATS + N_SENSORS  # + one-hot location\n\n\ndef _build_train_features(paths: Paths, stride: int, max_per_class, seed,\n                          video_proj_dim, video_transform=None,\n                          use_orientation=False):\n    \"\"\"Recording-by-recording, vectorised feature build (fast, low memory).\"\"\"\n    from .data import build_dataset_windows, load_train_recording, subsample_windows\n    windows = build_dataset_windows(paths, stride)\n    windows = subsample_windows(windows, max_per_class, seed)\n\n    all_F, all_V, all_y = [], [], []\n    for rec in windows[\"rec\"].unique():\n        r = load_train_recording(paths, rec)   # loads inertial + video (mmap)\n        sub = windows[windows[\"rec\"] == rec]\n        starts = sub[\"inertial_start\"].to_numpy()\n        vstarts = sub[\"video_start\"].to_numpy()\n        ys = sub[\"target\"].to_numpy()\n\n        # Vectorised window gather for this recording.\n        idx = starts[:, None] + np.arange(INERTIAL_WINDOW)[None, :]   # (M,50)\n        inerts = r.inertial[idx].astype(np.float64)                    # (M,50,4,3)\n        inerts = np.nan_to_num(inerts)\n        vidx = vstarts[:, None] + np.arange(VIDEO_WINDOW)[None, :]     # (M,15)\n        vids = np.asarray(r.video[vidx], dtype=np.float64)             # (M,15,768)\n        vids = np.nan_to_num(vids)\n\n        base = extract_acc_features_vect(inerts)\n        if use_orientation:\n            base = np.concatenate([base, extract_orientation_invariant(inerts)], axis=-1)\n        all_F.append(base)\n        if video_transform is not None:\n            center, comps = video_transform\n            all_V.append(transform_video_pca(vids, center, comps))\n        else:\n            all_V.append(extract_video_features_vect(vids, video_proj_dim))\n        all_y.append(ys)\n\n    F = np.concatenate(all_F, axis=0)   # (N, 400)\n    V = np.concatenate(all_V, axis=0)   # (N, V)\n    y = np.concatenate(all_y, axis=0)   # (N,)\n    return F, V, y\n\n\ndef build_train_features(paths: Paths, stride: int = 25,\n                         max_per_class: int | None = None,\n                         seed: int = 42, video_proj_dim: int = 64,\n                         video_transform=None, use_orientation=False):\n    \"\"\"Build train feature matrix + labels.\n\n    Returns (X (M, F), y (M,), video_X (M, V), sensor_onehot (M, S)).\n    Each window -> N_SENSORS rows (one per sensor view).\n    \"\"\"\n    F, V, ys = _build_train_features(paths, stride, max_per_class, seed,\n                                     video_proj_dim, video_transform,\n                                     use_orientation)\n    N = F.shape[0]\n    per_sensor = F.shape[1] // N_SENSORS\n    F = F.reshape(N, N_SENSORS, per_sensor)\n\n    X = _interleaved(F, N)                 # (N*4, per_sensor)\n    y = np.repeat(ys, N_SENSORS)\n    vid = np.tile(V, (N_SENSORS, 1))\n    sensor = np.repeat(np.eye(N_SENSORS)[None], N, 0).reshape(N * N_SENSORS, -1)\n    return X, y, vid, sensor\n\n\ndef _gather_video_arrays(paths: Paths, stride: int, max_per_class, seed,\n                         max_windows: int = 30000):\n    \"\"\"Return video (N,15,768) for a SUBSET of training windows (for PCA fit).\n\n    Subsamples at the window-metadata level so we never hold all windows' video\n    in memory (the full set would be tens of GB).\n    \"\"\"\n    from .data import build_dataset_windows, load_train_recording, subsample_windows\n    windows = build_dataset_windows(paths, stride)\n    windows = subsample_windows(windows, max_per_class, seed)\n    if max_windows is not None and len(windows) > max_windows:\n        rng = np.random.RandomState(seed)\n        windows = windows.sample(max_windows, random_state=seed)\n    vids = []\n    cur_rec = None; cur_r = None\n    for _, row in windows.iterrows():\n        rec = row[\"rec\"]\n        if rec != cur_rec:\n            cur_r = load_train_recording(paths, rec); cur_rec = rec\n        v0 = int(row[\"video_start\"])\n        vids.append(np.nan_to_num(np.asarray(cur_r.video[v0:v0 + VIDEO_WINDOW], dtype=np.float64)))\n    return np.array(vids)\n\n\ndef fit_video_pca(paths: Paths, stride: int = 25, max_per_class: int = None,\n                  seed: int = 42, n_components: int = 64,\n                  max_fit_samples: int = 30000):\n    \"\"\"Fit PCA on pooled train video features (mean+std of each 768-dim frame).\n\n    Fits on a subsample to keep it fast and memory-safe.\n    \"\"\"\n    vids = _gather_video_arrays(paths, stride, max_per_class, seed,\n                                max_windows=max_fit_samples)\n    mean = vids.mean(axis=1)               # (N,768)\n    std = vids.std(axis=1)\n    feats = np.concatenate([mean, std], axis=1)   # (N,1536)\n    center = feats.mean(axis=0)\n    Xc = feats - center\n    try:\n        from sklearn.decomposition import TruncatedSVD\n        svd = TruncatedSVD(n_components=n_components, random_state=seed)\n        svd.fit(Xc)\n        components = svd.components_.T        # (1536, n_components)\n    except Exception:\n        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)\n        components = Vt[:n_components].T\n    return center, components\n\n\ndef transform_video_pca(videos: np.ndarray, center, components) -> np.ndarray:\n    v = np.asarray(videos, dtype=np.float64)\n    mean = v.mean(axis=1)\n    std = v.std(axis=1)\n    feats = np.concatenate([mean, std], axis=1)\n    return (feats - center) @ components\n\n\ndef _interleaved(F, N):\n    \"\"\"F (N,4,76) -> X (N*4, 76) interleaved by (window, sensor).\"\"\"\n    return F.reshape(N * N_SENSORS, -1)\n\n\ndef build_test_features(paths: Paths, video_proj_dim: int = 64,\n                        video_transform=None, use_orientation=False):\n    \"\"\"Build test feature matrix aligned to test meta row order.\n\n    Returns (X (N, F), ids, sensor_onehot (N, S), video_X (N, V)).\n    \"\"\"\n    from .data import build_test_dataset\n    inertial, video, sensor_ids, ids, _ = build_test_dataset(paths)\n    inertial = np.nan_to_num(np.asarray(inertial, dtype=np.float64))  # (N,50,3)\n    video = np.nan_to_num(np.asarray(video, dtype=np.float64))        # (N,15,768)\n\n    F = extract_acc_features_vect(inertial)          # (N,100) single sensor\n    if use_orientation:\n        F = np.concatenate([F, extract_orientation_invariant(inertial)], axis=-1)  # (N,259)\n    if video_transform is not None:\n        center, comps = video_transform\n        V = transform_video_pca(video, center, comps)   # (N,V)\n    else:\n        V = extract_video_features_vect(video, video_proj_dim)  # (N,V)\n\n    sensor_oh = np.zeros((len(ids), N_SENSORS))\n    sensor_oh[np.arange(len(ids)), sensor_ids] = 1.0\n    # Xt is the sensor features only (matches train X); caller adds video+sensor.\n    return F, ids, sensor_oh, V",
    "run_boost": "\"\"\"Gradient-boosting classifier on engineered features (proven WEAR winner).\n\nRich time/frequency features per sensor view + pooled VideoMAE features, trained\nwith LightGBM (multi-class), then test predictions.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features,\n    build_test_features,\n    SENSOR_FEAT_DIM,\n)\nfrom .inference import write_submission\n\n\ndef _check_video_alignment(paths):\n    \"\"\"Cheap diagnostic: compare train vs test video feature scale (mean/std of values).\"\"\"\n    from .data import load_train_recording, train_stems, build_test_dataset\n    train_vals = []\n    for stem in train_stems(paths)[:4]:\n        r = load_train_recording(paths, stem)\n        v = np.asarray(r.video)\n        train_vals.append(v.ravel())\n    train_all = np.concatenate(train_vals)\n    _, test_video, _, _, _ = build_test_dataset(paths)\n    tv = np.asarray(test_video).ravel()\n    print(f\"[video-align] train mean={train_all.mean():.4f} std={train_all.std():.4f} | \"\n          f\"test mean={tv.mean():.4f} std={tv.std():.4f}\")\n    if train_all.std() > 0 and abs(train_all.std() / (tv.std() + 1e-8) - 1.0) > 0.5:\n        print(\"NOTE: train/test video std differ >50% - check alignment\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=None)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--max-depth\", type=int, default=-1)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--use-video\", type=int, default=1)\n    ap.add_argument(\"--video-pca\", type=int, default=1,\n                    help=\"1 to use PCA video features (fit on train), 0 for random projection\")\n    ap.add_argument(\"--ensemble\", type=int, default=1,\n                    help=\"number of diverse boosters to average (ensemble)\")\n    ap.add_argument(\"--seed\", type=int, default=42)\n    args = ap.parse_args()\n\n    cfg = Config()\n    device_note = \"cpu (boosting)\"\n\n    _check_video_alignment(cfg.paths)\n\n    # Fit video PCA on train (optional but recommended).\n    video_transform = None\n    if args.use_video and args.video_pca:\n        from .feature_data import fit_video_pca\n        print(\"fitting video PCA...\", flush=True)\n        video_transform = fit_video_pca(\n            cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n            seed=args.seed, n_components=args.video_proj_dim)\n        print(\"video PCA fit done\", flush=True)\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=args.seed, video_proj_dim=args.video_proj_dim,\n        video_transform=video_transform)\n    print(f\"train samples (per-sensor views) = {X.shape}, classes={np.unique(y).tolist()}\", flush=True)\n\n    if args.use_video:\n        F = np.concatenate([X, vid, sensor], axis=1)\n    else:\n        F = X\n    print(f\"feature matrix: {F.shape}\", flush=True)\n\n    import lightgbm as lgb\n    # Class weights (inverse frequency) for macro-F1 oriented training.\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n\n    # Ensemble of boosters with diverse feature subsets + seeds (boosting ensemble).\n    n_ensemble = max(1, args.ensemble)\n    import lightgbm as lgb\n    test_preds = []\n    for bi in range(n_ensemble):\n        seed = args.seed + bi\n        col_frac = 0.7 + 0.2 * (bi % 2)   # alternate 0.7 / 0.9\n        params = dict(\n            objective=\"multiclass\", num_class=N_CLASSES,\n            n_estimators=args.n_estimators, learning_rate=args.lr,\n            num_leaves=args.num_leaves, max_depth=args.max_depth,\n            subsample=0.8, colsample_bytree=col_frac, reg_lambda=1.0,\n            min_child_samples=30, n_jobs=8, random_state=seed,\n            verbose=-1,\n        )\n        model = None\n        try:\n            params[\"device_type\"] = \"gpu\"\n            model = lgb.LGBMClassifier(**params)\n            model.fit(F, y, sample_weight=sw)\n            print(f\"LightGBM[{bi}] trained on GPU\", flush=True)\n        except Exception as e:\n            print(f\"GPU LightGBM failed ({e}); using CPU\", flush=True)\n            params[\"device_type\"] = \"cpu\"\n            model = lgb.LGBMClassifier(**params)\n            model.fit(F, y, sample_weight=sw)\n            print(f\"LightGBM[{bi}] trained on CPU\", flush=True)\n\n        print(\"building test features...\", flush=True)\n        Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths,\n                                                       args.video_proj_dim,\n                                                       video_transform)\n        if args.use_video:\n            Ft = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n        else:\n            Ft = Xt\n        print(f\"test feature matrix: {Ft.shape}\", flush=True)\n        p = model.predict_proba(Ft)\n        if p.shape[1] < N_CLASSES:\n            pad = np.zeros((p.shape[0], N_CLASSES - p.shape[1]))\n            p = np.concatenate([p, pad], axis=1)\n        test_preds.append(p)\n\n    probs = np.mean(test_preds, axis=0)         # average ensemble\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "fame_model": "\"\"\"FAME-style feature-augmented multi-view neural network.\n\nAdapts the prior WEAR winner (FAME) for the current test setup:\n  * Feature augmentation: append frequency-domain features to the raw inertial\n    channels (each sensor gets raw 3 axes + magnitude + per-axis freq features).\n  * Channel-wise random sign-flipping augmentation (FAME's main driver of\n    generalization).\n  * Multi-view: a shared per-sensor encoder produces per-sensor embeddings\n    (views); a sensor-location embedding conditions the view; predictions are\n    made per sensor and the present sensor is used at test time.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\ndef _append_freq_channels(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"x: (B, N_SENSORS, 3, T) -> (B, N_SENSORS, 3+2, T).\n\n    Appends 2 frequency-domain proxy features (zero-crossing rate and\n    differencing energy) as constant-over-time channels, per sensor.\n    \"\"\"\n    B, S, A, T = x.shape\n    xm = x - x.mean(dim=-1, keepdim=True)\n    zc = ((xm[:, :, :, 1:] * xm[:, :, :, :-1]) < 0).float().mean(dim=-1)   # (B,S,3)\n    d = (xm[:, :, :, 1:] - xm[:, :, :, :-1]).abs().mean(dim=-1)            # (B,S,3)\n    extra = torch.stack([zc.mean(dim=-1), d.mean(dim=-1)], dim=-1)         # (B,S,2)\n    extra = extra[:, :, :, None].expand(B, S, 2, T)                        # (B,S,2,T)\n    return torch.cat([x, extra], dim=2)                                    # (B,S,5,T)\n\n\nclass FAMEInceptionBlock(nn.Module):\n    def __init__(self, cin, out, dropout=0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(nn.MaxPool1d(3, 1, 1),\n                                  nn.Conv1d(cin, bottleneck, 1, bias=False))\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False) if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        br = [m(b) for m in self.branches] + [self.pool(x)]\n        out = torch.cat(br, dim=1) + self.skip(x)\n        return self.drop(F.gelu(self.norm(out)))\n\n\nclass FAMEVideoEncoder(nn.Module):\n    def __init__(self, hidden=192, layers=2, heads=4):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(VIDEO_DIM), nn.Linear(VIDEO_DIM, hidden), nn.GELU())\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden); nn.init.trunc_normal_(pos, std=0.02)\n        self.pos = nn.Parameter(pos)\n        self.enc = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(hidden, heads, hidden * 4, 0.15, \"gelu\",\n                                       batch_first=True, norm_first=True),\n            layers, enable_nested_tensor=False)\n        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))\n\n    def forward(self, x):  # (B,15,768)\n        h = self.proj(x) + self.pos\n        h = self.enc(h)\n        att = h.mean(1)\n        return self.head(torch.cat([att, h.mean(1)], dim=-1))\n\n\nclass FAMEViewModel(nn.Module):\n    \"\"\"Shared per-sensor encoder + view-specific branches + sensor conditioning.\"\"\"\n\n    def __init__(self, n_classes=N_CLASSES, hidden=192, channels=128,\n                 video_layers=2, sensor_dim=16, dropout=0.2):\n        super().__init__()\n        self.inertial_encoder = nn.Sequential(\n            FAMEInceptionBlock(6, channels),\n            FAMEInceptionBlock(channels, channels),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden), nn.LayerNorm(hidden),\n            nn.GELU(), nn.Dropout(dropout))\n        self.video_encoder = FAMEVideoEncoder(hidden=hidden, layers=video_layers)\n        self.sensor_embed = nn.Embedding(N_SENSORS, sensor_dim)\n        gate_in = hidden + hidden + sensor_dim\n        self.gate = nn.Sequential(nn.Linear(gate_in, hidden), nn.Sigmoid())\n        self.classifier = nn.Sequential(\n            nn.Linear(hidden * 2 + sensor_dim, hidden),\n            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, n_classes))\n        self.sensor_dropout = 0.2\n        self.sign_flip = True\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        inert = inertial.permute(0, 2, 3, 1)            # (B,4,50,3)\n        if self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inert.device) >= self.sensor_dropout\n            inert = inert * keep[:, :, None, None].float()\n        # per-sensor: (B*4, 3, 50) -> add magnitude -> (B*4,4,50)\n        x = inert.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)                  # (B*4,4,50)\n        # sign-flip augmentation\n        if self.training and self.sign_flip:\n            flip = (torch.rand(x.shape[0], 1, 1, device=x.device) < 0.5).float() * 2 - 1\n            x = x * flip\n        # append frequency features to the 3 raw axes -> (B*4, 5, 50)\n        x3 = x[:, :3]                                   # (B*4,3,50)\n        xf = _append_freq_channels(\n            x3.reshape(B, N_SENSORS, 3, INERTIAL_WINDOW)\n        ).reshape(B * N_SENSORS, 5, INERTIAL_WINDOW)\n        # combine: 3 axes + 2 freq + magnitude = 6 channels\n        x = torch.cat([xf, x[:, 3:4]], dim=1)           # (B*4,6,50)\n        h = self.inertial_encoder(x)\n        pooled = torch.cat([h.mean(2), h.amax(2)], dim=1)\n        i_feat = self.head(pooled).reshape(B, N_SENSORS, -1)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        pos = self.sensor_embed.weight[None].expand(B, N_SENSORS, -1)\n        gate = torch.sigmoid(self.gate(torch.cat([i_feat, v_feat, pos], dim=-1)))\n        fused = gate * i_feat + (1 - gate) * v_feat\n        inter = i_feat * v_feat\n        logits = self.classifier(torch.cat([fused, inter, pos], dim=-1))\n        return logits",
    "run_fame": "\"\"\"Train the FAME-style feature-augmented multi-view model and submit.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .fame_model import FAMEViewModel\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag=\"\", seed=0,\n           label_smooth=0.05, warmup_epochs=1, grad_norm=5.0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer):\n    from . import inference as inf\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device); vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=10)\n    ap.add_argument(\"--lr\", type=float, default=7e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=4)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"fame\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        model = FAMEViewModel().to(device)\n        nparam = sum(p.numel() for p in model.parameters())\n        print(f\"seed {seed}: params={nparam/1e6:.2f}M\")\n        _train(model, dl, args.epochs, args.lr, 1e-3, device, use_amp,\n               tag=f\"fame-{seed}\", seed=seed)\n        ids, probs = _predict_test(model, cfg.paths, device, use_amp, normalizer)\n        ref_ids = ids\n        acc = probs if acc is None else acc + probs\n\n    blended = acc / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, blended, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_ensemble": "\"\"\"Fast diverse-model ensemble + Test-Time Augmentation (GPU).\n\nTrains 3 diverse models that all run on GPU (fast):\n  * FAME-style multi-view neural (freq channels + sign-flip + video)\n  * Strong fusion neural (deep video transformer + lean inertial + video)\n  * Small MLP on engineered features (sub-window/freq + pooled video)\n\nApplies Test-Time Augmentation (noise + feature flip) and averages predictions.\nDesigned to complete in ~30-45 min on a single GPU.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import InertialNormalizer, WearTrainDataset, build_dataset_windows, subsample_windows\nfrom .fame_model import FAMEViewModel\nfrom .strong_model import StrongFusionModel\nfrom .inference import WearTestDataset, write_submission\n\n\ndef _collate_loader(ds, bs, shuffle, worker_seed):\n    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=4,\n                      pin_memory=True,\n                      worker_init_fn=lambda w: np.random.seed(worker_seed + w),\n                      persistent_workers=True)\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag, seed):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    crit = nn.CrossEntropyLoss(label_smoothing=0.05)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warm = 1 * len(dl)\n\n    def ll(step):\n        if step < warm:\n            return step / max(1, warm)\n        p = (step - warm) / max(1, total - warm)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r, nb = 0.0, 0\n        for b in dl:\n            inert = b[\"inertial\"].to(device)\n            vid = b[\"video\"].to(device)\n            tgt = b[\"target\"].to(device)\n            valid = b[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = crit(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item(); nb += 1\n        print(f\"[{tag}] ep {ep+1}/{epochs} loss={r/max(1,nb):.4f}\", flush=True)\n    return model\n\n\ndef _predict_tta(model, paths, device, use_amp, normalizer, n_tta=3):\n    \"\"\"Predict test with TTA (noise + sign-flip on inertial, averaged).\"\"\"\n    model.eval()\n    all_probs = None\n    all_ids = None\n    for t in range(n_tta):\n        ds = WearTestDataset(paths, normalizer=normalizer)\n        dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                        collate_fn=_test_collate)\n        probs = []\n        ids_out = None\n        with torch.inference_mode():\n            for inert, vid, sensor_id, ids in dl:\n                inert = inert.to(device); vid = vid.to(device); sensor_id = sensor_id.to(device)\n                inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)  # (B,50,4,3)\n                present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n                present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n                inert = inert * present[:, None, :, None]\n                if t > 0:\n                    # TTA: small noise + channel sign flip (on present sensors)\n                    noise = torch.randn_like(inert) * 0.05\n                    flip = (torch.rand(inert.shape[0], 1, N_SENSORS, 1, device=device) < 0.1).float() * 2 - 1\n                    inert = inert * flip + noise\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = model(inert, vid)\n                p = torch.softmax(logits.float(), dim=-1)\n                p = p[torch.arange(p.shape[0], device=device), sensor_id, :].cpu().numpy()\n                probs.append(p)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        p = np.concatenate(probs)\n        o = np.argsort(ids_out)\n        p = p[o]\n        all_probs = p if all_probs is None else all_probs + p\n        all_ids = ids_out if all_ids is None else np.sort(ids_out)\n    return all_ids, all_probs / n_tta\n\n\ndef _test_collate(batch):\n    import torch\n    inert = torch.as_tensor(np.stack([b[\"inertial\"] for b in batch]))\n    vid = torch.as_tensor(np.stack([b[\"video\"] for b in batch]))\n    sid = torch.as_tensor([b[\"sensor_id\"] for b in batch])\n    ids = torch.as_tensor([b[\"id\"] for b in batch])\n    return inert, vid, sid, ids\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=6)\n    ap.add_argument(\"--fame-seeds\", type=int, default=2)\n    ap.add_argument(\"--strong-seeds\", type=int, default=2)\n    ap.add_argument(\"--tta\", type=int, default=3)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\", flush=True)\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = _collate_loader(ds, 256, True, cfg.data.seed)\n\n    # ---- Train diverse models ----\n    ensemble_probs = None\n    ids = None\n\n    for s in range(args.fame_seeds):\n        m = FAMEViewModel().to(device)\n        _train(m, dl, args.epochs, 7e-4, 1e-3, device, use_amp, f\"fame{s}\", cfg.data.seed + s)\n        i, p = _predict_tta(m, cfg.paths, device, use_amp, normalizer, args.tta)\n        ids = i\n        ensemble_probs = p if ensemble_probs is None else ensemble_probs + p\n        del m; torch.cuda.empty_cache()\n\n    for s in range(args.strong_seeds):\n        m = StrongFusionModel(video_layers=2, video_hidden=192).to(device)\n        _train(m, dl, args.epochs, 6e-4, 1e-3, device, use_amp, f\"strong{s}\", cfg.data.seed + s)\n        i, p = _predict_tta(m, cfg.paths, device, use_amp, normalizer, args.tta)\n        ids = i\n        ensemble_probs = ensemble_probs + p\n        del m; torch.cuda.empty_cache()\n\n    n_models = args.fame_seeds + args.strong_seeds\n    ensemble_probs = ensemble_probs / n_models\n    ensemble_probs = ensemble_probs.copy()\n    ensemble_probs[:, 0] *= np.exp(args.null_bias)\n    ensemble_probs = ensemble_probs / ensemble_probs.sum(1, keepdims=True)\n\n    from pathlib import Path\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, ensemble_probs, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "video_only": "\"\"\"Video-only temporal baseline (the critical diagnostic).\n\nThe hypothesis (from analysis): mean-pooling the 15x768 VideoMAE sequence\ndestroys the temporal information that distinguishes the 19 activities.  This\nmodel keeps the full 15-frame sequence, models it temporally (Transformer),\nand classifies without pooling to a single mean vector.\n\nPer ChatGPT's spec: small (2 layers, 4 heads, hidden 256, ~1M params).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import N_CLASSES\n\n\nclass VideoOnlyTransformer(nn.Module):\n    \"\"\"(B,15,768) -> (B,19). Temporal Transformer, attention pooling.\"\"\"\n\n    def __init__(self, hidden: int = 256, layers: int = 2, heads: int = 4,\n                 n_classes: int = N_CLASSES, dropout: float = 0.2):\n        super().__init__()\n        self.proj = nn.Sequential(\n            nn.LayerNorm(768), nn.Linear(768, hidden), nn.GELU())\n        pos = torch.zeros(1, 15, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.pos = nn.Parameter(pos)\n        self.enc = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(hidden, heads, hidden * 4, dropout,\n                                       \"gelu\", batch_first=True, norm_first=True),\n            layers, enable_nested_tensor=False)\n        # attention pooling\n        self.attn = nn.Sequential(nn.Linear(hidden, 64), nn.Tanh(), nn.Linear(64, 1))\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, n_classes))\n\n    def forward(self, video):\n        \"\"\"video: (B,15,768) -> (B,19).\"\"\"\n        h = self.proj(video) + self.pos\n        h = self.enc(h)                       # (B,15,hidden)\n        scores = self.attn(h).squeeze(-1)\n        w = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * w).sum(dim=1)\n        pooled = torch.cat([attended, h.mean(dim=1)], dim=-1)\n        return self.head(pooled)",
    "run_video": "\"\"\"Train + submit the video-only temporal baseline.\n\nDiagnostic #1: how much signal does the full 15-frame VideoMAE sequence carry?\nIf this scores ~0.75-0.85, video is the key and our 0.657 was a feature bottleneck.\nIf ~0.55-0.65, the winning teams exploit something else.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .config import Config, N_CLASSES\nfrom .data import (\n    build_dataset_windows, subsample_windows, load_train_recording,\n    VIDEO_WINDOW, VIDEO_FRAME_OFFSET,\n)\nfrom .video_only import VideoOnlyTransformer\n\n\nclass VideoWinDataset(Dataset):\n    \"\"\"Preloaded per-window video (vectorized per recording) -> (15,768), target.\"\"\"\n\n    def __init__(self, windows, paths, augment=False, seed=0):\n        self.windows = windows.reset_index(drop=True)\n        self.augment = augment\n        self.rng = np.random.RandomState(seed)\n        vids, ys = [], []\n        for rec in windows[\"rec\"].unique():\n            r = load_train_recording(paths, rec)\n            sub = windows[windows[\"rec\"] == rec]\n            vstarts = sub[\"video_start\"].to_numpy()\n            vidx = vstarts[:, None] + np.arange(VIDEO_WINDOW)[None, :]\n            v = np.asarray(r.video[vidx], dtype=np.float32)\n            vids.append(np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0))\n            ys.append(sub[\"target\"].to_numpy())\n        self.vids = np.concatenate(vids) if vids else np.zeros((0, VIDEO_WINDOW, 768), np.float32)\n        self.ys = np.concatenate(ys) if ys else np.zeros(0, np.int64)\n\n    def __len__(self):\n        return len(self.windows)\n\n    def __getitem__(self, idx):\n        v = self.vids[idx]\n        if self.augment:\n            # mild frame noise + feature masking\n            v = v + self.rng.randn(*v.shape).astype(np.float32) * 0.02\n        return v, int(self.ys[idx])\n\n\ndef _collate(batch):\n    vids = np.stack([b[0] for b in batch])\n    tgt = np.array([b[1] for b in batch])\n    return torch.as_tensor(vids), torch.as_tensor(tgt, dtype=torch.long)\n\n\nclass VideoTestDataset(Dataset):\n    def __init__(self, paths):\n        from .data import build_test_dataset\n        _, video, _, ids, _ = build_test_dataset(paths)\n        self.video = np.nan_to_num(np.asarray(video, dtype=np.float32))\n        self.ids = ids\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, idx):\n        return self.video[idx], self.ids[idx]\n\n\ndef _collate_test(batch):\n    vids = torch.as_tensor(np.stack([b[0] for b in batch]))\n    ids = torch.as_tensor([b[1] for b in batch])\n    return vids, ids\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=300)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--lr\", type=float, default=5e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=3)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.reservoir.max_per_class = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.reservoir.seed)\n    print(f\"windows = {len(windows)}\", flush=True)\n\n    ds = VideoWinDataset(windows, cfg.paths, augment=True, seed=cfg.reservoir.seed)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=0,\n                    collate_fn=_collate)\n    tds = VideoTestDataset(cfg.paths)\n    tdl = DataLoader(tds, batch_size=args.batch_size, shuffle=False, num_workers=0,\n                     collate_fn=_collate_test)\n\n    crit = nn.CrossEntropyLoss(label_smoothing=0.05)\n    acc_probs = None\n    test_ids = None\n    for seed in range(args.seeds):\n        torch.manual_seed(seed)\n        model = VideoOnlyTransformer().to(device)\n        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n        total = args.epochs * len(dl)\n        warm = 1 * len(dl)\n\n        def ll(step):\n            if step < warm:\n                return step / max(1, warm)\n            p = (step - warm) / max(1, total - warm)\n            return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n        model.train()\n        for ep in range(args.epochs):\n            r, nb = 0.0, 0\n            for video, tgt in dl:\n                video = video.to(device)\n                tgt = tgt.to(device)\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    loss = crit(model(video), tgt)\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                scaler.step(opt)\n                scaler.update()\n                sched.step()\n                r += loss.item(); nb += 1\n            print(f\"[video] seed{seed} ep {ep+1}/{args.epochs} loss={r/max(1,nb):.4f}\", flush=True)\n\n        # predict test\n        model.eval()\n        probs = []\n        ids_out = None\n        with torch.inference_mode():\n            for video, ids in tdl:\n                video = video.to(device)\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = model(video)\n                p = torch.softmax(logits.float(), dim=-1).cpu().numpy()\n                probs.append(p)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        p = np.concatenate(probs)\n        o = np.argsort(ids_out)\n        p = p[o]\n        test_ids = np.sort(ids_out)\n        acc_probs = p if acc_probs is None else acc_probs + p\n\n    probs = acc_probs / args.seeds\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(test_ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_video_knn": "\"\"\"Video kNN / retrieval baseline (ChatGPT's #1 bet, tested cheaply).\n\nEven though a video classifier generalizes poorly (0.45), retrieval matches each\ntest window to the most similar TRAINING windows, which might transfer better if\ntest windows visually resemble train windows of the same class.\n\nNo neural training needed - just extract video embeddings (mean of 15 frames,\noptionally + PCA) and find nearest training windows by cosine distance.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config\nfrom .data import (\n    build_dataset_windows, subsample_windows, load_train_recording,\n    build_test_dataset, VIDEO_WINDOW,\n)\n\n\ndef _video_embedding(v):\n    \"\"\"v: (15,768) -> embedding (mean frame, and mean of frame norms).\"\"\"\n    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)\n    mean = v.mean(axis=0)                 # (768,)\n    std = v.std(axis=0)\n    return np.concatenate([mean, std])    # (1536,)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=300)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--k\", type=int, default=10)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--distance\", type=str, default=\"cosine\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.reservoir.seed)\n    print(f\"train windows = {len(windows)}\", flush=True)\n\n    # Extract train video embeddings + labels\n    print(\"extracting train video embeddings...\", flush=True)\n    train_feats, train_y = [], []\n    for rec in windows[\"rec\"].unique():\n        r = load_train_recording(cfg.paths, rec)\n        sub = windows[windows[\"rec\"] == rec]\n        vstarts = sub[\"video_start\"].to_numpy()\n        ys = sub[\"target\"].to_numpy()\n        for v0, y in zip(vstarts, ys):\n            v = np.asarray(r.video[v0:v0 + VIDEO_WINDOW], dtype=np.float32)\n            train_feats.append(_video_embedding(v))\n            train_y.append(int(y))\n    X_tr = np.stack(train_feats)          # (T,1536)\n    y_tr = np.array(train_y)\n    # L2-normalize for cosine\n    X_tr = X_tr / (np.linalg.norm(X_tr, axis=1, keepdims=True) + 1e-9)\n    print(f\"train embeddings: {X_tr.shape}\", flush=True)\n\n    # Test embeddings\n    print(\"extracting test video embeddings...\", flush=True)\n    _, video, _, ids, _ = build_test_dataset(cfg.paths)\n    test_feats = np.stack([_video_embedding(np.asarray(video[i], np.float32)) for i in range(len(ids))])\n    X_te = test_feats / (np.linalg.norm(test_feats, axis=1, keepdims=True) + 1e-9)\n    print(f\"test embeddings: {X_te.shape}\", flush=True)\n\n    # kNN by cosine (dot product of L2-normalized vectors)\n    print(f\"computing {args.k}-NN...\", flush=True)\n    sim = X_te @ X_tr.T                 # (N, T)\n    k = min(args.k, X_tr.shape[0])\n    top_idx = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]\n    # gather labels of top-k, weight by similarity^2\n    n_classes = 19\n    probs = np.zeros((len(X_te), n_classes))\n    for i in range(len(X_te)):\n        idxs = top_idx[i]\n        s = sim[i, idxs]\n        w = np.clip(s, 0, None) ** 2 + 1e-9\n        labels = y_tr[idxs]\n        for j, lab in enumerate(labels):\n            probs[i, lab] += w[j]\n        probs[i] /= probs[i].sum()\n\n    # null bias\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    o = np.argsort(ids)\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids[o], probs[o], cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_sensor_spec": "\"\"\"Priority 1: 4 separate sensor-specialist LightGBMs (Submission A).\n\nEach sensor gets its OWN model using the champion features. At test, a window\nis routed to its sensor's specialist. This tests whether sensor-specific\nmodeling (vs one shared LightGBM + one-hot) improves generalization.\n\nExpected: +0.01 to +0.06 if sensor physics differ meaningfully.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features, build_test_features,\n)\nfrom .features import extract_acc_features_vect, extract_video_features_vect\n\n\ndef _fit_one(X, y, seed, n_est, lr, leaves):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(X, y, sample_weight=sw)\n    return m\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--use-video\", type=int, default=1)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n\n    # X is per-sensor-view flattened: (N*4, 100). Ordering is\n    # (w0s0,w0s1,w0s2,w0s3, w1s0,...). sensor one-hot tells which.\n    N_views = X.shape[0]\n    # split by sensor\n    sens_id = sensor.argmax(axis=1)   # (N*4,)\n    feat_vid = np.concatenate([X, vid, sensor], axis=1) if args.use_video else X\n    print(f\"total views={N_views}\", flush=True)\n\n    models = {}\n    for s in range(N_SENSORS):\n        mask = sens_id == s\n        Xs = feat_vid[mask]\n        ys = y[mask]\n        print(f\"sensor {s}: {Xs.shape[0]} views\", flush=True)\n        models[s] = _fit_one(Xs, ys, cfg.reservoir.seed + s,\n                             args.n_estimators, args.lr, args.num_leaves)\n        print(f\"sensor {s} model trained\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    if args.use_video:\n        Ft = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    else:\n        Ft = Xt\n    N = len(ids)\n    test_sens = sensor_t.argmax(axis=1)   # (N,)\n\n    probs = np.zeros((N, N_CLASSES))\n    for s in range(N_SENSORS):\n        mask = test_sens == s\n        if not mask.any():\n            continue\n        p = models[s].predict_proba(Ft[mask])\n        if p.shape[1] < N_CLASSES:\n            pad = np.zeros((p.shape[0], N_CLASSES - p.shape[1]))\n            p = np.concatenate([p, pad], axis=1)\n        probs[mask] = p\n\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_domain_weight": "\"\"\"Priority 3: domain-density importance weighting (Submission 3).\n\nMotivation: \"more training data hurts\" (0.657 -> 0.586). Some train windows are\nharmful for the test domain. We train a domain classifier (train=0, test=1) on\nchampion features, then weight each train window by P(test|x)/P(train|x) so\ntest-like examples get higher weight, and train the champion LightGBM with\nthose sample weights (per-sensor domain model, but SHARED LightGBM).\n\nDe-risked: ChatGPT notes to check whether the method \"consistently survives\ndomain changes\" on synthetic train-splits, but the champion pipeline + weights\nis the executable version.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import build_train_features, build_test_features\n\n\ndef _domain_weight(X_src, X_tgt, seed):\n    \"\"\"Return importance weight per source sample = P(tgt|x)/P(src|x) via a\n    train/test binary classifier's odds.\"\"\"\n    from sklearn.linear_model import LogisticRegression\n    n_src = len(X_src)\n    X = np.concatenate([X_src, X_tgt], axis=0)\n    y = np.concatenate([np.zeros(n_src), np.ones(len(X_tgt))])\n    clf = LogisticRegression(max_iter=1000, C=1.0)\n    clf.fit(X, y)\n    p = clf.predict_proba(X_src)[:, 1]          # P(test|x)\n    p = np.clip(p, 1e-4, 1 - 1e-4)\n    w = p / (1 - p)                              # odds = P(test)/P(train)\n    # normalize to mean 1\n    w = w / w.mean()\n    return w\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    print(f\"test views={Fte.shape}\", flush=True)\n\n    # Domain weights per sensor (shared model, per-sensor domain fit)\n    weights = np.ones(len(Ftr))\n    sens_id = sensor.argmax(axis=1)\n    test_sens = sensor_t.argmax(axis=1)\n    for s in range(N_SENSORS):\n        src = Ftr[sens_id == s]\n        tgt = Fte[test_sens == s]\n        if len(tgt) < 50:\n            continue\n        w = _domain_weight(src, tgt, cfg.reservoir.seed + s)\n        weights[sens_id == s] = w\n        print(f\"sensor {s}: src={len(src)} tgt={len(tgt)} weight mean={w.mean():.3f} \"\n              f\"std={w.std():.3f}\", flush=True)\n    # clip extreme weights\n    lo, hi = np.percentile(weights, [1, 99])\n    weights = np.clip(weights, lo, hi)\n    weights = weights / weights.mean()\n\n    # Champion LightGBM with domain weights\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    cls_w = 1.0 / counts[y]\n    cls_w = cls_w / cls_w.mean()\n    sw = weights * cls_w\n    sw = sw / sw.mean()\n\n    model = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=args.n_estimators, learning_rate=args.lr,\n        num_leaves=args.num_leaves, subsample=0.8, colsample_bytree=0.8,\n        reg_lambda=1.0, min_child_samples=30, n_jobs=8,\n        random_state=cfg.reservoir.seed, verbose=-1)\n    model.fit(Ftr, y, sample_weight=sw)\n    print(\"domain-weighted LightGBM trained\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose_duplicates": "\"\"\"Diagnostic #2: detect cross-sensor duplicate/temporal-group windows in TEST.\n\nThe test has ~3050 windows per sensor (4 sensors -> ~12,200 ~= 12,234). If the\ntest comes from ~3050 distinct temporal moments, each captured by multiple\nsensors, then the VIDEO at those moments is near-identical (same camera). This\ndiagnostic finds near-duplicate video windows and reports group structure.\n\nNo training, no submission. Cheap read-only analysis.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport pandas as pd\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--sample\", type=int, default=20000,\n                    help=\"max windows to embed for duplicate detection\")\n    ap.add_argument(\"--threshold\", type=float, default=0.02,\n                    help=\"L2 distance threshold on normalized embedding for 'duplicate'\")\n    args = ap.parse_args()\n\n    from .config import Config\n    from .data import build_test_dataset\n    cfg = Config()\n\n    print(\"loading test data...\", flush=True)\n    inertial, video, sensor_ids, ids, sbj = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n    print(f\"test: {len(ids)} windows, subjects={np.unique(sbj).tolist()}\", flush=True)\n\n    # Embed each window's video as mean frame + std (compressed).\n    v = np.asarray(video, dtype=np.float64)\n    n = min(len(v), args.sample)\n    vid = v[:n]\n    mean = vid.mean(axis=1)          # (n,768)\n    std = vid.std(axis=1)\n    emb = np.concatenate([mean, std], axis=1)   # (n,1536)\n    # L2 normalize\n    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    print(f\"embedded {n} video windows\", flush=True)\n\n    # Fast near-duplicate detection via sorted L2 (brute force on sample is fine for n~12k)\n    # Reduce memory: only compute for a subsample if needed.\n    n2 = len(emb)\n    print(\"computing pairwise distances (subsample)...\", flush=True)\n    sub = np.random.RandomState(0).choice(n2, min(3000, n2), replace=False)\n    D = emb[sub] @ emb.T            # cosine similarity (n_sub, n2)\n    sim = D                          # higher = more similar\n    # For each sampled window, count how many OTHERS are above threshold similarity\n    near_dup_counts = (sim > (1 - args.threshold)).sum(axis=1) - 1  # exclude self\n    frac_has_dup = (near_dup_counts > 0).mean()\n    print(f\"sampled {len(sub)} windows; frac with >=1 near-duplicate video: {frac_has_dup:.3f}\")\n    print(f\"max near-duplicates for one window: {near_dup_counts.max()}\")\n\n    # Are near-duplicates from DIFFERENT sensors (cross-sensor) or same?\n    cross = 0\n    same_sensor = 0\n    for i, si in enumerate(sub):\n        sims = sim[i]\n        near = np.where(sims > (1 - args.threshold))[0]\n        near = near[near != si]\n        if len(near) == 0:\n            continue\n        for j in near:\n            if sensor_names[j] != sensor_names[si]:\n                cross += 1\n            else:\n                same_sensor += 1\n    print(f\"near-duplicate pairs: cross-sensor={cross} same-sensor={same_sensor}\")\n    if cross > same_sensor:\n        print(\"STRONG SIGNAL: cross-sensor duplicate windows exist -> shared temporal moments!\")\n    else:\n        print(\"Mostly same-sensor or no strong cross-sensor duplicate structure.\")\n\n    # Save a readable report (this notebook is a DIAGNOSTIC - do NOT submit it).\n    report = {\n        \"n_test\": len(ids),\n        \"subjects\": np.unique(sbj).tolist(),\n        \"sensor_counts\": pd.Series(sensor_names).value_counts().to_dict(),\n        \"frac_with_dup\": float(frac_has_dup),\n        \"max_dups\": int(near_dup_counts.max()),\n        \"cross_sensor_pairs\": int(cross),\n        \"same_sensor_pairs\": int(same_sensor),\n    }\n    import json\n    from pathlib import Path\n    rep = Path(\"/kaggle/working/dup_report.json\")\n    rep.write_text(json.dumps(report, indent=2))\n    print(\"report -> /kaggle/working/dup_report.json\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_propagate": "\"\"\"Cross-sensor prediction propagation (multi-threshold).\n\nThe test contains ~243 cross-sensor duplicate moments (shared temporal moments\nacross sensors). We train the champion LightGBM, predict all test windows, find\ncross-sensor near-duplicate groups via video similarity, and propagate the\nbest prediction within each group. Supports evaluating MULTIPLE thresholds in\none run (writes each as a named candidate + primary to submission.csv).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES\nfrom .feature_data import build_train_features, build_test_features\n\n\ndef _train_champion(Ftr, y, cfg, n_est, lr, leaves, seed):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(Ftr, y, sample_weight=sw)\n    return m\n\n\ndef _propagate(probs, sim, sensor_names, threshold, agg):\n    \"\"\"Apply propagation for one threshold. Returns propagated probs copy.\"\"\"\n    n = sim.shape[0]\n    adj = sim > (1 - threshold)\n    np.fill_diagonal(adj, False)\n    visited = np.zeros(n, dtype=bool)\n    p_prop = probs.copy()\n    propagated = 0\n    for i in range(n):\n        if visited[i]:\n            continue\n        comp = [i]\n        visited[i] = True\n        stack = [i]\n        while stack:\n            u = stack.pop()\n            nb = np.where(adj[u])[0]\n            for vv in nb:\n                if not visited[vv]:\n                    visited[vv] = True\n                    comp.append(vv)\n                    stack.append(vv)\n        if len(comp) > 1 and len(set(sensor_names[comp])) >= 2:\n            p = p_prop[comp]                       # (g,19)\n            if agg == \"softvote\":\n                best = int(p.sum(axis=0).argmax())\n            else:  # maxconf\n                preds = p.argmax(axis=1)\n                conf = p.max(axis=1)\n                best = preds[int(conf.argmax())]\n            p_prop[comp] = 0.0\n            p_prop[comp, best] = 1.0\n            propagated += len(comp)\n    return p_prop, propagated\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--dup-threshold\", type=float, default=0.02)\n    ap.add_argument(\"--agg\", type=str, default=\"maxconf\",\n                    choices=[\"maxconf\", \"softvote\"])\n    ap.add_argument(\"--thresholds\", type=str, default=None,\n                    help=\"comma-separated thresholds to evaluate in one run\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n\n    model = _train_champion(Ftr, y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, cfg.reservoir.seed)\n    print(\"champion trained\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = np.clip(probs, 1e-9, None)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    # Align probs to id order (build_test_features returns meta order == id order).\n    order = np.argsort(ids)\n    probs = probs[order]\n    sorted_ids = np.sort(ids)\n\n    from .data import build_test_dataset\n    _, video, _, _, _ = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n\n    # Precompute video cosine similarity matrix once (shared across thresholds).\n    v = np.asarray(video, dtype=np.float64)\n    mean = v.mean(axis=1)\n    std = v.std(axis=1)\n    emb = np.concatenate([mean, std], axis=1)\n    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    sim = emb @ emb.T\n    print(f\"similarity matrix: {sim.shape}\", flush=True)\n\n    thresh_list = [float(t) for t in args.thresholds.split(\",\")] if args.thresholds else [args.dup_threshold]\n    from .inference import write_submission\n    out_dir = Path(\"/kaggle/working\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    for thr in thresh_list:\n        p_prop, propagated = _propagate(probs, sim, sensor_names, thr, args.agg)\n        print(f\"thr={thr}: propagated {propagated} windows\", flush=True)\n        final = p_prop.copy()\n        final[:, 0] *= np.exp(args.null_bias)\n        final = final / final.sum(1, keepdims=True)\n        fname = \"submission.csv\" if abs(thr - args.dup_threshold) < 1e-9 else f\"submission_prop_{thr:.3f}.csv\"\n        sub = out_dir / fname\n        write_submission(sorted_ids, final, cfg.paths.sample_submission, sub)\n        print(f\"wrote -> {sub}\", flush=True)\n\n    print(\"done\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose_groups": "\"\"\"Group-coverage diagnostic: can we recover the ~3,050 latent 4-sensor moments?\n\nUses the FULL 15x768 video as a join key (ChatGPT's #1 priority). Reports the\n2/3/4-sensor group size distribution and coverage. NO submission - read-only.\nThis decides whether building full group-fusion is worth it.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _video_fingerprint(v, pca=None):\n    \"\"\"Full-sequence fingerprint of a (15,768) window -> compact vector.\n\n    mean, std, first, last frame + frame-difference mean + optional PCA on\n    the flattened 15*768 (compressed by a random projection).\n    \"\"\"\n    v = np.asarray(v, dtype=np.float64)\n    mean = v.mean(axis=0)                  # (768,)\n    std = v.std(axis=0)\n    first = v[0]\n    last = v[-1]\n    d = np.diff(v, axis=0).mean(axis=0)    # (768,)\n    feats = np.concatenate([mean, std, first, last, d])  # (3840,)\n    if pca is not None:\n        feats = (feats - pca[0]) @ pca[1]\n    return feats\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--threshold\", type=float, default=0.03)\n    ap.add_argument(\"--subsample\", type=int, default=12234)\n    ap.add_argument(\"--pca-components\", type=int, default=128)\n    args = ap.parse_args()\n\n    from .config import Config\n    from .data import build_test_dataset\n    cfg = Config()\n\n    print(\"loading test data...\", flush=True)\n    _, video, sensor_ids, ids, sbj = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n    subj = meta[\"sbj_id\"].to_numpy()\n    N = len(ids)\n    print(f\"test: {N} windows, subjects={np.unique(subj).tolist()}\", flush=True)\n\n    v = np.asarray(video, dtype=np.float64)[:args.subsample]\n\n    # Build fingerprints (raw 3840-dim) then fit a random projection for compactness.\n    print(\"computing video fingerprints...\", flush=True)\n    feats_raw = np.stack([_video_fingerprint(v[i]) for i in range(len(v))])\n    # center + random projection (deterministic)\n    center = feats_raw.mean(axis=0)\n    rng = np.random.RandomState(0)\n    proj = rng.randn(feats_raw.shape[1], args.pca_components) / np.sqrt(feats_raw.shape[1])\n    feats = (feats_raw - center) @ proj\n    feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-9)\n    print(f\"fingerprints: {feats.shape}\", flush=True)\n\n    # Nearest-neighbor cross-sensor matching within same subject.\n    # For each window, find windows of a DIFFERENT sensor, same subject, that are\n    # near-identical (candidate same-moment views).\n    print(\"matching cross-sensor windows (same subject)...\", flush=True)\n    sim = feats @ feats.T          # (N,N)\n    n = len(feats)\n    groups = []\n    assigned = np.zeros(n, dtype=bool)\n    for i in range(n):\n        if assigned[i]:\n            continue\n        # candidate same-moment: same subject, diff sensor, above threshold\n        cand = np.where(\n            (subj == subj[i]) & (sensor_names != sensor_names[i]) & (sim[i] > (1 - args.threshold))\n        )[0]\n        cand = cand[cand != i]\n        if len(cand) == 0:\n            continue\n        # build group: i + all candidates (union, then dedupe by scanning)\n        member = {int(i)}\n        for c in cand:\n            member.add(int(c))\n        # expand: any member's own matches add to group\n        changed = True\n        while changed:\n            changed = False\n            for m in list(member):\n                mcand = np.where(\n                    (subj == subj[m]) & (sensor_names != sensor_names[m]) & (sim[m] > (1 - args.threshold))\n                )[0]\n                for c in mcand:\n                    if c not in member:\n                        member.add(int(c)); changed = True\n        member = sorted(member)\n        if len(member) >= 2:\n            for m in member:\n                assigned[m] = True\n            groups.append(member)\n\n    # Stats\n    sizes = [len(g) for g in groups]\n    sensors_per_group = [len(set(sensor_names[g])) for g in groups]\n    print(f\"\\n=== GROUP RECOVERY (threshold={args.threshold}) ===\")\n    print(f\"total groups: {len(groups)}\")\n    print(f\"windows in groups: {sum(sizes)} / {n}\")\n    print(f\"coverage: {sum(sizes)/n:.3f}\")\n    print(f\"group sizes: 2-sensor={sum(1 for s in sizes if s==2)}, \"\n          f\"3-sensor={sum(1 for s in sizes if s==3)}, \"\n          f\"4-sensor={sum(1 for s in sizes if s>=4)}\")\n    # distinct sensors per group\n    print(f\"groups with >=2 distinct sensors: {sum(1 for sp in sensors_per_group if sp>=2)}\")\n    print(f\"groups with 3 distinct sensors: {sum(1 for sp in sensors_per_group if sp==3)}\")\n    print(f\"groups with 4 distinct sensors: {sum(1 for sp in sensors_per_group if sp>=4)}\")\n\n    import json\n    from pathlib import Path\n    rep = {\n        \"threshold\": args.threshold,\n        \"total_groups\": len(groups),\n        \"windows_in_groups\": int(sum(sizes)),\n        \"coverage\": float(sum(sizes)/n),\n        \"size2\": int(sum(1 for s in sizes if s==2)),\n        \"size3\": int(sum(1 for s in sizes if s==3)),\n        \"size4plus\": int(sum(1 for s in sizes if s>=4)),\n        \"sens3\": int(sum(1 for sp in sensors_per_group if sp==3)),\n        \"sens4\": int(sum(1 for sp in sensors_per_group if sp>=4)),\n    }\n    Path(\"/kaggle/working/group_report.json\").write_text(json.dumps(rep, indent=2))\n    print(\"report -> /kaggle/working/group_report.json\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_recording_select": "\"\"\"Recording-level domain selection (explains 'more data hurts' + CV-uselessness).\n\nHypothesis: the test domain corresponds to a specific subset of training\nRECORDINGS (session/location conditions). Adding more training recordings that\nare OFF-domain hurts (0.657 -> 0.586). We select the train recordings whose\nfeature distribution best matches each test subject, then train the champion on\nonly those recordings.\n\nSteps:\n1. Build per-window features, grouped by recording.\n2. Build test features, grouped by subject.\n3. For each test subject, find the K closest train recordings (mean feature dist).\n4. Train champion on the union of selected recordings.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features, build_test_features,\n)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--k-recordings\", type=int, default=14,\n                    help=\"number of closest train recordings to keep per test subject\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features (per-sensor views)...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=None,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    n_view = Ftr.shape[0]\n    print(f\"all train views={n_view}\", flush=True)\n\n    # Build per-recording feature means. We need recording labels per view.\n    from .data import build_dataset_windows, load_train_recording, InertialNormalizer\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    grp, rec_names = _recording_groups(windows, cfg.paths, normalizer)\n    rec_id = {r: i for i, r in enumerate(rec_names)}\n\n    # Per-recording mean of full feature vector (per sensor view).\n    rec_means = {}\n    for r in rec_names:\n        idx = np.where(grp == rec_id[r])[0]\n        rec_means[r] = Ftr[idx].mean(axis=0)\n    rec_list = list(rec_names)\n\n    # Test features grouped by subject.\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    test_subj = meta[\"sbj_id\"].to_numpy()\n\n    # For each test subject, find K closest train recordings (mean feature distance).\n    selected_recs = set()\n    for subj in np.unique(test_subj):\n        subj_idx = np.where(test_subj == subj)[0]\n        subj_mean = Fte[subj_idx].mean(axis=0)\n        # distance to each recording mean\n        dists = []\n        for r in rec_list:\n            d = np.linalg.norm(rec_means[r] - subj_mean)\n            dists.append((d, r))\n        dists.sort()\n        for _, r in dists[:args.k_recordings]:\n            selected_recs.add(r)\n    print(f\"selected {len(selected_recs)}/{len(rec_list)} recordings for training\", flush=True)\n\n    # Train on selected recordings only.\n    sel_idx = np.where(np.isin(grp, [rec_id[r] for r in selected_recs]))[0]\n    Fs = Ftr[sel_idx]\n    ys = y[sel_idx]\n    print(f\"training views after selection: {Fs.shape}\", flush=True)\n\n    import lightgbm as lgb\n    counts = np.bincount(ys, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[ys]\n    sw = sw / sw.mean()\n    model = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=args.n_estimators, learning_rate=args.lr,\n        num_leaves=args.num_leaves, subsample=0.8, colsample_bytree=0.8,\n        reg_lambda=1.0, min_child_samples=30, n_jobs=8,\n        random_state=cfg.reservoir.seed, verbose=-1)\n    model.fit(Fs, ys, sample_weight=sw)\n    print(\"champion trained on selected recordings\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\ndef _recording_groups(windows, paths, normalizer):\n    \"\"\"Recording-index per per-sensor view, matching build_train_features order.\n\n    Order: for each recording (in unique order), for each window, for each valid\n    sensor s in 0..3 -> append recording index. Returns (grp (M,), rec_names).\n    \"\"\"\n    from .config import N_SENSORS\n    from .data import load_train_recording, INERTIAL_WINDOW\n    rec_names = list(windows[\"rec\"].unique())\n    rec_id = {r: i for i, r in enumerate(rec_names)}\n    grp = []\n    for rec in rec_names:\n        r = load_train_recording(paths, rec)\n        sub = windows[windows[\"rec\"] == rec]\n        starts = sub[\"inertial_start\"].to_numpy()\n        idx = starts[:, None] + np.arange(INERTIAL_WINDOW)[None, :]\n        w = np.asarray(r.inertial[idx])\n        valid = np.isfinite(w).all(axis=(1, 3))   # (M,4)\n        gid = rec_id[rec]\n        for s in range(N_SENSORS):\n            keep = valid[:, s]\n            grp.extend([gid] * int(keep.sum()))\n    return np.array(grp, dtype=np.int64), rec_names\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_hierarchical": "\"\"\"Hierarchical classification (family -> variant) targeting macro-F1.\n\nMacro-F1 loses points distinguishing similar VARIANTS (jogging vs jogging-arm-\nrotation, push-ups vs complex, sit-ups vs complex, lunges vs complex). We split\nthe 19 classes into 8 families and train:\n  * Family model (8-way) on champion features + video.\n  * Per-family variant models (on features + video) to distinguish within family.\nAt test, predict family, then variant within that family.\n\nThis forces the model to spend capacity on the hard within-family distinctions\nthat drive macro-F1 down.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import build_train_features, build_test_features\n\n# Family -> list of class ids (0-18)\nFAMILIES = [\n    [0],                 # null\n    [1, 2, 3, 4, 5],     # jogging + variants\n    [6, 7, 8, 9, 10],    # stretching + variants\n    [11, 12],            # push-ups + complex\n    [13, 14],            # sit-ups + complex\n    [16, 17],            # lunges + complex\n    [15],                # burpees\n    [18],                # bench-dips\n]\n# class -> family index\nCLASS_TO_FAM = np.zeros(N_CLASSES, dtype=np.int64)\nfor fi, classes in enumerate(FAMILIES):\n    for c in classes:\n        CLASS_TO_FAM[c] = fi\nN_FAM = len(FAMILIES)\n\n\ndef _train_lgbm(X, y, cfg, n_est, lr, leaves, n_class, seed):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=n_class).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=n_class,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(X, y, sample_weight=sw)\n    return m\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-weight\", type=float, default=0.3,\n                    help=\"weight on video branch probabilities in final blend\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n\n    # --- Family model ---\n    fam_y = CLASS_TO_FAM[y]\n    print(\"training family model...\", flush=True)\n    fam_model = _train_lgbm(Ftr, fam_y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, N_FAM, cfg.reservoir.seed)\n    fam_probs = fam_model.predict_proba(Fte)          # (N, N_FAM)\n    if fam_probs.shape[1] < N_FAM:\n        fam_probs = np.concatenate([fam_probs, np.zeros((len(fam_probs), N_FAM - fam_probs.shape[1]))], axis=1)\n\n    # --- Per-family variant models ---\n    # Base class probs: start with family prob spread across family members.\n    N = len(Fte)\n    probs = np.zeros((N, N_CLASSES))\n    for fi, classes in enumerate(FAMILIES):\n        mask = fam_y == fi\n        # only classes actually present in this family's training set\n        present = sorted(set(y[mask].tolist()) & set(classes))\n        if mask.sum() < 100 or len(present) < 2:\n            # fall back to uniform within the family\n            for c in present:\n                probs[:, c] += fam_probs[:, fi] / len(present)\n            continue\n        Xf = Ftr[mask]\n        yf = y[mask]\n        n_var = len(present)\n        local_map = {c: j for j, c in enumerate(present)}\n        yf_local = np.array([local_map[c] for c in yf])\n        vm = _train_lgbm(Xf, yf_local, cfg, args.n_estimators, args.lr,\n                         args.num_leaves, n_var, cfg.reservoir.seed + fi + 1)\n        vp = vm.predict_proba(Fte)\n        if vp.shape[1] < n_var:\n            vp = np.concatenate([vp, np.zeros((N, n_var - vp.shape[1]))], axis=1)\n        # combine: family prob * variant prob within family\n        for j, c in enumerate(present):\n            probs[:, c] += fam_probs[:, fi] * vp[:, j]\n\n    # Ensure rows sum to 1\n    probs = probs / probs.sum(1, keepdims=True)\n\n    # Optional: blend with a plain 19-class champion (video-weighted).\n    if args.video_weight > 0:\n        print(\"training 19-class champion for blend...\", flush=True)\n        champ = _train_lgbm(Ftr, y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, N_CLASSES, cfg.reservoir.seed + 99)\n        champ_probs = champ.predict_proba(Fte)\n        if champ_probs.shape[1] < N_CLASSES:\n            champ_probs = np.concatenate([champ_probs, np.zeros((N, N_CLASSES - champ_probs.shape[1]))], axis=1)\n        probs = probs * (1 - args.video_weight) + champ_probs * args.video_weight\n        probs = probs / probs.sum(1, keepdims=True)\n\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
}
for _m, _c in SRC.items():
    (WROOT / f'{_m}.py').write_text(_c, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
for _m in list(sys.modules):
    if _m == 'wearfusion' or _m.startswith('wearfusion.'):
        del sys.modules[_m]
from wearfusion import *  # noqa
import wearfusion


In [ ]:
import os, pathlib, shutil, sys
WROOT = pathlib.Path('/kaggle/working/wearfusion')
if WROOT.exists():
    shutil.rmtree(WROOT)
WROOT.mkdir(parents=True, exist_ok=True)
(WROOT / '__init__.py').write_text('')
SRC = {
    "config": "\"\"\"Central configuration for the WEAR fusion pipeline.\n\nSupports running either inside a Kaggle notebook (data mounted under\n``/kaggle/input/...``) or locally with a copy of the data directory.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n# ---------------------------------------------------------------------------\n# Constants / data schema\n# ---------------------------------------------------------------------------\nCOMPETITION_NAME = \"3rd-wear-dataset-challenge-hasca-2026\"\nCOMPETITION_DIR = Path(\"/kaggle/input/competitions\") / COMPETITION_NAME\n\nSENSOR_LOCATIONS = [\"right_arm\", \"right_leg\", \"left_leg\", \"left_arm\"]\nSENSOR_TO_ID = {s: i for i, s in enumerate(SENSOR_LOCATIONS)}\nN_SENSORS = len(SENSOR_LOCATIONS)\nN_AXES = 3\n\n# Test metadata uses these location strings.\nTEST_SENSOR_LOCATIONS = SENSOR_LOCATIONS\n\nN_CLASSES = 19\nCLASS_NAMES = [\n    \"null\",\n    \"jogging\",\n    \"jogging (rotating arms)\",\n    \"jogging (skipping)\",\n    \"jogging (sidesteps)\",\n    \"jogging (butt-kicks)\",\n    \"stretching (triceps)\",\n    \"stretching (lunging)\",\n    \"stretching (shoulders)\",\n    \"stretching (hamstrings)\",\n    \"stretching (lumbar rotation)\",\n    \"push-ups\",\n    \"push-ups (complex)\",\n    \"sit-ups\",\n    \"sit-ups (complex)\",\n    \"burpees\",\n    \"lunges\",\n    \"lunges (complex)\",\n    \"bench-dips\",\n]\nLABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}\n\n# Modality shapes (per 1-second window).\nINERTIAL_HZ = 50\nVIDEO_FPS = 30\nINERTIAL_WINDOW = 50          # 50 samples @ 50 Hz\nVIDEO_WINDOW = 15             # frames used in test features\nVIDEO_DIM = 768               # VideoMAEv2-Base feature dim\n\n# For training, video features are extracted at 30 fps while inertial at 50 Hz.\n# The 15 usable video frames map to inertial time as (see baseline): the npy\n# row index `i` corresponds to clip centred on frame `i`; we use rows\n# video_start+8 .. video_start+22 (15 rows) to avoid edge leakage.\nVIDEO_FRAME_OFFSET = 8\n\n\n# ---------------------------------------------------------------------------\n# Runtime environment helpers\n# ---------------------------------------------------------------------------\ndef is_kaggle() -> bool:\n    return os.path.exists(\"/kaggle/input\")\n\n\n@dataclass\nclass Paths:\n    \"\"\"Resolve data/output paths for the current runtime.\"\"\"\n\n    competition_dir: Path = field(\n        default_factory=lambda: COMPETITION_DIR if is_kaggle() else Path(\"data\")\n    )\n    output_dir: Path = field(\n        default_factory=lambda: Path(\"/kaggle/working\" if is_kaggle() else \"runs\")\n    )\n\n    @property\n    def train_inertial_dir(self) -> Path:\n        return self.competition_dir / \"train\" / \"inertial_feat\"\n\n    @property\n    def train_video_dir(self) -> Path:\n        return self.competition_dir / \"train\" / \"videomae_feat\"\n\n    @property\n    def test_dir(self) -> Path:\n        return self.competition_dir / \"test\"\n\n    @property\n    def test_inertial(self) -> Path:\n        return self.test_dir / \"test_inertial_data.npy\"\n\n    @property\n    def test_video(self) -> Path:\n        return self.test_dir / \"test_videomae_data.npy\"\n\n    @property\n    def test_meta(self) -> Path:\n        return self.test_dir / \"test_meta_data.csv\"\n\n    @property\n    def sample_submission(self) -> Path:\n        return self.competition_dir / \"sample_submission.csv\"\n\n\n# ---------------------------------------------------------------------------\n# Training configuration\n# ---------------------------------------------------------------------------\n@dataclass\nclass DataConfig:\n    \"\"\"Windowing + sampling configuration.\"\"\"\n\n    window_stride: int = 25          # inertial-sample stride between windows\n    max_windows_per_class_per_subject: int = 600  # cap for compute (None = all)\n    seed: int = 42\n    # Training-time augmentation knobs.\n    augment_scale: float = 0.15      # uniform(1-s, 1+s) amplitude scaling\n    augment_noise: float = 0.02      # gaussian noise std (relative)\n    augment_shift: int = 2           # max random time shift (samples)\n    single_sensor_prob: float = 0.6  # prob of presenting a single random sensor\n                                     # during training (mirrors the 1-sensor test)\n\n\n@dataclass\nclass ModelConfig:\n    \"\"\"Architecture knobs for the fusion model.\"\"\"\n\n    name: str = \"wear_fusion\"\n    inertial_hidden: int = 192\n    inertial_blocks: int = 3            # Inception blocks in the inertial encoder\n    inertial_channels: int = 128\n    video_hidden: int = 192\n    video_transformer_layers: int = 2\n    video_heads: int = 4\n    sensor_embed_dim: int = 16\n    classifier_hidden: int = 256\n    dropout: float = 0.2\n    modality_dropout: float = 0.1\n    scale: float = 1.0                 # global width multiplier (applied in model)\n\n\n@dataclass\nclass TrainConfig:\n    \"\"\"Training loop hyper-parameters.\"\"\"\n\n    epochs: int = 6\n    batch_size: int = 256\n    lr: float = 8e-4\n    weight_decay: float = 1e-3\n    label_smoothing: float = 0.05\n    warmup_epochs: int = 1\n    num_workers: int = 4\n    seed: int = 42\n    amp: bool = True\n    accumulate_grad_steps: int = 1\n    class_weighting: str = \"none\"   # \"none\" | \"inverse\" | \"sqrt_inverse\"\n    max_grad_norm: float = 5.0\n\n\n@dataclass\nclass EvalConfig:\n    \"\"\"Ensemble / threshold tuning knobs.\"\"\"\n\n    null_bias: float = 0.75            # multiplicative boost on null logit at inference\n    n_val_subjects: int = 4            # subjects held out per grouped CV fold\n    blend_weight: float = 1.0          # reserved\n\n\n@dataclass\nclass ReservoirSpec:\n    \"\"\"One echo-state reservoir configuration.\"\"\"\n\n    state_size: int = 384\n    spectral_radius: float = 0.85\n    leak: float = 0.35\n    sparsity: float = 0.03\n    washout: int = 5\n\n\n@dataclass\nclass ReservoirConfig:\n    \"\"\"Reservoir / CNN / readout / video / fusion / TTA configuration.\"\"\"\n\n    # CNN front-end\n    cnn_channels: int = 128\n    cnn_blocks: int = 3\n    cnn_proj: int = 64\n    cnn_epochs: int = 5\n    cnn_seeds: int = 2\n    # Reservoirs\n    reservoirs: tuple = (\n        (\"r1\", 384, 0.70, 0.20, 0.03),\n        (\"r2\", 384, 0.85, 0.35, 0.03),\n        (\"r3\", 512, 0.95, 0.50, 0.02),\n        (\"r4\", 512, 1.05, 0.70, 0.02),\n    )\n    washout: int = 5\n    # Readouts\n    ridge_alphas: tuple = (0.01, 0.1, 1.0, 10.0)\n    readout_boots: int = 3\n    readout_fit_limit: int = 40000   # max sensor-views used to fit ridge readouts\n    reservoir_batch: int = 2048      # batch size for reservoir feature extraction\n    # Video branch\n    video_hidden: int = 256\n    video_layers: int = 4\n    video_heads: int = 8\n    video_epochs: int = 5\n    video_seeds: int = 2\n    # TTA\n    tta_aug: tuple = (\"none\", \"x\", \"y\", \"z\", \"all\")\n    # Late fusion\n    fusion_alpha: float = 0.7          # weight on inertial branch\n    sensor_fusion: bool = False        # per-sensor alpha (only if OOF stable)\n    max_per_class: int = 300\n    stride: int = 25\n    seed: int = 42\n\n\n@dataclass\nclass Config:\n    data: DataConfig = field(default_factory=DataConfig)\n    model: ModelConfig = field(default_factory=ModelConfig)\n    train: TrainConfig = field(default_factory=TrainConfig)\n    eval: EvalConfig = field(default_factory=EvalConfig)\n    reservoir: ReservoirConfig = field(default_factory=ReservoirConfig)\n    paths: Paths = field(default_factory=Paths)\n\n    def save(self, path: Path) -> None:\n        import json\n\n        path.write_text(\n            json.dumps(\n                {\n                    \"data\": _asdict(self.data),\n                    \"model\": _asdict(self.model),\n                    \"train\": _asdict(self.train),\n                    \"eval\": _asdict(self.eval),\n                    \"reservoir\": _asdict(self.reservoir),\n                },\n                indent=2,\n            )\n        )\n\n\ndef _asdict(obj):\n    from dataclasses import asdict\n\n    return asdict(obj)",
    "data": "\"\"\"Data loading, windowing and LOSO-style grouped splitting for the WEAR data.\n\nModality alignment\n------------------\n* Inertial is recorded at 50 Hz -> a 1-second window is 50 samples.\n* Video (VideoMAE) features are 30 fps -> 15 usable frames per second.\n* We map an inertial window start ``i`` to the 15 video rows\n  ``j = i * 3 // 5`` (rows ``j .. j+15`` offset by ``VIDEO_FRAME_OFFSET``),\n  matching the reference \"TS/Emb\" pipeline so frames near window edges don't\n  leak context.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import (\n    CLASS_NAMES,\n    INERTIAL_WINDOW,\n    LABEL_TO_ID,\n    N_SENSORS,\n    SENSOR_LOCATIONS,\n    SENSOR_TO_ID,\n    VIDEO_DIM,\n    VIDEO_FRAME_OFFSET,\n    VIDEO_WINDOW,\n    Paths,\n)\n\n# Inertial CSV columns.\nINERTIAL_COLS = [f\"{loc}_acc_{ax}\" for loc in SENSOR_LOCATIONS for ax in \"xyz\"]\n# Reshape order: each sensor -> 3 axes (xyz).\nSENSOR_AXES = np.array(\n    [[f\"{loc}_acc_{ax}\" for ax in \"xyz\"] for loc in SENSOR_LOCATIONS]\n)\n\n\ndef _read_label_ids(labels: pd.Series) -> np.ndarray:\n    \"\"\"Map raw label strings/NaN to class ids (NaN -> 0 / 'null').\"\"\"\n    out = np.zeros(len(labels), dtype=np.int64)\n    seen = labels.to_numpy()\n    for i, name in enumerate(seen):\n        if isinstance(name, str) and name in LABEL_TO_ID:\n            out[i] = LABEL_TO_ID[name]\n        else:\n            out[i] = 0  # NaN / unknown -> null\n    return out\n\n\nclass Recording:\n    \"\"\"Prepared single participant recording (inertial + video + labels).\"\"\"\n\n    def __init__(self, sbj_id: int, inertial: np.ndarray, video: np.ndarray,\n                 labels: np.ndarray):\n        self.sbj_id = sbj_id\n        # inertial: (T, N_SENSORS, 3)\n        self.inertial = inertial\n        # video: (T_v, VIDEO_DIM) where T_v = T * 30 // 50 (approx)\n        self.video = video\n        # labels: (T,)\n        self.labels = labels\n\n    @property\n    def n_samples(self) -> int:\n        return self.inertial.shape[0]\n\n\ndef _load_inertial_csv(path: Path) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Load a train inertial CSV -> (inertial (T,4,3), labels (T,)).\"\"\"\n    df = pd.read_csv(path, low_memory=False, dtype={\"label\": \"string\"})\n    inertial = df[INERTIAL_COLS].to_numpy(np.float64).reshape(\n        len(df), N_SENSORS, 3\n    )\n    labels = _read_label_ids(df[\"label\"])\n    return inertial, labels\n\n\ndef _load_video_npy(path: Path) -> np.ndarray:\n    return np.load(path, mmap_mode=\"r\")\n\n\ndef load_train_recording(paths: Paths, stem: str) -> Recording:\n    \"\"\"Load one recording by filename stem, e.g. ``sbj_0`` or ``sbj_0_2``.\"\"\"\n    inertial_path = paths.train_inertial_dir / f\"{stem}.csv\"\n    video_path = paths.train_video_dir / f\"{stem}.npy\"\n\n    inertial, labels = _load_inertial_csv(inertial_path)\n    video = _load_video_npy(video_path)\n\n    sbj_id = _sbj_id_from_stem(stem)\n    return Recording(sbj_id, inertial, video, labels)\n\n\ndef _sbj_id_from_stem(stem: str) -> int:\n    import re\n\n    m = re.match(r\"sbj_(\\d+)\", stem)\n    return int(m.group(1))\n\n\ndef train_stems(paths: Paths) -> list[str]:\n    \"\"\"Return sorted train recording stems (e.g. ['sbj_0', 'sbj_0_2', ...]).\"\"\"\n    stems = [p.name[:-4] for p in paths.train_inertial_dir.glob(\"*.csv\")]\n    return sorted(stems)\n\n\ndef subject_ids_for_stems(stems: list[str]) -> dict[int, list[str]]:\n    \"\"\"Map subject id -> list of recording stems for that subject.\"\"\"\n    subj: dict[int, list[str]] = {}\n    for s in stems:\n        subj.setdefault(_sbj_id_from_stem(s), []).append(s)\n    return subj\n\n\ndef build_windows(rec: Recording, stride: int) -> pd.DataFrame:\n    \"\"\"Generate window metadata for a recording (matches the reference).\n\n    Windows are drawn *within contiguous label segments* (so a window never\n    straddles two different activities), aligned to the segment start.  This\n    mirrors the proven reference pipeline.\n\n    Returns a DataFrame with columns:\n        rec, sbj_id, target, inertial_start, video_start\n    \"\"\"\n    T = rec.n_samples\n    labels = rec.labels\n\n    # Segment boundaries: where the label changes.\n    bounds = np.flatnonzero(labels[1:] != labels[:-1]) + 1\n    seg_starts = np.concatenate(([0], bounds))\n    seg_ends = np.concatenate((bounds, [T]))\n\n    starts_all, targets_all, video_all = [], [], []\n    for s, e in zip(seg_starts, seg_ends):\n        lab = labels[s]\n        if e - s < INERTIAL_WINDOW:\n            continue\n        first = -(-s // stride) * stride          # ceil(s / stride) * stride\n        if first + INERTIAL_WINDOW > e:\n            continue\n        seg_starts_idx = np.arange(first, e - INERTIAL_WINDOW + 1, stride)\n        starts_all.append(seg_starts_idx)\n        targets_all.append(np.full(len(seg_starts_idx), lab, dtype=np.int64))\n        video_all.append((seg_starts_idx * 3 // 5 + VIDEO_FRAME_OFFSET).astype(np.int64))\n\n    if not starts_all:\n        return pd.DataFrame(columns=[\"rec\", \"sbj_id\", \"target\",\n                                     \"inertial_start\", \"video_start\"])\n\n    starts = np.concatenate(starts_all)\n    return pd.DataFrame(\n        {\n            \"rec\": np.repeat(\"\", len(starts)),\n            \"sbj_id\": np.repeat(rec.sbj_id, len(starts)),\n            \"target\": np.concatenate(targets_all),\n            \"inertial_start\": starts,\n            \"video_start\": np.concatenate(video_all),\n        }\n    )\n\n\ndef build_dataset_windows(paths: Paths, stride: int) -> pd.DataFrame:\n    \"\"\"Build the full window metadata table for all train recordings.\"\"\"\n    frames = []\n    for stem in train_stems(paths):\n        rec = load_train_recording(paths, stem)\n        w = build_windows(rec, stride)\n        w[\"rec\"] = stem\n        frames.append(w)\n    return pd.concat(frames, ignore_index=True)\n\n\nclass InertialNormalizer:\n    \"\"\"Per-(sensor, axis) z-score computed from training data.\n\n    Applied identically to train and test windows so that scaling is\n    consistent across the model's inputs (critical for cross-subject\n    generalization with unseen test subjects).\n    \"\"\"\n\n    def __init__(self, mean: np.ndarray | None = None,\n                 std: np.ndarray | None = None):\n        # mean/std shape (N_SENSORS, 3)\n        self.mean = mean\n        self.std = std\n\n    @classmethod\n    def fit(cls, paths: Paths, max_frames: int = 3_000_000,\n            stems: list[str] | None = None) -> \"InertialNormalizer\":\n        \"\"\"Compute per-(sensor,axis) mean/std from finite train samples.\n\n        If ``stems`` is given, only those recording stems are used (fold-only\n        fitting avoids normalizer leakage across grouped CV folds).\n        \"\"\"\n        if stems is None:\n            stems = train_stems(paths)\n        sums = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        sumsq = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        count = np.zeros((N_SENSORS, 3), dtype=np.float64)\n        for stem in stems:\n            rec = load_train_recording(paths, stem)\n            data = rec.inertial.reshape(-1, N_SENSORS, 3)  # (T,4,3)\n            valid = np.isfinite(data)\n            v = np.where(valid, data, 0.0)\n            cnt = valid.sum(axis=0)\n            sums += v.sum(axis=0)\n            sumsq += (v * v).sum(axis=0)\n            count += cnt\n            if count.min() > max_frames:\n                break\n        count = np.maximum(count, 1e-8)\n        mean = sums / count\n        var = np.maximum(sumsq / count - mean * mean, 1e-8)\n        std = np.sqrt(var)\n        return cls(mean.astype(np.float32), std.astype(np.float32))\n\n    def transform(self, inertial: np.ndarray) -> np.ndarray:\n        \"\"\"inertial: (..., N_SENSORS, 3) -> standardized (in-place copy).\"\"\"\n        out = inertial.astype(np.float32, copy=True)\n        out = np.where(np.isfinite(out), out, self.mean)\n        out = (out - self.mean) / self.std\n        return out\n\n    def save(self, path: Path) -> None:\n        np.savez(path, mean=self.mean, std=self.std)\n\n    @classmethod\n    def load(cls, path: Path) -> \"InertialNormalizer\":\n        d = np.load(path)\n        return cls(d[\"mean\"], d[\"std\"])\n\n\nclass PerWindowNormalizer:\n    \"\"\"Normalise each window independently per (sensor, axis).\n\n    For every 1-second window we subtract the window's own per-axis mean and\n    divide by its per-axis std (across the 50 time samples).  This is robust to\n    arbitrary per-subject sensor scale/offset (calibration differences), which\n    is likely the source of the train/test inertial shift for the unseen test\n    subjects.  Needs no statistics from the unseen subjects.\n    \"\"\"\n\n    def __init__(self, eps: float = 1e-4):\n        self.eps = eps\n\n    def transform(self, inertial: np.ndarray) -> np.ndarray:\n        \"\"\"inertial: (..., T, N_SENSORS, 3) or (..., T, 3).\"\"\"\n        out = inertial.astype(np.float32, copy=True)\n        out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)\n        axes = tuple(range(out.ndim - 2))       # all but (time, feat)\n        mean = out.mean(axis=-2, keepdims=True)\n        std = out.std(axis=-2, keepdims=True)\n        out = (out - mean) / (std + self.eps)\n        return out\n\n\ndef subsample_windows(windows: pd.DataFrame,\n                      max_per_class_per_subject: int | None,\n                      seed: int) -> pd.DataFrame:\n    \"\"\"Balance-ish subsample: cap windows per (sbj, class) group.\"\"\"\n    if max_per_class_per_subject is None:\n        return windows\n    df = windows.copy()\n    # Shuffle deterministically, then keep the first `max` per (sbj, class).\n    rng = np.random.RandomState(seed)\n    df[\"_r\"] = rng.rand(len(df))\n    df = df.sort_values(\"_r\")\n    df[\"_n\"] = df.groupby([\"sbj_id\", \"target\"])[\"_r\"].cumcount()\n    df = df[df[\"_n\"] < max_per_class_per_subject]\n    df = df.drop(columns=[\"_r\", \"_n\"])\n    return df.reset_index(drop=True)\n\n\nclass WearTrainDataset:\n    \"\"\"PyTorch dataset yielding (inertial (50,4,3), video (15,768), target, valid).\"\"\"\n\n    def __init__(self, windows: pd.DataFrame, paths: Paths,\n                 normalizer: InertialNormalizer | None = None,\n                 augment: bool = False, data_cfg=None):\n        self.windows = windows.reset_index(drop=True)\n        self.paths = paths\n        self.normalizer = normalizer\n        self.augment = augment\n        self.dcfg = data_cfg\n        self._rec_cache: dict[str, Recording] = {}\n\n    def _get_rec(self, rec: str) -> Recording:\n        if rec not in self._rec_cache:\n            self._rec_cache[rec] = load_train_recording(self.paths, rec)\n        return self._rec_cache[rec]\n\n    def __len__(self) -> int:\n        return len(self.windows)\n\n    def __getitem__(self, idx: int):\n        row = self.windows.iloc[idx]\n        rec = self._get_rec(row[\"rec\"])\n\n        i0 = int(row[\"inertial_start\"])\n        # Optional random time shift (keeps the window within the recording).\n        if self.augment and self.dcfg is not None and self.dcfg.augment_shift:\n            shift = int(np.random.randint(-self.dcfg.augment_shift,\n                                          self.dcfg.augment_shift + 1))\n            lo = max(0, i0 + shift)\n            hi = lo + INERTIAL_WINDOW\n            if hi > rec.n_samples:\n                hi = rec.n_samples\n                lo = hi - INERTIAL_WINDOW\n            inertial = np.asarray(rec.inertial[lo:hi],\n                                  dtype=np.float32).copy()\n            # Re-sync video start with the (possibly shifted) inertial start.\n            i0 = lo\n        else:\n            inertial = np.asarray(rec.inertial[i0:i0 + INERTIAL_WINDOW],\n                                  dtype=np.float32).copy()\n\n        # Compute sensor validity from the raw signal (before any fill).\n        valid = np.isfinite(inertial).all(axis=(0, 2))  # (4,) which sensors valid\n\n        if self.normalizer is not None:\n            inertial = self.normalizer.transform(inertial)\n\n        if self.augment and self.dcfg is not None:\n            s = 1.0 + float(np.random.uniform(-self.dcfg.augment_scale,\n                                              self.dcfg.augment_scale))\n            inertial = inertial * s\n            if self.dcfg.augment_noise:\n                inertial = inertial + np.random.randn(*inertial.shape).astype(\n                    np.float32) * self.dcfg.augment_noise\n\n        # Fill non-finite values (missing sensors) with 0 as the reference does,\n        # so conv/pool layers never receive NaN.\n        inertial = np.nan_to_num(inertial, nan=0.0, posinf=0.0, neginf=0.0)\n\n        v0 = int(row[\"video_start\"])\n        video = np.asarray(rec.video[v0:v0 + VIDEO_WINDOW],\n                           dtype=np.float32).copy()  # (15,768) writable\n\n        return {\n            \"inertial\": inertial,\n            \"video\": video,\n            \"target\": int(row[\"target\"]),\n            \"valid\": valid,\n        }\n\n\ndef build_test_dataset(paths: Paths):\n    \"\"\"Return test tensors + metadata: (inertial (N,50,3), video (N,15,768), sensor_ids (N,), ids (N,), sbj (N,)).\"\"\"\n    inertial = np.load(paths.test_inertial, mmap_mode=\"r\")  # (N,50,3) or (N,3,50)\n    # Transpose inertial if stored as (N, 3, 50) per window.\n    if inertial.ndim == 3 and inertial.shape[1] == 3 and inertial.shape[2] == 50:\n        inertial = np.transpose(inertial, (0, 2, 1))        # (N,50,3)\n    # The raw test video is stored per-window as (N, 768, 15); transpose to\n    # (N, 15, 768) to match the model's expected frame-first layout.\n    video = np.load(paths.test_video, mmap_mode=\"r\")        # (N,768,15)\n    if video.ndim == 3 and video.shape[2] == 15 and video.shape[1] == 768:\n        video = np.transpose(video, (0, 2, 1))              # (N,15,768)\n    meta = pd.read_csv(paths.test_meta)\n\n    sensor_ids = meta[\"sensor_location\"].map(SENSOR_TO_ID).to_numpy(np.int64)\n    ids = meta[\"id\"].to_numpy(np.int64)\n    sbj = meta[\"sbj_id\"].to_numpy(np.int64)\n    return inertial, video, sensor_ids, ids, sbj",
    "models": "\"\"\"Fusion model for inertial + VideoMAE features.\n\nDesign notes\n------------\n* Test windows carry a *single* inertial sensor location while training\n  carries all four.  We therefore use a *shared* per-sensor inertial encoder\n  and condition predictions on a learned sensor-location embedding, so the\n  model can make a prediction from one or many sensors.\n\n* The inertial encoder is a multi-scale 1D-Inception CNN (parallel kernels)\n  with channel attention and temporal global pooling, applied to\n  (4 channels = 3 axes + magnitude).\n\n* The video encoder projects the 768-d VideoMAE frames to a small hidden dim,\n  passes them through a lightweight Transformer, and attention-pools across\n  the 15 frames.\n\n* Fusion is a learned gated blend of inertial and video embeddings plus an\n  additive interaction term, followed by an MLP classifier (19 classes).\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\n# ---------------------------------------------------------------------------\n# Building blocks\n# ---------------------------------------------------------------------------\nclass ConvNormAct(nn.Module):\n    def __init__(self, cin: int, cout: int, kernel: int, groups: int = 1,\n                 stride: int = 1):\n        super().__init__()\n        self.conv = nn.Conv1d(cin, cout, kernel, stride=stride,\n                              padding=kernel // 2, groups=groups, bias=False)\n        self.norm = nn.GroupNorm(min(8, cout), cout)\n        self.act = nn.GELU()\n\n    def forward(self, x):\n        return self.act(self.norm(self.conv(x)))\n\n\nclass InceptionBlock(nn.Module):\n    \"\"\"Multi-scale conv block (kernels 5/11/21) with a 1x1 pooling branch.\"\"\"\n\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        # Branch width chosen so concatenation == out.\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)          # (B, bottleneck*4, T)\n        out = out + self.skip(x)\n        return self.dropout(F.gelu(self.norm(out)))\n\n\nclass ChannelAttention(nn.Module):\n    \"\"\"Squeeze-excite over channels.\"\"\"\n\n    def __init__(self, channels: int, reduction: int = 8):\n        super().__init__()\n        self.fc = nn.Sequential(\n            nn.Linear(channels, max(8, channels // reduction)),\n            nn.ReLU(inplace=True),\n            nn.Linear(max(8, channels // reduction), channels),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):  # (B,C,T)\n        w = x.mean(dim=2)  # (B,C)\n        w = self.fc(w).unsqueeze(-1)\n        return x * w\n\n\nclass InertialEncoder(nn.Module):\n    \"\"\"Shared per-sensor encoder: (B, 4, 50) -> (B, hidden).\"\"\"\n\n    def __init__(self, hidden: int, blocks: int, channels: int,\n                 dropout: float):\n        super().__init__()\n        # Input: 4 channels (x,y,z,magnitude)\n        layers = [InceptionBlock(4, channels)]\n        for _ in range(blocks - 1):\n            layers.append(InceptionBlock(channels, channels))\n        self.blocks = nn.Sequential(*layers)\n        self.attention = ChannelAttention(channels)\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B, 4, 50)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)  # (B, 4, 50)\n        h = self.blocks(x)              # (B, C, 50)\n        h = self.attention(h)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)  # (B, 2C)\n        return self.head(pooled)\n\n\nclass VideoEncoder(nn.Module):\n    \"\"\"(B, 15, 768) -> (B, hidden) with Transformer + attention pooling.\"\"\"\n\n    def __init__(self, hidden: int, layers: int, heads: int, dropout: float):\n        super().__init__()\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM),\n            nn.Linear(VIDEO_DIM, hidden),\n            nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            hidden, heads, dim_feedforward=hidden * 4, dropout=dropout,\n            activation=\"gelu\", batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(\n            layer, layers, enable_nested_tensor=False\n        )\n        self.attn = nn.Sequential(\n            nn.Linear(hidden, 64), nn.Tanh(), nn.Linear(64, 1)\n        )\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B, 15, 768)\n        h = self.projection(x) + self.position\n        h = self.transformer(h)\n        scores = self.attn(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)       # (B, hidden)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\nclass WearFusionNet(nn.Module):\n    \"\"\"Single multi-sensor fusion model.\"\"\"\n\n    def __init__(self, cfg, n_classes: int = N_CLASSES):\n        super().__init__()\n        m = cfg\n        # Apply the global width multiplier to the hidden dims.\n        s = getattr(m, \"scale\", 1.0)\n        i_hidden = int(m.inertial_hidden * s)\n        i_channels = int(m.inertial_channels * s)\n        v_hidden = int(m.video_hidden * s)\n        c_hidden = int(m.classifier_hidden * s)\n\n        self.inertial_encoder = InertialEncoder(\n            i_hidden, m.inertial_blocks, i_channels, m.dropout\n        )\n        self.video_encoder = VideoEncoder(\n            v_hidden, m.video_transformer_layers, m.video_heads, m.dropout\n        )\n        self.sensor_embed = nn.Embedding(N_SENSORS, m.sensor_embed_dim)\n\n        gate_in = i_hidden + v_hidden + m.sensor_embed_dim\n        self.gate = nn.Sequential(\n            nn.Linear(gate_in, i_hidden), nn.Sigmoid()\n        )\n        cls_in = i_hidden + i_hidden + m.sensor_embed_dim\n        self.classifier = nn.Sequential(\n            nn.Linear(cls_in, c_hidden),\n            nn.LayerNorm(c_hidden),\n            nn.GELU(),\n            nn.Dropout(m.dropout),\n            nn.Linear(c_hidden, n_classes),\n        )\n        self.modality_dropout = m.modality_dropout\n        self.sensor_dropout = 0.2   # probability of zeroing a sensor in training\n\n    def forward(self, inertial, video, active_mask=None):\n        \"\"\"inertial: (B, 50, 4, 3) | video: (B, 15, 768).\n\n        ``active_mask``: optional (B, 4) float tensor in {0,1} indicating which\n        sensor slots are present (used to mimic the single-sensor test setup).\n        Returns per-sensor logits (B, 4, n_classes).\n        \"\"\"\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 3, 1)         # (B, 4, 50, 3)\n\n        if active_mask is not None:\n            inertial = inertial * active_mask[:, :, None, None].float()\n        elif self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inertial.device) >= self.sensor_dropout\n            keep = keep[:, :, None, None]\n            inertial = inertial * keep.float()\n\n        inertial = inertial.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        inertial_feat = self.inertial_encoder(inertial)   # (B*4, hidden)\n        inertial_feat = inertial_feat.reshape(B, N_SENSORS, -1)\n\n        # Video: shared across sensors, expanded.\n        video_feat = self.video_encoder(video)           # (B, hidden)\n        video_feat = video_feat[:, None].expand(B, N_SENSORS, -1)\n\n        # Fixed per-slot sensor position embedding (discriminative location).\n        pos = self.sensor_embed.weight[None, :, :].expand(B, N_SENSORS, -1)\n\n        if self.training:\n            inertial_feat, video_feat = self._modality_dropout(\n                inertial_feat, video_feat, self.modality_dropout\n            )\n\n        gate_in = torch.cat([inertial_feat, video_feat, pos], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * inertial_feat + (1 - gate) * video_feat\n        interaction = inertial_feat * video_feat\n        features = torch.cat([fused, interaction, pos], dim=-1)  # (B,4,cls_in)\n\n        logits = self.classifier(features)               # (B,4,19)\n        return logits\n\n    @staticmethod\n    def _modality_dropout(i_feat, v_feat, p: float):\n        \"\"\"Zero out an entire modality during training with prob p.\"\"\"\n        if p <= 0:\n            return i_feat, v_feat\n        scale = 1.0 / (1.0 - p)\n        # (B,4,1)\n        drop_i = (torch.rand(i_feat.shape[:2], device=i_feat.device) < p)[..., None]\n        drop_v = (torch.rand(v_feat.shape[:2], device=v_feat.device) < p)[..., None]\n        i_feat = torch.where(drop_i, torch.zeros_like(i_feat), i_feat * scale)\n        v_feat = torch.where(drop_v, torch.zeros_like(v_feat), v_feat * scale)\n        return i_feat, v_feat\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)",
    "train": "\"\"\"Training loop with grouped (participant-independent) CV and AMP.\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_SENSORS, N_CLASSES\nfrom .data import (\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .models import WearFusionNet, count_parameters\n\n\n@dataclass\nclass FoldResult:\n    fold: int\n    oof_targets: np.ndarray\n    oof_logits: np.ndarray       # (n, n_sensors, n_classes) or (n, n_classes)\n    macro_f1: float\n    seed: int\n\n\ndef _worker_init(seed: int):\n    def _init(worker_id):\n        np.random.seed(seed + worker_id)\n    return _init\n\n\ndef _random_single_sensor_mask(valid, device=None):\n    \"\"\"Return a (B, 4) bool mask picking a random valid sensor/window.\"\"\"\n    B, S = valid.shape\n    vf = valid.to(torch.float32)\n    noise = torch.rand(B, S, device=vf.device)\n    cand = (vf - 1.0) * 1e9 + noise\n    idx = cand.argmax(dim=1)                      # (B,)\n    mask = torch.zeros(B, S, dtype=torch.bool, device=vf.device)\n    mask[torch.arange(B, device=vf.device), idx] = True\n    return mask\n\n\ndef make_dataloaders(cfg: Config, train_idx, val_idx, windows, paths,\n                     seed: int, normalizer=None):\n    train_ws = windows.iloc[train_idx].reset_index(drop=True)\n    val_ws = windows.iloc[val_idx].reset_index(drop=True)\n\n    train_ds = WearTrainDataset(train_ws, paths, normalizer=normalizer,\n                                augment=True, data_cfg=cfg.data)\n    val_ds = WearTrainDataset(val_ws, paths, normalizer=normalizer,\n                              augment=False, data_cfg=cfg.data)\n\n    train_dl = DataLoader(\n        train_ds, batch_size=cfg.train.batch_size, shuffle=True,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n    val_dl = DataLoader(\n        val_ds, batch_size=cfg.train.batch_size * 2, shuffle=False,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n    return train_dl, val_dl\n\n\ndef grouped_split(subject_ids: np.ndarray, n_val_subjects: int = 4,\n                  seed: int = 42):\n    \"\"\"Leave-out a disjoint set of subjects -> (train_idx, val_idx).\"\"\"\n    subjects = np.unique(subject_ids)\n    rng = np.random.RandomState(seed)\n    val_subjects = set(rng.choice(subjects, size=n_val_subjects, replace=False))\n    val_idx = np.where(np.isin(subject_ids, list(val_subjects)))[0]\n    train_idx = np.where(~np.isin(subject_ids, list(val_subjects)))[0]\n    return train_idx, val_idx, val_subjects\n\n\ndef train_one_fold(cfg: Config, windows, paths, train_idx, val_idx,\n                   fold: int, seed: int, device: torch.device,\n                   out_dir: Path, normalizer=None) -> FoldResult:\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n\n    model = WearFusionNet(cfg.model).to(device)\n    n_params = count_parameters(model)\n    print(f\"[fold {fold}] params={n_params/1e6:.3f}M\")\n\n    train_dl, val_dl = make_dataloaders(cfg, train_idx, val_idx, windows,\n                                        paths, seed, normalizer=normalizer)\n\n    # Class weights for macro-F1 oriented training.\n    train_targets = windows.iloc[train_idx][\"target\"].to_numpy()\n    class_w = class_weights_from_counts(count_class_labels(train_targets),\n                                        cfg.train.class_weighting,\n                                        device=device)\n    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing,\n                                    weight=class_w)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay\n    )\n    total_steps = cfg.train.epochs * len(train_dl)\n    warmup_steps = cfg.train.warmup_epochs * len(train_dl)\n\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return step / max(1, warmup_steps)\n        # cosine decay\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        return 0.5 * (1 + math_cos(progress * math_pi()))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=cfg.train.amp and device.type == \"cuda\")\n    use_amp = cfg.train.amp and device.type == \"cuda\"\n\n    model.train()\n    global_step = 0\n    t0 = time.time()\n    for epoch in range(cfg.train.epochs):\n        running = 0.0\n        nb = 0\n        for batch in train_dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)          # (B, 4)\n\n            # Single-sensor conditioning: with probability p, present only one\n            # random (valid) sensor so the model learns to predict from a\n            # single sensor, matching the 1-sensor test windows.\n            active_mask = None\n            if cfg.data.single_sensor_prob > 0 and \\\n                    torch.rand(1).item() < cfg.data.single_sensor_prob:\n                active_mask = _random_single_sensor_mask(\n                    valid, device=device)\n                inertial = inertial * active_mask[:, None, :, None].float()\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n                # Per-sensor loss restricted to the active/present sensors.\n                tgt = target[:, None].expand(-1, N_SENSORS).reshape(-1)\n                logits = logits.reshape(-1, logits.shape[-1])\n                if active_mask is not None:\n                    valid_flat = active_mask.reshape(-1)\n                else:\n                    valid_flat = valid.reshape(-1)\n                if valid_flat.any():\n                    loss = criterion(logits[valid_flat], tgt[valid_flat])\n                else:\n                    loss = torch.tensor(0.0, device=device)\n\n            scaler.scale(loss).backward()\n            if cfg.train.max_grad_norm:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(),\n                                               cfg.train.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            global_step += 1\n            running += loss.item() * (1.0 if valid_flat.any() else 1.0)\n            nb += 1\n\n        print(f\"  [fold {fold}] epoch {epoch+1}/{cfg.train.epochs} \"\n              f\"loss={running/max(1,nb):.4f} lr={scheduler.get_last_lr()[0]:.2e} \"\n              f\"time={time.time()-t0:.0f}s\")\n\n    # ---- Validation OOF (single-sensor, matching test) ----\n    model.eval()\n    all_logits = []\n    all_targets = []\n    with torch.inference_mode():\n        for batch in val_dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].numpy()\n            valid = batch[\"valid\"].to(device)\n            # Present a random single valid sensor per window (as in test).\n            active_mask = _random_single_sensor_mask(valid, device=device)\n            inertial = inertial * active_mask[:, None, :, None].float()\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n                # Take only the active sensor's logits.\n                active_idx = active_mask.long().argmax(dim=1)\n                logits = logits[torch.arange(logits.shape[0], device=device),\n                                active_idx, :]\n            all_logits.append(logits.float().cpu().numpy())\n            all_targets.append(target)\n\n    logits = np.concatenate(all_logits, axis=0)   # (N,19) single-sensor logits\n    targets = np.concatenate(all_targets, axis=0)  # (N,)\n\n    probs = softmax(logits, axis=-1)              # (N,19)\n    preds = probs.argmax(axis=1)\n    f1 = macro_f1(targets, preds, n_classes=N_CLASSES)\n\n    return FoldResult(\n        fold=fold,\n        oof_targets=targets,\n        oof_logits=probs,\n        macro_f1=f1,\n        seed=seed,\n    )\n\n\n# ---------------------------------------------------------------------------\n# Small helpers\n# ---------------------------------------------------------------------------\ndef class_weights_from_counts(counts: np.ndarray, mode: str,\n                              n_classes: int = N_CLASSES,\n                              device=None) -> torch.Tensor | None:\n    \"\"\"Compute per-class loss weights for macro-F1 oriented training.\"\"\"\n    if mode == \"none\":\n        return None\n    counts = counts.astype(np.float64) + 1.0\n    w = 1.0 / counts\n    if mode == \"sqrt_inverse\":\n        w = 1.0 / np.sqrt(counts)\n    # Normalize so mean weight is 1.\n    w = w / w.mean()\n    t = torch.as_tensor(w, dtype=torch.float32)\n    return t.to(device) if device is not None else t\n\n\ndef count_class_labels(targets: np.ndarray, n_classes: int = N_CLASSES) -> np.ndarray:\n    counts = np.bincount(targets, minlength=n_classes).astype(np.float64)\n    return counts\n\n\ndef math_cos(x):\n    return float(np.cos(x))\n\n\ndef math_pi():\n    return float(np.pi)\n\n\ndef softmax(x, axis=-1):\n    e = np.exp(x - x.max(axis=axis, keepdims=True))\n    return e / e.sum(axis=axis, keepdims=True)\n\n\ndef macro_f1(y_true, y_pred, n_classes=None):\n    from sklearn.metrics import f1_score\n\n    if n_classes is None:\n        n_classes = max(y_true.max(), y_pred.max()) + 1\n    return float(f1_score(y_true, y_pred, average=\"macro\", labels=list(range(n_classes))))\n\n\ndef run_cv(cfg: Config, n_folds: int = 1, n_val_subjects: int = 4,\n           device=None, out_dir: Path | None = None):\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"device={device}\")\n\n    out_dir = out_dir or (cfg.paths.output_dir / \"cv\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building window table...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"windows total = {len(windows)}\")\n\n    # Fit a global inertial normalizer once from training data.\n    from .data import InertialNormalizer\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    print(f\"normalizer mean/ (sensor,axis): {normalizer.mean.shape}\")\n\n    subject_ids = windows[\"sbj_id\"].to_numpy()\n    results = []\n    for fold in range(n_folds):\n        train_idx, val_idx, val_subjects = grouped_split(\n            subject_ids, n_val_subjects=n_val_subjects, seed=cfg.data.seed + fold\n        )\n        print(f\"\\n=== fold {fold}: val subjects = {sorted(val_subjects)} \"\n              f\"train={len(train_idx)} val={len(val_idx)} ===\")\n        res = train_one_fold(cfg, windows, cfg.paths, train_idx, val_idx,\n                             fold, cfg.train.seed + fold, device, out_dir,\n                             normalizer=normalizer)\n        print(f\"fold {fold} macro-F1 = {res.macro_f1:.4f}\")\n        results.append(res)\n\n    mean_f1 = float(np.mean([r.macro_f1 for r in results]))\n    print(f\"\\nmean macro-F1 over {n_folds} folds = {mean_f1:.4f}\")\n\n    # Optimise per-class logit offsets on the pooled OOF set.\n    from .tuning import optimize_class_offsets, apply_offsets, argmax_preds\n    if n_folds > 0 and results:\n        all_probs = np.concatenate([r.oof_logits for r in results], axis=0)\n        all_targets = np.concatenate([r.oof_targets for r in results], axis=0)\n        offsets = optimize_class_offsets(all_probs, all_targets,\n                                         n_classes=N_CLASSES)\n        tuned_f1 = macro_f1(all_targets,\n                            argmax_preds(apply_offsets(all_probs, offsets)))\n        print(f\"\\n[OOF tuning] raw macro-F1={mean_f1:.4f} | \"\n              f\"offset-tuned macro-F1={tuned_f1:.4f}\")\n        print(\"offsets:\", np.round(offsets, 3).tolist())\n        json_summary = {\n            \"folds\": [{\"fold\": r.fold, \"macro_f1\": r.macro_f1}\n                      for r in results],\n            \"mean_macro_f1\": mean_f1,\n            \"offset_tuned_macro_f1\": tuned_f1,\n            \"class_offsets\": offsets.tolist(),\n        }\n    else:\n        json_summary = {\"folds\": [], \"mean_macro_f1\": mean_f1}\n\n    cfg.save(out_dir / \"config.json\")\n    (out_dir / \"cv_summary.json\").write_text(\n        json.dumps(json_summary, indent=2)\n    )\n    return results, mean_f1\n\n\ndef train_full(cfg: Config, device=None, out_path: Path | None = None):\n    \"\"\"Train on the entire training set (no holdout) and return the model.\n\n    Used for the final submission model once hyper-parameters are fixed.\n    \"\"\"\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"full-data windows = {len(windows)}\")\n\n    from .data import InertialNormalizer\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer,\n                          augment=True, data_cfg=cfg.data)\n    dl = DataLoader(\n        ds, batch_size=cfg.train.batch_size, shuffle=True,\n        num_workers=cfg.train.num_workers, pin_memory=True,\n        worker_init_fn=_worker_init(cfg.train.seed),\n        persistent_workers=(cfg.train.num_workers > 0),\n    )\n\n    torch.manual_seed(cfg.train.seed)\n    model = WearFusionNet(cfg.model).to(device)\n    class_w = class_weights_from_counts(count_class_labels(windows[\"target\"].to_numpy()),\n                                        cfg.train.class_weighting, device=device)\n    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing,\n                                    weight=class_w)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay\n    )\n    total_steps = cfg.train.epochs * len(dl)\n    warmup_steps = cfg.train.warmup_epochs * len(dl)\n\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return step / max(1, warmup_steps)\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        return 0.5 * (1 + math_cos(progress * math_pi()))\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n    scaler = torch.amp.GradScaler(\"cuda\",\n                                  enabled=cfg.train.amp and device.type == \"cuda\")\n    use_amp = cfg.train.amp and device.type == \"cuda\"\n\n    model.train()\n    t0 = time.time()\n    for epoch in range(cfg.train.epochs):\n        running = 0.0\n        nb = 0\n        for batch in dl:\n            inertial = batch[\"inertial\"].to(device)\n            video = batch[\"video\"].to(device)\n            target = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n\n            active_mask = None\n            if cfg.data.single_sensor_prob > 0 and \\\n                    torch.rand(1).item() < cfg.data.single_sensor_prob:\n                active_mask = _random_single_sensor_mask(valid, device=device)\n                inertial = inertial * active_mask[:, None, :, None].float()\n\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)\n                tgt = target[:, None].expand(-1, N_SENSORS).reshape(-1)\n                logits = logits.reshape(-1, logits.shape[-1])\n                valid_flat = (active_mask if active_mask is not None else valid).reshape(-1)\n                loss = criterion(logits[valid_flat], tgt[valid_flat]) if valid_flat.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if cfg.train.max_grad_norm:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(),\n                                               cfg.train.max_grad_norm)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            running += loss.item()\n            nb += 1\n        print(f\"[full] epoch {epoch+1}/{cfg.train.epochs} \"\n              f\"loss={running/max(1,nb):.4f} time={time.time()-t0:.0f}s\")\n\n    if out_path is not None:\n        out_path = Path(out_path)\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        # Save model config as a plain dict so the checkpoint stays\n        # picklable with torch's default weights_only=True.\n        from dataclasses import asdict\n        torch.save({\"model\": model.state_dict(),\n                    \"cfg\": asdict(cfg.model),\n                    \"normalizer_mean\": normalizer.mean.tolist(),\n                    \"normalizer_std\": normalizer.std.tolist()},\n                   out_path)\n        print(f\"saved model -> {out_path}\")\n    return model\n\n\ndef load_model_for_inference(model_path: Path, device) -> WearFusionNet:\n    \"\"\"Load a saved full-data model.\"\"\"\n    ckpt = torch.load(model_path, map_location=device, weights_only=True)\n    from .config import Config as _C, ModelConfig\n    model_cfg = ModelConfig(**ckpt[\"cfg\"])\n    model = WearFusionNet(model_cfg).to(device)\n    model.load_state_dict(ckpt[\"model\"])\n    model.eval()\n    model._normalizer_mean = np.asarray(ckpt.get(\"normalizer_mean\"), dtype=np.float32)\n    model._normalizer_std = np.asarray(ckpt.get(\"normalizer_std\"), dtype=np.float32)\n    return model",
    "inference": "\"\"\"Test-time inference and submission generation.\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .config import (\n    INERTIAL_WINDOW,\n    N_CLASSES,\n    N_SENSORS,\n    VIDEO_DIM,\n    VIDEO_WINDOW,\n    Config,\n)\nfrom .data import build_test_dataset\nfrom .models import WearFusionNet\n\n\nclass WearTestDataset(Dataset):\n    def __init__(self, paths, null_video: bool = False, normalizer=None):\n        inertial, video, sensor_ids, ids, sbj = build_test_dataset(paths)\n        self.sensor_ids = sensor_ids\n        self.ids = ids\n        self.inertial = np.nan_to_num(np.asarray(inertial, dtype=np.float32).copy())\n        if normalizer is not None:\n            if hasattr(normalizer, \"mean\"):\n                # Global per-(sensor,axis) normalizer: use the window's sensor.\n                mean = normalizer.mean[sensor_ids]   # (N, 3)\n                std = normalizer.std[sensor_ids]     # (N, 3)\n                self.inertial = (self.inertial - mean[:, None, :]) / std[:, None, :]\n            else:\n                # Generic per-window normalizer (e.g. PerWindowNormalizer).\n                self.inertial = normalizer.transform(self.inertial)\n        self.video = np.nan_to_num(np.asarray(video, dtype=np.float32).copy())\n        if null_video:\n            self.video = np.zeros_like(self.video)\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, idx):\n        return {\n            \"inertial\": self.inertial[idx],      # (50,3)\n            \"video\": self.video[idx],            # (15,768)\n            \"sensor_id\": self.sensor_ids[idx],\n            \"id\": self.ids[idx],\n        }\n\n\ndef _collate(batch):\n    inertial = torch.as_tensor(\n        np.stack([b[\"inertial\"] for b in batch])\n    )                       # (B,50,3)\n    video = torch.as_tensor(np.stack([b[\"video\"] for b in batch]))\n    sensor_id = torch.as_tensor([b[\"sensor_id\"] for b in batch])\n    ids = torch.as_tensor([b[\"id\"] for b in batch])\n    return inertial, video, sensor_id, ids\n\n\ndef predict_test(model: WearFusionNet, paths, device, batch_size=512,\n                 null_bias: float = 0.0, video_weight: float = 1.0,\n                 amp=True, normalizer=None):\n    \"\"\"Return (ids, probs (N,19)). Applies null bias to class 0.\"\"\"\n    ds = WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,\n                    collate_fn=_collate)\n    model.eval()\n    use_amp = amp and device.type == \"cuda\"\n\n    all_probs = []\n    all_ids = []\n    with torch.inference_mode():\n        for inertial, video, sensor_id, ids in dl:\n            inertial = inertial.to(device)\n            video = video.to(device)\n            sensor_id = sensor_id.to(device)\n            # reshape single sensor -> (B,50,4,3); NaN out non-present sensors\n            inertial = inertial.unsqueeze(2).expand(\n                -1, -1, N_SENSORS, -1)  # (B,50,4,3)\n            present = torch.zeros(inertial.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inertial = inertial * present[:, None, :, None]\n\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inertial, video)  # (B,4,19)\n            probs = torch.softmax(logits.float(), dim=-1)\n\n            # Use only the prediction from the window's present sensor slot.\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]           # (B,19)\n\n            # Video-weighting (if we ever want to down-weight video).\n            if video_weight != 1.0:\n                # Simple heuristic: blend toward uniform along video axis.\n                pass\n\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n\n    if null_bias:\n        probs = probs.copy()\n        probs[:, 0] *= np.exp(null_bias)\n\n    order = np.argsort(ids)\n    return ids[order], probs[order]\n\n\ndef write_submission(ids, probs, sample_submission_path, out_path):\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    if probs.ndim == 2:\n        preds = probs.argmax(axis=1)\n    else:\n        preds = probs\n    df = pd.DataFrame({\"id\": ids, \"target_feature\": preds})\n    df.to_csv(out_path, index=False)\n    return df\n\n\ndef predict_ensemble(models, paths, device, batch_size=512, null_bias=0.0,\n                     amp=True, normalizer=None):\n    \"\"\"Average probabilities across an ensemble of model objects.\"\"\"\n    ds = WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,\n                    collate_fn=_collate)\n    use_amp = amp and device.type == \"cuda\"\n    acc = None\n    all_ids = None\n    for m in models:\n        m.eval()\n        m_acc = None\n        ids_out = None\n        with torch.inference_mode():\n            for inertial, video, sensor_id, ids in dl:\n                inertial = inertial.to(device)\n                video = video.to(device)\n                sensor_id = sensor_id.to(device)\n                inertial = inertial.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n                present = torch.zeros(inertial.shape[0], N_SENSORS, device=device)\n                present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n                inertial = inertial * present[:, None, :, None]\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = m(inertial, video)\n                probs = torch.softmax(logits.float(), dim=-1)\n                probs = probs[torch.arange(probs.shape[0], device=device),\n                              sensor_id, :].cpu().numpy()\n                m_acc = probs if m_acc is None else np.concatenate([m_acc, probs], 0)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        m_acc = m_acc[np.argsort(ids_out)]\n        acc = m_acc if acc is None else acc + m_acc\n        all_ids = ids_out if all_ids is None else all_ids\n\n    acc = acc / len(models)\n    if null_bias:\n        acc = acc.copy()\n        acc[:, 0] *= np.exp(null_bias)\n    return all_ids, acc",
    "tuning": "\"\"\"OOF-based ensemble blending and per-class threshold optimisation.\n\nThese tools operate on out-of-fold (OOF) probabilities.  Because macro-F1\ntreats all classes equally, a plain argmax is often sub-optimal; a small\nper-class logit offset can meaningfully improve the metric, especially for\nthe dominant ``null`` class.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .train import macro_f1, softmax\n\n\ndef argmax_preds(probs: np.ndarray) -> np.ndarray:\n    return probs.argmax(axis=1)\n\n\ndef apply_offsets(probs: np.ndarray, offsets: np.ndarray) -> np.ndarray:\n    \"\"\"Add a per-class logit offset (i.e. multiply prob by exp(offset)).\"\"\"\n    p = probs * np.exp(offsets[None, :])\n    p = p / p.sum(axis=1, keepdims=True)\n    return p\n\n\ndef optimize_class_offsets(probs: np.ndarray, targets: np.ndarray,\n                           n_classes: int, iters: int = 3,\n                           grid: np.ndarray | None = None) -> np.ndarray:\n    \"\"\"Coordinate-ascent per-class logit offsets to maximise macro-F1.\n\n    Returns ``offsets`` shape (n_classes,).  Works on OOF probabilities.\n    The search is intentionally conservative to limit overfitting: each class\n    is nudged by at most one grid step per iteration and never away from 0\n    unless it strictly improves the metric.\n    \"\"\"\n    if grid is None:\n        grid = np.linspace(-1.0, 1.0, 13)\n    offsets = np.zeros(n_classes, dtype=np.float64)\n    best = macro_f1(targets, argmax_preds(apply_offsets(probs, offsets)))\n    for _ in range(iters):\n        for c in range(n_classes):\n            cur = offsets[c]\n            best_off, best_val = cur, best\n            for g in grid:\n                cand = cur + g\n                offsets[c] = cand\n                p = apply_offsets(probs, offsets)\n                v = macro_f1(targets, argmax_preds(p))\n                if v > best_val + 1e-6:\n                    best_val, best_off = v, cand\n            offsets[c] = best_off\n            best = best_val\n    return offsets\n\n\ndef optimize_ensemble_weights(model_probs: list[np.ndarray],\n                              targets: np.ndarray,\n                              n_classes: int,\n                              iters: int = 50) -> np.ndarray:\n    \"\"\"Iterative weight search for a softmax-probability ensemble.\n\n    ``model_probs``: list of (N, n_classes) probability matrices.\n    Returns non-negative weights summing to 1 that maximise macro-F1.\n    \"\"\"\n    K = len(model_probs)\n    # Start with a probability-averaging init (weights 1/K).\n    w = np.ones(K, dtype=np.float64) / K\n    best = _ensemble_f1(model_probs, w, targets)\n    rng = np.random.RandomState(0)\n    for _ in range(iters):\n        cand = w + rng.normal(0, 0.15, K)\n        cand = np.clip(cand, 0, None)\n        if cand.sum() <= 0:\n            continue\n        cand = cand / cand.sum()\n        v = _ensemble_f1(model_probs, cand, targets)\n        if v > best:\n            best, w = v, cand\n    return w\n\n\ndef _ensemble_f1(model_probs, weights, targets):\n    blended = sum(wi * p for wi, p in zip(weights, model_probs))\n    return macro_f1(targets, argmax_preds(blended))\n\n\ndef combine_models(probs_list: list[np.ndarray],\n                   weights: np.ndarray | None = None) -> np.ndarray:\n    \"\"\"Weighted sum of probability matrices (soft-voting).\"\"\"\n    if weights is None:\n        weights = np.ones(len(probs_list)) / len(probs_list)\n    return sum(w * p for w, p in zip(weights, probs_list))",
    "train_ensemble": "\"\"\"Phase 2: train an ensemble of diverse models, blend OOF, and predict test.\n\nRun on Kaggle (2x T4) after the single-model CV.  Trains ``n_models`` models\n(each on its own grouped fold or the full data), collects OOF probabilities,\noptimises ensemble weights + per-class offsets, and writes a submission.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .config import Config, N_CLASSES\nfrom .data import (\n    InertialNormalizer,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .train import (\n    count_class_labels,\n    grouped_split,\n    make_dataloaders,\n    train_one_fold,\n    macro_f1,\n    softmax,\n)\nfrom .tuning import (\n    optimize_class_offsets,\n    optimize_ensemble_weights,\n    apply_offsets,\n    argmax_preds,\n    combine_models,\n)\nfrom .inference import predict_test, predict_ensemble, write_submission\n\n\ndef collect_oof(cfg: Config, windows, paths, normalizer, n_folds, device,\n                seeds, out_dir) -> tuple[np.ndarray, np.ndarray, list]:\n    \"\"\"Run grouped CV per seed and return pooled OOF probs/targets + models list.\n\n    Returns (all_probs (N,19), all_targets (N,), model_list).\n    OOF entries from different folds/seeds are concatenated.\n    \"\"\"\n    subject_ids = windows[\"sbj_id\"].to_numpy()\n    all_probs, all_targets = [], []\n    # Per-seed OOF alignment: we keep each (fold,seed) OOF separately.\n    n_val = cfg.eval.n_val_subjects\n    for seed in seeds:\n        train_idx, val_idx, val_subjects = grouped_split(\n            subject_ids, n_val_subjects=n_val, seed=seed\n        )\n        res = train_one_fold(cfg, windows, paths, train_idx, val_idx,\n                             fold=f\"seed{seed}\", seed=seed, device=device,\n                             out_dir=out_dir, normalizer=normalizer)\n        all_probs.append(res.oof_logits)\n        all_targets.append(res.oof_targets)\n    return (np.concatenate(all_probs, axis=0),\n            np.concatenate(all_targets, axis=0))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--models\", type=int, default=4, help=\"number of seeds\")\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--max-per-class\", type=int, default=1000)\n    ap.add_argument(\"--class-weighting\", type=str, default=\"sqrt_inverse\")\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--single-sensor-prob\", type=float, default=0.6)\n    ap.add_argument(\"--scale\", type=float, default=1.0)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.train.epochs = args.epochs\n    cfg.train.batch_size = args.batch_size\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    cfg.train.class_weighting = args.class_weighting\n    cfg.model.scale = args.scale\n    cfg.data.single_sensor_prob = args.single_sensor_prob\n    cfg.eval.null_bias = args.null_bias\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    out_dir = cfg.paths.output_dir / \"ensemble\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(\n        windows, cfg.data.max_windows_per_class_per_subject, cfg.data.seed\n    )\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    seeds = [cfg.train.seed + i for i in range(args.models)]\n    all_probs, all_targets = collect_oof(cfg, windows, cfg.paths, normalizer,\n                                         args.models, device, seeds, out_dir)\n    print(f\"pooled OOF probs {all_probs.shape}, targets {all_targets.shape}\")\n\n    # Per-seed OOF splits for ensemble-weight optimisation.\n    # (We optimise weights over the concatenated OOF; for a proper estimate we\n    # treat each seed's OOF as a \"model\".)\n    raw_f1 = macro_f1(all_targets, argmax_preds(all_probs))\n    print(f\"single best OOF macro-F1 = {raw_f1:.4f}\")\n\n    offsets = optimize_class_offsets(all_probs, all_targets, N_CLASSES)\n    tuned = macro_f1(all_targets, argmax_preds(apply_offsets(all_probs, offsets)))\n    print(f\"offset-tuned OOF macro-F1 = {tuned:.4f}\")\n    # Per-class offsets tuned on OOF frequently overfit the held-out subjects\n    # and hurt the real test set. For the submission we rely on the safe\n    # null-bias lever only, and merely report the tuned figure for reference.\n    apply_offs = False\n    print(\"per-class offsets applied to test? False (safe null-bias only)\")\n\n    # Tune the null-class multiplicative bias on the (now single-sensor) OOF.\n    best_bias, best_bias_f1 = 0.0, raw_f1\n    for b in np.linspace(0.0, 2.0, 21):\n        p = all_probs.copy()\n        p[:, 0] *= np.exp(b)\n        p = p / p.sum(1, keepdims=True)\n        f = macro_f1(all_targets, argmax_preds(p))\n        if f > best_bias_f1:\n            best_bias_f1, best_bias = f, b\n    print(f\"null-bias tuned on OOF: {best_bias:.2f} -> macro-F1 {best_bias_f1:.4f}\")\n    null_bias = best_bias\n\n    # --- full-data training + test prediction for each seed ---\n    from .train import train_full\n    models = []\n    for i, seed in enumerate(seeds):\n        cfg.train.seed = seed\n        cfg.data.seed = seed\n        mpath = out_dir / f\"full_seed{seed}.pt\"\n        m = train_full(cfg, device=device, out_path=mpath)\n        models.append(m)\n        print(f\"trained full model seed {seed}\")\n\n    # Ensemble prediction: average probs over models, apply tuned null bias.\n    all_ids, ens_probs = predict_ensemble(models, cfg.paths, device,\n                                          null_bias=0.0, normalizer=normalizer)\n    if null_bias:\n        ens_probs = ens_probs.copy()\n        ens_probs[:, 0] *= np.exp(null_bias)\n        ens_probs = ens_probs / ens_probs.sum(1, keepdims=True)\n    write_submission(all_ids, ens_probs, cfg.paths.sample_submission,\n                     out_dir / \"submission.csv\")\n    print(f\"submission written with {len(all_ids)} rows\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "reference_baseline": "\"\"\"Faithful reproduction of the proven reference baseline (scored 0.61278).\n\nThis ports the \"TS/Emb 3WDC | Temporal Fusion Ensemble\" notebook by Nomannic\ninto our framework so we can (a) validate our data/inference pipeline against a\nknown-good public-leaderboard score, and (b) establish a correct floor to\nimprove on.\n\nReference architecture (from the notebook):\n  * PooledFusionModel  -- per-sensor statistical pooling + video stats\n  * TemporalFusionModel-- Inception inertial CNN + Transformer video + gated fusion\n  * Blend: 0.60 * temporal + 0.40 * pooled, null_bias = exp(0.75) on class 0.\n\nTraining details (reference):\n  * Trains on ALL 4 sensors per window (per-sensor predictions + valid mask).\n  * 150 windows / subject / class; pooled 3 epochs, temporal 4 epochs.\n  * CrossEntropyLoss(label_smoothing=0.05) over valid sensors.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nfrom .config import (\n    INERTIAL_WINDOW,\n    N_CLASSES,\n    N_SENSORS,\n    VIDEO_DIM,\n    VIDEO_WINDOW,\n)\n\n\n# ---------------------------------------------------------------------------\n# Reference building blocks\n# ---------------------------------------------------------------------------\nclass RefInceptionBlock(nn.Module):\n    \"\"\"Reference multi-scale conv block (kernels 5/11/21).\"\"\"\n\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(8, out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)\n        out = out + self.skip(x)\n        return self.dropout(torch.nn.functional.gelu(self.norm(out)))\n\n\n# ---------------------------------------------------------------------------\n# PooledFusionModel\n# ---------------------------------------------------------------------------\nclass PooledFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES, scale: float = 1.0):\n        super().__init__()\n        ih = int(64 * scale)\n        vh = int(192 * scale)\n        ch = int(128 * scale)\n        self.inertial_encoder = nn.Sequential(\n            nn.Linear(16, ih), nn.LayerNorm(ih), nn.GELU(), nn.Dropout(0.15),\n        )\n        self.video_encoder = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM * 2),\n            nn.Linear(VIDEO_DIM * 2, vh),\n            nn.GELU(),\n            nn.Dropout(0.2),\n        )\n        self.sensor_embedding = nn.Embedding(N_SENSORS, 8)\n        self.classifier = nn.Sequential(\n            nn.Linear(ih + vh + 8, ch),\n            nn.GELU(),\n            nn.Dropout(0.2),\n            nn.Linear(ch, n_classes),\n        )\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        # Per-sensor statistical features (16 per sensor).\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 1, 3)                   # (B,4,50,3)\n        mag = torch.linalg.vector_norm(inertial, dim=-1)          # (B,4,50)\n        mean = inertial.mean(dim=2)                                # (B,4,3)\n        std = inertial.std(dim=2)                                  # (B,4,3)\n        amin = inertial.amin(dim=2)\n        amax = inertial.amax(dim=2)\n        feats = torch.cat([\n            mean, std, amin, amax,                                  # 12\n            mag.mean(dim=2, keepdim=True),\n            mag.std(dim=2, keepdim=True),\n            mag.amin(dim=2, keepdim=True),\n            mag.amax(dim=2, keepdim=True),                          # 4\n        ], dim=-1)                                                 # (B,4,16)\n\n        i_feat = self.inertial_encoder(feats)                      # (B,4,64)\n\n        v_mean = video.mean(dim=1)                                 # (B,768)\n        v_std = video.std(dim=1)\n        v_feat = self.video_encoder(torch.cat([v_mean, v_std], dim=-1))  # (B,192)\n        v_feat = v_feat[:, None].expand(B, N_SENSORS, -1)\n\n        sens = self.sensor_embedding.weight[None].expand(B, N_SENSORS, -1)\n\n        cls_in = torch.cat([i_feat, v_feat, sens], dim=-1)         # (B,4,264)\n        logits = self.classifier(cls_in)                           # (B,4,19)\n        return logits\n\n\n# ---------------------------------------------------------------------------\n# TemporalFusionModel\n# ---------------------------------------------------------------------------\nclass RefInertialEncoder(nn.Module):\n    def __init__(self, dropout: float = 0.2, scale: float = 1.0):\n        super().__init__()\n        c = int(128 * scale)\n        h = int(192 * scale)\n        self.blocks = nn.Sequential(\n            RefInceptionBlock(4, c),\n            RefInceptionBlock(c, c),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(2 * c, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,3,50) -> (B,h); adds magnitude channel\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)  # (B,1,50)\n        x = torch.cat([x, mag], dim=1)                          # (B,4,50)\n        h = self.blocks(x)          # (B,c,50)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)  # (B,2c)\n        return self.head(pooled)\n\n\nclass RefVideoEncoder(nn.Module):\n    def __init__(self, dropout: float = 0.2, scale: float = 1.0):\n        super().__init__()\n        h = int(192 * scale)\n        ff = int(384 * scale)\n        heads = 4\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM), nn.Linear(VIDEO_DIM, h), nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, h)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            h, heads, dim_feedforward=ff, dropout=0.15, activation=\"gelu\",\n            batch_first=True, norm_first=True,\n        )\n        self.temporal = nn.TransformerEncoder(layer, 1,\n                                              enable_nested_tensor=False)\n        self.attention = nn.Sequential(nn.Linear(h, int(64 * scale)), nn.Tanh(),\n                                       nn.Linear(int(64 * scale), 1))\n        self.head = nn.Sequential(\n            nn.Linear(2 * h, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,15,768)\n        h = self.projection(x) + self.position\n        h = self.temporal(h)\n        scores = self.attention(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\nclass TemporalFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES, scale: float = 1.0):\n        super().__init__()\n        h = int(192 * scale)\n        self.inertial_encoder = RefInertialEncoder(scale=scale)\n        self.video_encoder = RefVideoEncoder(scale=scale)\n        self.sensor_embedding = nn.Embedding(N_SENSORS, int(16 * scale))\n        se = int(16 * scale)\n        self.gate = nn.Sequential(nn.Linear(2 * h + se, h), nn.Sigmoid())\n        self.classifier = nn.Sequential(\n            nn.Linear(2 * h + se, h),\n            nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.3),\n            nn.Linear(h, n_classes),\n        )\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        h = self.inertial_encoder.head[0].out_features\n        # Per-sensor inertial (B*4, 3, 50) -> (B,4,h)\n        inert = inertial.permute(0, 2, 3, 1).reshape(B * N_SENSORS, 3,\n                                                     INERTIAL_WINDOW)\n        i_feat = self.inertial_encoder(inert).reshape(B, N_SENSORS, h)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        sens = self.sensor_embedding.weight[None].expand(B, N_SENSORS, -1)\n\n        gate_in = torch.cat([i_feat, v_feat, sens], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * i_feat + (1 - gate) * v_feat\n        features = torch.cat([fused, i_feat * v_feat, sens], dim=-1)\n        logits = self.classifier(features)   # (B,4,19)\n        return logits\n\n\n# ---------------------------------------------------------------------------\n# Train / eval helpers matching the reference\n# ---------------------------------------------------------------------------\ndef blend_probabilities(pooled_logits, temporal_logits, temporal_weight=0.60,\n                        null_bias=0.75):\n    \"\"\"Blend per-sensor logits -> (N,19) probabilities with null bias.\"\"\"\n    p_pooled = torch.softmax(pooled_logits, dim=-1)\n    p_temporal = torch.softmax(temporal_logits, dim=-1)\n    p = temporal_weight * p_temporal + (1 - temporal_weight) * p_pooled\n    # (N,4,19) -> average over sensors -> (N,19)\n    p = p.mean(dim=1)\n    if null_bias:\n        p[:, 0] *= math.exp(null_bias)\n    return p / p.sum(dim=1, keepdim=True)",
    "run_reference": "\"\"\"Run the proven reference baseline (reproduce ~0.61278 on the leaderboard).\n\nTrains the Pooled + Temporal fusion models exactly as the reference notebook,\nthen blends them and writes a submission.  Use this to validate that our data /\ninference pipeline reproduces the known-good score.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .reference_baseline import PooledFusionModel, TemporalFusionModel\n\n\nclass _NullVideo:\n    \"\"\"Wrap a dataset and return zeroed video (inertial-only mode).\"\"\"\n\n    def __init__(self, inner):\n        self.inner = inner\n\n    def __len__(self):\n        return len(self.inner)\n\n    def __getitem__(self, idx):\n        b = dict(self.inner[idx])\n        b[\"video\"] = np.zeros_like(b[\"video\"])\n        return b\n\n\ndef _train_model(model, dl, epochs, lr, wd, device, use_amp, label_smooth=0.05,\n                 warmup_epochs=1, grad_norm=5.0, tag=\"\", seed=0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer,\n                  inertial_only=False):\n    from .inference import WearTestDataset\n    from .data import build_test_dataset\n    from . import inference as inf\n\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device)\n            vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            if inertial_only:\n                vid = torch.zeros_like(vid)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)   # (B,4,19)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]       # (B,19)\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs-pooled\", type=int, default=3)\n    ap.add_argument(\"--epochs-temporal\", type=int, default=4)\n    ap.add_argument(\"--temporal-weight\", type=float, default=0.60)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--normalize\", type=int, default=0,\n                    help=\"0=none, 1=global z-score, 2=per-window z-score\")\n    ap.add_argument(\"--scale\", type=float, default=1.0,\n                    help=\"model width multiplier (1.0 = original)\")\n    ap.add_argument(\"--seeds\", type=int, default=1,\n                    help=\"train this many seeds and average their blended probs\")\n    ap.add_argument(\"--inertial-only\", type=int, default=0,\n                    help=\"1 to zero the video modality (inertial-only model)\")\n    ap.add_argument(\"--smooth-window\", type=int, default=0,\n                    help=\"temporal majority-vote smoothing half-window k (0=off)\")\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"reference\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = None\n    if args.normalize == 1:\n        normalizer = InertialNormalizer.fit(cfg.paths)\n        print(\"using global inertial normalisation\")\n    elif args.normalize == 2:\n        from .data import PerWindowNormalizer\n        normalizer = PerWindowNormalizer()\n        print(\"using per-window inertial normalisation\")\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    if args.inertial_only:\n        ds = _NullVideo(ds)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc_blend = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        pooled = PooledFusionModel(scale=args.scale).to(device)\n        temporal = TemporalFusionModel(scale=args.scale).to(device)\n        print(f\"seed {seed}: Pooled={sum(p.numel() for p in pooled.parameters())/1e6:.3f}M \"\n              f\"Temporal={sum(p.numel() for p in temporal.parameters())/1e6:.3f}M\")\n\n        _train_model(pooled, dl, args.epochs_pooled, lr=2e-3, wd=1e-4,\n                     device=device, use_amp=use_amp, tag=f\"pooled-{seed}\", seed=seed)\n        _train_model(temporal, dl, args.epochs_temporal, lr=8e-4, wd=1e-3,\n                     device=device, use_amp=use_amp, tag=f\"temporal-{seed}\", seed=seed)\n\n        pooled_ids, pooled_probs = _predict_test(pooled, cfg.paths, device,\n                                                 use_amp, normalizer,\n                                                 args.inertial_only)\n        temp_ids, temp_probs = _predict_test(temporal, cfg.paths, device,\n                                             use_amp, normalizer,\n                                             args.inertial_only)\n        assert (pooled_ids == temp_ids).all()\n        blend = args.temporal_weight * temp_probs + (1 - args.temporal_weight) * pooled_probs\n        ref_ids = pooled_ids\n        acc_blend = blend if acc_blend is None else acc_blend + blend\n\n    blended = acc_blend / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    # Optional temporal majority-vote smoothing (per subject, in id order).\n    if args.smooth_window > 0:\n        import pandas as pd\n        from .temporal_smooth import smooth_majority_vote\n        meta = pd.read_csv(cfg.paths.test_meta)\n        # probs are aligned to meta rows sorted by id == meta row order (ids 0..N-1)\n        smoothed = smooth_majority_vote(blended, meta, k=args.smooth_window)\n        preds = smoothed\n        print(f\"temporal smoothing applied (k={args.smooth_window})\")\n    else:\n        preds = blended.argmax(axis=1)\n\n    # Write to the standard Kaggle submission location so it's easy to submit.\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, preds, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose": "\"\"\"Diagnostic: which modality carries the signal?\n\nRuns a fast grouped CV (held-out subjects) with the reference architecture in\nthree configurations:\n  * inertial-only  (video replaced by zeros)\n  * video-only     (inertial replaced by zeros)\n  * fusion         (both)\n\nReports macro-F1 per config.  This tells us whether video is adding signal or\nif there's an alignment problem, guiding where to invest.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import build_dataset_windows, subsample_windows\nfrom .train import grouped_split, make_dataloaders, macro_f1, softmax\nfrom .reference_baseline import TemporalFusionModel\n\n\nclass NullVideoDataset:\n    \"\"\"Wraps WearTrainDataset and returns zero video.\"\"\"\n    def __init__(self, ds):\n        self.ds = ds\n\n    def __len__(self):\n        return len(self.ds)\n\n    def __getitem__(self, idx):\n        b = self.ds[idx]\n        b = dict(b)\n        b[\"video\"] = np.zeros_like(b[\"video\"])\n        return b\n\n\ndef _run(cfg, windows, train_idx, val_idx, normalizer, device, use_amp,\n         null_video, null_inertial, epochs=3, tag=\"\"):\n    from .data import WearTrainDataset\n\n    def make(idx, nv, ni):\n        ws = windows.iloc[idx].reset_index(drop=True)\n        ds = WearTrainDataset(ws, cfg.paths, normalizer=normalizer)\n        if nv:\n            ds = NullVideoDataset(ds)\n        if ni:\n            # zero inertial\n            class NID:\n                def __init__(s, inner): s.inner = inner\n                def __len__(s): return len(s.inner)\n                def __getitem__(s, i):\n                    b = dict(s.inner[i]); b[\"inertial\"] = np.zeros_like(b[\"inertial\"]); return b\n            ds = NID(ds)\n        return ds\n\n    tr_ds = make(train_idx, null_video, null_inertial)\n    va_ds = make(val_idx, null_video, null_inertial)\n    tr_dl = DataLoader(tr_ds, batch_size=cfg.train.batch_size, shuffle=True,\n                       num_workers=0, pin_memory=True)\n    va_dl = DataLoader(va_ds, batch_size=cfg.train.batch_size * 2, shuffle=False,\n                       num_workers=0, pin_memory=True)\n\n    torch.manual_seed(0)\n    model = TemporalFusionModel().to(device)\n    crit = nn.CrossEntropyLoss(label_smoothing=cfg.train.label_smoothing)\n    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        for b in tr_dl:\n            inert = b[\"inertial\"].to(device); vid = b[\"video\"].to(device)\n            tgt = b[\"target\"].to(device); valid = b[\"valid\"].to(device)\n            opt.zero_grad()\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = crit(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()\n        # simple LR step (diagnostic)\n        for g in opt.param_groups:\n            g[\"lr\"] *= 0.7\n\n    model.eval()\n    yt, yp = [], []\n    with torch.no_grad():\n        for b in va_dl:\n            inert = b[\"inertial\"].to(device); vid = b[\"video\"].to(device)\n            logits = model(inert, vid).float()\n            pr = softmax(logits.cpu().numpy(), -1).mean(axis=1).argmax(1)\n            yt.append(b[\"target\"].numpy()); yp.append(pr)\n    f1 = macro_f1(np.concatenate(yt), np.concatenate(yp), n_classes=N_CLASSES)\n    print(f\"[{tag}] macro-F1 = {f1:.4f}\")\n    return f1\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=3)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n\n    from .data import InertialNormalizer\n    normalizer = None  # no normalization (match reference)\n\n    subj = windows[\"sbj_id\"].to_numpy()\n    tr, va, vs = grouped_split(subj, n_val_subjects=4, seed=cfg.data.seed)\n    print(f\"val subjects {sorted(vs)} | train {len(tr)} val {len(va)}\")\n\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=True, null_inertial=False, epochs=args.epochs, tag=\"inertial-only\")\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=False, null_inertial=True, epochs=args.epochs, tag=\"video-only\")\n    _run(cfg, windows, tr, va, normalizer, device, use_amp,\n         null_video=False, null_inertial=False, epochs=args.epochs, tag=\"fusion\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "temporal_smooth": "\"\"\"Temporal majority-vote smoothing of test predictions.\n\nThe test windows come from continuous recordings of 4 unseen subjects.  Sorting\neach subject's windows by ``id`` recovers its 1-second time axis; adjacent\nwindows are the same activity ~90%+ of the time.  We reassign each window to the\nmajority class of its +/-k neighborhood (per subject, in time order), correcting\nthe scattered single-window misclassifications that hurt macro-F1 most.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef temporal_order(test_meta: pd.DataFrame):\n    \"\"\"Return an array of ``test_meta`` row indices in per-subject time order.\n\n    Sorts by (sbj_id, id).  Returns (sorted_ids, subject_boundaries) where\n    ``sorted_ids`` are meta row positions in temporal order and\n    ``subject_boundaries`` are the end indices of each subject block.\n    \"\"\"\n    meta = test_meta.reset_index(drop=True)\n    order = meta.sort_values([\"sbj_id\", \"id\"]).index.to_numpy()\n    subj = meta.loc[order, \"sbj_id\"].to_numpy()\n    boundaries = np.flatnonzero(subj[1:] != subj[:-1]) + 1\n    return order, boundaries\n\n\ndef smooth_majority_vote(probs: np.ndarray, meta: pd.DataFrame,\n                         k: int = 2) -> np.ndarray:\n    \"\"\"Majority-vote over +/-k neighbors per subject (in time order).\n\n    ``probs``: (N, n_classes) in the same order as ``meta`` rows.\n    Returns smoothed class ids (N,).\n    \"\"\"\n    preds = probs.argmax(axis=1)\n    order, boundaries = temporal_order(meta)\n\n    # Place predictions in temporal order, per subject block.\n    blocks = np.split(order, boundaries)\n    out = preds.copy()\n    for blk in blocks:\n        seq = preds[blk]                       # temporal class sequence (L,)\n        L = len(seq)\n        if L <= 1:\n            continue\n        smoothed = np.empty(L, dtype=seq.dtype)\n        for i in range(L):\n            lo, hi = max(0, i - k), min(L, i + k + 1)\n            # majority (ties broken by center / lowest class)\n            counts = np.bincount(seq[lo:hi], minlength=seq.max() + 1)\n            smoothed[i] = counts.argmax()\n        out[blk] = smoothed\n    return out\n\n\ndef smooth_by_probability_average(probs: np.ndarray, meta: pd.DataFrame,\n                                  k: int = 2) -> np.ndarray:\n    \"\"\"Average probabilities over +/-k neighbors per subject, then argmax.\"\"\"\n    order, boundaries = temporal_order(meta)\n    blocks = np.split(order, boundaries)\n    out = np.empty(probs.shape[0], dtype=np.int64)\n    for blk in blocks:\n        seq = probs[blk]                       # (L, C)\n        L = len(seq)\n        for i in range(L):\n            lo, hi = max(0, i - k), min(L, i + k + 1)\n            out[blk[i]] = seq[lo:hi].mean(axis=0).argmax()\n    return out",
    "strong_model": "\"\"\"Stronger fusion model for the WEAR challenge.\n\nDesign rationale (based on measured results):\n  * Video carries transferable signal on the real test set (fusion > inertial-only\n    on the leaderboard), so we invest heavily in a deep VideoMAE encoder.\n  * The inertial branch stays lean (scaling inertial capacity overfit the test),\n    using the proven per-sensor multi-scale Inception encoder + normalisation.\n  * Fusion is a learned gated blend + interaction, with a larger classifier.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\n# ---------------------------------------------------------------------------\n# Inception building block (proven, kept lean)\n# ---------------------------------------------------------------------------\nclass SInceptionBlock(nn.Module):\n    def __init__(self, cin: int, out: int, dropout: float = 0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(\n            nn.MaxPool1d(3, stride=1, padding=1),\n            nn.Conv1d(cin, bottleneck, 1, bias=False),\n        )\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False)\n                     if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        branches = [br(b) for br in self.branches]\n        branches.append(self.pool(x))\n        out = torch.cat(branches, dim=1)\n        out = out + self.skip(x)\n        return self.dropout(F.gelu(self.norm(out)))\n\n\nclass SInertialEncoder(nn.Module):\n    \"\"\"Per-sensor lean Inception encoder: (B,3,50) -> (B,h). Adds magnitude.\"\"\"\n\n    def __init__(self, hidden: int = 192, blocks: int = 3,\n                 channels: int = 128, dropout: float = 0.2):\n        super().__init__()\n        layers = [SInceptionBlock(4, channels)]\n        for _ in range(blocks - 1):\n            layers.append(SInceptionBlock(channels, channels))\n        self.blocks = nn.Sequential(*layers)\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):  # (B,3,50)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)          # (B,4,50)\n        h = self.blocks(x)                      # (B,C,50)\n        pooled = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)\n        return self.head(pooled)\n\n\n# ---------------------------------------------------------------------------\n# Strong VideoMAE encoder (deep transformer over the 15x768 features)\n# ---------------------------------------------------------------------------\nclass SVideoEncoder(nn.Module):\n    def __init__(self, hidden: int = 256, layers: int = 4, heads: int = 8,\n                 ff_mult: int = 4, dropout: float = 0.15):\n        super().__init__()\n        self.projection = nn.Sequential(\n            nn.LayerNorm(VIDEO_DIM),\n            nn.Linear(VIDEO_DIM, hidden),\n            nn.GELU(),\n        )\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.position = nn.Parameter(pos)\n        layer = nn.TransformerEncoderLayer(\n            hidden, heads, dim_feedforward=hidden * ff_mult, dropout=dropout,\n            activation=\"gelu\", batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(\n            layer, layers, enable_nested_tensor=False\n        )\n        self.attn = nn.Sequential(\n            nn.Linear(hidden, hidden // 2), nn.Tanh(),\n            nn.Linear(hidden // 2, 1),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden),\n            nn.LayerNorm(hidden),\n            nn.GELU(),\n            nn.Dropout(0.2),\n        )\n\n    def forward(self, x):  # (B,15,768)\n        h = self.projection(x) + self.position\n        h = self.transformer(h)                # (B,15,H)\n        scores = self.attn(h).squeeze(-1)\n        weights = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * weights).sum(dim=1)\n        mean = h.mean(dim=1)\n        return self.head(torch.cat([attended, mean], dim=1))\n\n\n# ---------------------------------------------------------------------------\n# Strong fusion model\n# ---------------------------------------------------------------------------\nclass StrongFusionModel(nn.Module):\n    def __init__(self, n_classes: int = N_CLASSES,\n                 video_hidden: int = 256, video_layers: int = 4,\n                 video_heads: int = 8, inertial_hidden: int = 192,\n                 inertial_blocks: int = 3, inertial_channels: int = 128,\n                 cls_hidden: int = 384, sensor_dim: int = 16,\n                 dropout: float = 0.2, modality_dropout: float = 0.1):\n        super().__init__()\n        self.inertial_encoder = SInertialEncoder(\n            inertial_hidden, inertial_blocks, inertial_channels, dropout)\n        self.video_encoder = SVideoEncoder(\n            video_hidden, video_layers, video_heads, dropout=dropout)\n        self.sensor_embed = nn.Embedding(N_SENSORS, sensor_dim)\n\n        # Project both modalities to a common fusion dim for gating.\n        fusion_dim = inertial_hidden\n        self.i_proj = nn.Linear(inertial_hidden, fusion_dim)\n        self.v_proj = nn.Linear(video_hidden, fusion_dim)\n        self.sensor_proj = nn.Linear(sensor_dim, sensor_dim)\n\n        gate_in = fusion_dim * 2 + sensor_dim\n        self.gate = nn.Sequential(nn.Linear(gate_in, fusion_dim),\n                                  nn.Sigmoid())\n        cls_in = fusion_dim * 2 + sensor_dim\n        self.classifier = nn.Sequential(\n            nn.Linear(cls_in, cls_hidden),\n            nn.LayerNorm(cls_hidden),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(cls_hidden, n_classes),\n        )\n        self.modality_dropout = modality_dropout\n        self.sensor_dropout = 0.2\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        inertial = inertial.permute(0, 2, 3, 1)          # (B,4,50,3)\n\n        # Sensor dropout (robust to single-sensor test).\n        if self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inertial.device) >= self.sensor_dropout\n            inertial = inertial * keep[:, :, None, None].float()\n\n        i_feat = self.inertial_encoder(\n            inertial.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        ).reshape(B, N_SENSORS, -1)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        pos = self.sensor_embed.weight[None, :, :].expand(B, N_SENSORS, -1)\n\n        # Project to common fusion dim.\n        i_feat = self.i_proj(i_feat)\n        v_feat = self.v_proj(v_feat)\n        pos = self.sensor_proj(pos)\n\n        if self.training:\n            i_feat, v_feat = self._modality_dropout(\n                i_feat, v_feat, self.modality_dropout)\n\n        gate_in = torch.cat([i_feat, v_feat, pos], dim=-1)\n        gate = self.gate(gate_in)\n        fused = gate * i_feat + (1 - gate) * v_feat\n        interaction = i_feat * v_feat\n        features = torch.cat([fused, interaction, pos], dim=-1)\n        return self.classifier(features)                   # (B,4,n_classes)\n\n    @staticmethod\n    def _modality_dropout(i_feat, v_feat, p):\n        if p <= 0:\n            return i_feat, v_feat\n        scale = 1.0 / (1.0 - p)\n        drop_i = (torch.rand(i_feat.shape[:2], device=i_feat.device) < p)[..., None]\n        drop_v = (torch.rand(v_feat.shape[:2], device=v_feat.device) < p)[..., None]\n        i_feat = torch.where(drop_i, torch.zeros_like(i_feat), i_feat * scale)\n        v_feat = torch.where(drop_v, torch.zeros_like(v_feat), v_feat * scale)\n        return i_feat, v_feat",
    "run_strong": "\"\"\"Train the strong fusion model and produce a test submission.\n\nUses our verified data pipeline (segment windowing, per-sensor NaN handling)\nwith global z-score normalisation.  The video branch is deep (the transferable\nmodality on the real test set); the inertial branch stays lean (scaling it\noverfit the test).  Trains ``seeds`` models and averages their probabilities.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .strong_model import StrongFusionModel\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag=\"\", seed=0,\n           label_smooth=0.05, warmup_epochs=1, grad_norm=5.0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer):\n    from . import inference as inf\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device)\n            vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--lr\", type=float, default=6e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=4)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-layers\", type=int, default=4)\n    ap.add_argument(\"--video-hidden\", type=int, default=256)\n    ap.add_argument(\"--video-heads\", type=int, default=8)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"strong\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    print(\"global normalisation ON\")\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        model = StrongFusionModel(\n            video_hidden=args.video_hidden, video_layers=args.video_layers,\n            video_heads=args.video_heads).to(device)\n        nparam = sum(p.numel() for p in model.parameters())\n        print(f\"seed {seed}: params={nparam/1e6:.2f}M\")\n        _train(model, dl, args.epochs, args.lr, 1e-3, device, use_amp,\n               tag=f\"strong-{seed}\", seed=seed)\n        ids, probs = _predict_test(model, cfg.paths, device, use_amp, normalizer)\n        ref_ids = ids\n        acc = probs if acc is None else acc + probs\n\n    blended = acc / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, blended, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "features": "\"\"\"Fast, vectorised feature engineering for the WEAR challenge.\n\nBased on the prior WEAR winners (FAME: frequency-domain features; 1st winner:\nrich features + gradient boosting).  All feature extraction is vectorised over\nthe batch so it runs in seconds, not hours.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nN_ACC_FEATS = 19  # per axis/magnitude: 13 time + 6 frequency\n\n\ndef _freq_features_vect(x: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"x: (..., N) zero-mean signals -> (..., 8) freq features. Vectorised.\"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    x = x - x.mean(axis=-1, keepdims=True)\n    n = x.shape[-1]\n    mag = np.abs(np.fft.rfft(x, axis=-1))          # (..., n//2+1)\n    freqs = np.fft.rfftfreq(n, 1.0 / fs)\n    power = mag ** 2\n    total = power.sum(axis=-1, keepdims=True)\n    total = np.where(total < 1e-12, 1.0, total)\n    p = power / total\n\n    dom_freq = freqs[power.argmax(axis=-1)]\n    centroid = (freqs[None, :] * power).sum(axis=-1) / total[..., 0]\n    entropy = -np.sum(p * np.log(p + 1e-12), axis=-1)\n\n    low = power[..., freqs < 2].sum(axis=-1) / total[..., 0]\n    mid = power[..., (freqs >= 2) & (freqs < 8)].sum(axis=-1) / total[..., 0]\n    high = power[..., freqs >= 8].sum(axis=-1) / total[..., 0]\n\n    # spectral rolloff (95% energy) and spectral flux\n    cum = np.cumsum(p, axis=-1)\n    rolloff_idx = np.argmax(cum >= 0.95, axis=-1)\n    rolloff = freqs[rolloff_idx]\n    # spectral flux (first-order difference of magnitude)\n    flux = np.mean(np.abs(np.diff(p, axis=-1)), axis=-1)\n\n    return np.stack([dom_freq, centroid, entropy, low, mid, high,\n                     rolloff, flux], axis=-1)\n\n\ndef _time_features_vect(x: np.ndarray) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 13) time features. Vectorised, NaN-safe.\"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    n = x.shape[-1]\n    m = x.mean(axis=-1)\n    # stable std\n    s = np.sqrt(np.mean((x - m[..., None]) ** 2, axis=-1) + 1e-12)\n    rms = np.sqrt(np.mean(x ** 2, axis=-1) + 1e-12)\n    xmin = x.min(axis=-1)\n    xmax = x.max(axis=-1)\n    ptp = xmax - xmin\n    median = np.median(x, axis=-1)\n    # skew / kurtosis (central moments, NaN-safe)\n    zm = x - m[..., None]\n    z2 = np.mean(zm ** 2, axis=-1) + 1e-12\n    skew = np.mean(zm ** 3, axis=-1) / (z2 ** 1.5)\n    kurt = np.mean(zm ** 4, axis=-1) / (z2 ** 2) - 3.0\n    # zero crossings & mean crossings (per sample)\n    zc = (np.sign(zm[..., 1:]) * np.sign(zm[..., :-1]) < 0).sum(axis=-1) / max(1, n - 1)\n    mcr = (np.sign(x[..., 1:] - m[..., None]) * np.sign(x[..., :-1] - m[..., None]) < 0).sum(axis=-1) / max(1, n - 1)\n    # autocorr lag-1\n    xm = x - m[..., None]\n    num = np.mean(xm[..., :-1] * xm[..., 1:], axis=-1)\n    ar1 = num / (np.mean(xm[..., :-1] ** 2, axis=-1) + 1e-12)\n    # jerk proxy (std of first difference)\n    jerk = np.std(np.diff(x, axis=-1), axis=-1)\n    td = np.stack([m, s, rms, xmin, xmax, ptp, median, skew, kurt,\n                   zc, mcr, ar1, jerk], axis=-1)\n    return np.nan_to_num(td, nan=0.0, posinf=0.0, neginf=0.0)\n\n\ndef _axis_features_vect(x: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 21) features. Vectorised (13 time + 8 freq).\"\"\"\n    td = _time_features_vect(x)\n    fr = _freq_features_vect(x, fs)\n    return np.concatenate([td, fr], axis=-1)\n\n\ndef _subwindow_features_vect(x: np.ndarray) -> np.ndarray:\n    \"\"\"x: (..., N) -> (..., 4) sub-window trend features.\n\n    Splits the window in half and compares summary stats between the two\n    halves to capture temporal structure/drift.\n    \"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    n = x.shape[-1]\n    half = max(1, n // 2)\n    x0 = x[..., :half]\n    x1 = x[..., half:2 * half]\n    m0 = x0.mean(axis=-1)\n    m1 = x1.mean(axis=-1)\n    s0 = x0.std(axis=-1)\n    s1 = x1.std(axis=-1)\n    # energy ratio first/second half, and trend of mean/std\n    e0 = np.mean(x0 ** 2, axis=-1)\n    e1 = np.mean(x1 ** 2, axis=-1)\n    trend_mean = m1 - m0\n    trend_std = s1 - s0\n    energy_ratio = e1 / (e0 + 1e-9)\n    # slope of linear fit (normalized)\n    t = np.linspace(0, 1, n)\n    denom = np.sum((t - t.mean()) ** 2)\n    slope = np.sum((x - x.mean(axis=-1, keepdims=True)) * (t - t.mean()), axis=-1) / denom\n    return np.stack([trend_mean, trend_std, energy_ratio, slope], axis=-1)\n\n\ndef extract_sensor_correlations(windows: np.ndarray) -> np.ndarray:\n    \"\"\"windows: (N, 50, 4, 3) -> (N, 12) pairwise sensor magnitude correlations.\n\n    For each pair of sensors, correlation of their magnitude time-series.\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    N, T, S, A = w.shape\n    mag = np.linalg.norm(w, axis=-1)              # (N,T,S)\n    out = np.zeros((N, S * (S - 1) // 2), dtype=np.float64)\n    k = 0\n    for i in range(S):\n        for j in range(i + 1, S):\n            a = mag[:, :, i] - mag[:, :, i].mean(axis=-1, keepdims=True)\n            b = mag[:, :, j] - mag[:, :, j].mean(axis=-1, keepdims=True)\n            denom = np.sqrt((a ** 2).sum(-1) * (b ** 2).sum(-1)) + 1e-9\n            out[:, k] = (a * b).sum(-1) / denom\n            k += 1\n    return out\n\n\ndef fit_inertial_pca(F_base: np.ndarray, n_components: int = 32,\n                     seed: int = 42):\n    \"\"\"Fit PCA on base inertial feature matrix for feature augmentation.\"\"\"\n    Xc = F_base - F_base.mean(axis=0)\n    from sklearn.decomposition import TruncatedSVD\n    svd = TruncatedSVD(n_components=min(n_components, Xc.shape[1] - 1),\n                       random_state=seed)\n    svd.fit(Xc)\n    return Xc.mean(axis=0), svd.components_.T\n\n\ndef apply_inertial_pca(F_base: np.ndarray, center, components) -> np.ndarray:\n    return (F_base - center) @ components\n\n\ndef _lowpass(x, alpha=0.1):\n    \"\"\"Exponential moving average along last axis (approx gravity).\"\"\"\n    out = np.empty_like(x)\n    acc = x[..., 0].copy()\n    out[..., 0] = acc\n    for t in range(1, x.shape[-1]):\n        acc = alpha * x[..., t] + (1 - alpha) * acc\n        out[..., t] = acc\n    return out\n\n\ndef _gravity_body_decompose(a):\n    \"\"\"a: (..., T, 3) -> a_parallel (...,T,1), a_perp (...,T,1), gravity_norm (...,T,1).\"\"\"\n    a = np.asarray(a, dtype=np.float64)\n    g = _lowpass(a, alpha=0.15)                      # gravity estimate\n    gn = np.linalg.norm(g, axis=-1, keepdims=True) + 1e-9\n    ghat = g / gn\n    a_par = (a * ghat).sum(axis=-1, keepdims=True)   # projection along gravity\n    a_perp = np.linalg.norm(a - a_par * ghat, axis=-1, keepdims=True)\n    return a_par, a_perp, gn\n\n\ndef _rotation_invariant_stats(a):\n    \"\"\"a: (..., T, 3) -> rotation-invariant per-sample stats (..., T, 6).\"\"\"\n    x = a[..., 0]; y = a[..., 1]; z = a[..., 2]\n    return np.stack([x * x + y * y, y * y + z * z, x * x + z * z,\n                     x * y, x * z, y * z], axis=-1)\n\n\ndef extract_orientation_invariant(windows: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"Orientation-invariant features per sensor (vectorised).\n\n    windows: (N,T,3) test or (N,T,4,3) train. Returns (N, F) of added features.\n    Includes: magnitude (already partly in base, expanded here), gravity/body-frame\n    decomposition, rotation-invariant products, covariance eigenvalues.\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    if w.ndim == 4:                       # (N,T,4,3)\n        N, T, S, A = w.shape\n        cols = []\n        for s in range(S):\n            cols.append(_orientation_invariant_one(w[:, :, s], fs))\n        return np.concatenate(cols, axis=-1)\n    else:                                 # (N,T,3)\n        return _orientation_invariant_one(w, fs)\n\n\ndef _orientation_invariant_one(a, fs=50.0):\n    \"\"\"a: (N,T,3) -> (N, F) orientation-invariant features for one sensor.\"\"\"\n    mag = np.linalg.norm(a, axis=-1)                       # (N,T)\n    # magnitude derivatives\n    dm = np.gradient(mag, axis=-1)\n    d2m = np.gradient(dm, axis=-1)\n    a_par, a_perp, gn = _gravity_body_decompose(a)         # each (N,T,1)\n    ri = _rotation_invariant_stats(a)                      # (N,T,6)\n    # covariance eigenvalues (per window) of raw 3 axes\n    N = a.shape[0]\n    eig = np.zeros((N, 3), dtype=np.float64)\n    for i in range(N):\n        c = np.cov(a[i].T)                                  # 3x3\n        ev = np.linalg.eigvalsh(c)\n        eig[i] = ev[::-1]\n    # stack per-sample feature streams, then time-features each\n    streams = np.concatenate([\n        mag[:, :, None], dm[:, :, None], d2m[:, :, None],\n        a_par, a_perp, gn, ri], axis=-1)                    # (N,T,12)\n    feats = []\n    for k in range(streams.shape[-1]):\n        feats.append(_time_features_vect(streams[:, :, k]))\n    feats.append(eig)                                        # 3 eigenvalues\n    return np.concatenate(feats, axis=-1)\n\n\ndef extract_acc_features_vect(windows: np.ndarray, fs: float = 50.0) -> np.ndarray:\n    \"\"\"Vectorised feature extraction.\n\n    windows: (N, T, 3) single-sensor (test) or (N, T, 4, 3) all-sensors (train).\n    Returns (N, F).  For (N,T,4,3) we concatenate per-sensor features (each\n    sensor = 3 axes + magnitude).\n    \"\"\"\n    w = np.asarray(windows, dtype=np.float64)\n    if w.ndim == 4:  # (N,T,4,3)\n        N, T, S, A = w.shape\n        cols = []\n        for s in range(S):\n            for a in range(A):\n                cols.append(_axis_features_vect(w[:, :, s, a], fs))\n                cols.append(_subwindow_features_vect(w[:, :, s, a]))\n            mag = np.linalg.norm(w[:, :, s], axis=-1)\n            cols.append(_axis_features_vect(mag, fs))\n            cols.append(_subwindow_features_vect(mag))\n        return np.concatenate(cols, axis=-1)\n    else:  # (N,T,3)\n        N, T, A = w.shape\n        cols = []\n        for a in range(A):\n            cols.append(_axis_features_vect(w[:, :, a], fs))\n            cols.append(_subwindow_features_vect(w[:, :, a]))\n        mag = np.linalg.norm(w, axis=-1)\n        cols.append(_axis_features_vect(mag, fs))\n        cols.append(_subwindow_features_vect(mag))\n        return np.concatenate(cols, axis=-1)\n\n\ndef extract_video_features_vect(videos: np.ndarray,\n                                n_components: int = 64) -> np.ndarray:\n    \"\"\"videos: (N, 15, 768) -> (N, 2*n_components) pooled features.\"\"\"\n    v = np.asarray(videos, dtype=np.float64)\n    mean = v.mean(axis=1)                # (N,768)\n    std = v.std(axis=1)\n    feats = np.concatenate([mean, std], axis=-1)  # (N,1536)\n    rng = np.random.RandomState(0)\n    proj = rng.randn(1536, n_components) / np.sqrt(1536)\n    return feats @ proj",
    "feature_data": "\"\"\"Build train/test feature matrices for the boosting model.\n\nTrain windows have all 4 sensors; test windows have a single sensor location.\nTo keep feature dimensions consistent, we treat each sensor as a \"view\": every\nwindow yields one feature vector per sensor, tagged with a one-hot sensor\nlocation.  At test we use the present sensor's vector (matching its location).\n\nFeature extraction is fully vectorised (see features.py).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import N_SENSORS, Paths\nfrom .data import (\n    build_dataset_windows,\n    load_train_recording,\n    subsample_windows,\n    train_stems,\n    INERTIAL_WINDOW,\n    VIDEO_WINDOW,\n    VIDEO_FRAME_OFFSET,\n)\nfrom .features import (\n    extract_acc_features_vect,\n    extract_orientation_invariant,\n    extract_video_features_vect,\n)\n\n# Base per-sensor features (3 axes + magnitude, 25 each = 100) + orientation-invariant (159).\n_BASE_PER_SENSOR = 4 * (21 + 4)      # 100\n_ORIENT_PER_SENSOR = 159\nPER_SENSOR_FEATS = _BASE_PER_SENSOR + _ORIENT_PER_SENSOR  # 259\nSENSOR_FEAT_DIM = PER_SENSOR_FEATS + N_SENSORS  # + one-hot location\n\n\ndef _build_train_features(paths: Paths, stride: int, max_per_class, seed,\n                          video_proj_dim, video_transform=None,\n                          use_orientation=False):\n    \"\"\"Recording-by-recording, vectorised feature build (fast, low memory).\"\"\"\n    from .data import build_dataset_windows, load_train_recording, subsample_windows\n    windows = build_dataset_windows(paths, stride)\n    windows = subsample_windows(windows, max_per_class, seed)\n\n    all_F, all_V, all_y = [], [], []\n    for rec in windows[\"rec\"].unique():\n        r = load_train_recording(paths, rec)   # loads inertial + video (mmap)\n        sub = windows[windows[\"rec\"] == rec]\n        starts = sub[\"inertial_start\"].to_numpy()\n        vstarts = sub[\"video_start\"].to_numpy()\n        ys = sub[\"target\"].to_numpy()\n\n        # Vectorised window gather for this recording.\n        idx = starts[:, None] + np.arange(INERTIAL_WINDOW)[None, :]   # (M,50)\n        inerts = r.inertial[idx].astype(np.float64)                    # (M,50,4,3)\n        inerts = np.nan_to_num(inerts)\n        vidx = vstarts[:, None] + np.arange(VIDEO_WINDOW)[None, :]     # (M,15)\n        vids = np.asarray(r.video[vidx], dtype=np.float64)             # (M,15,768)\n        vids = np.nan_to_num(vids)\n\n        base = extract_acc_features_vect(inerts)\n        if use_orientation:\n            base = np.concatenate([base, extract_orientation_invariant(inerts)], axis=-1)\n        all_F.append(base)\n        if video_transform is not None:\n            center, comps = video_transform\n            all_V.append(transform_video_pca(vids, center, comps))\n        else:\n            all_V.append(extract_video_features_vect(vids, video_proj_dim))\n        all_y.append(ys)\n\n    F = np.concatenate(all_F, axis=0)   # (N, 400)\n    V = np.concatenate(all_V, axis=0)   # (N, V)\n    y = np.concatenate(all_y, axis=0)   # (N,)\n    return F, V, y\n\n\ndef build_train_features(paths: Paths, stride: int = 25,\n                         max_per_class: int | None = None,\n                         seed: int = 42, video_proj_dim: int = 64,\n                         video_transform=None, use_orientation=False):\n    \"\"\"Build train feature matrix + labels.\n\n    Returns (X (M, F), y (M,), video_X (M, V), sensor_onehot (M, S)).\n    Each window -> N_SENSORS rows (one per sensor view).\n    \"\"\"\n    F, V, ys = _build_train_features(paths, stride, max_per_class, seed,\n                                     video_proj_dim, video_transform,\n                                     use_orientation)\n    N = F.shape[0]\n    per_sensor = F.shape[1] // N_SENSORS\n    F = F.reshape(N, N_SENSORS, per_sensor)\n\n    X = _interleaved(F, N)                 # (N*4, per_sensor)\n    y = np.repeat(ys, N_SENSORS)\n    vid = np.tile(V, (N_SENSORS, 1))\n    sensor = np.repeat(np.eye(N_SENSORS)[None], N, 0).reshape(N * N_SENSORS, -1)\n    return X, y, vid, sensor\n\n\ndef _gather_video_arrays(paths: Paths, stride: int, max_per_class, seed,\n                         max_windows: int = 30000):\n    \"\"\"Return video (N,15,768) for a SUBSET of training windows (for PCA fit).\n\n    Subsamples at the window-metadata level so we never hold all windows' video\n    in memory (the full set would be tens of GB).\n    \"\"\"\n    from .data import build_dataset_windows, load_train_recording, subsample_windows\n    windows = build_dataset_windows(paths, stride)\n    windows = subsample_windows(windows, max_per_class, seed)\n    if max_windows is not None and len(windows) > max_windows:\n        rng = np.random.RandomState(seed)\n        windows = windows.sample(max_windows, random_state=seed)\n    vids = []\n    cur_rec = None; cur_r = None\n    for _, row in windows.iterrows():\n        rec = row[\"rec\"]\n        if rec != cur_rec:\n            cur_r = load_train_recording(paths, rec); cur_rec = rec\n        v0 = int(row[\"video_start\"])\n        vids.append(np.nan_to_num(np.asarray(cur_r.video[v0:v0 + VIDEO_WINDOW], dtype=np.float64)))\n    return np.array(vids)\n\n\ndef fit_video_pca(paths: Paths, stride: int = 25, max_per_class: int = None,\n                  seed: int = 42, n_components: int = 64,\n                  max_fit_samples: int = 30000):\n    \"\"\"Fit PCA on pooled train video features (mean+std of each 768-dim frame).\n\n    Fits on a subsample to keep it fast and memory-safe.\n    \"\"\"\n    vids = _gather_video_arrays(paths, stride, max_per_class, seed,\n                                max_windows=max_fit_samples)\n    mean = vids.mean(axis=1)               # (N,768)\n    std = vids.std(axis=1)\n    feats = np.concatenate([mean, std], axis=1)   # (N,1536)\n    center = feats.mean(axis=0)\n    Xc = feats - center\n    try:\n        from sklearn.decomposition import TruncatedSVD\n        svd = TruncatedSVD(n_components=n_components, random_state=seed)\n        svd.fit(Xc)\n        components = svd.components_.T        # (1536, n_components)\n    except Exception:\n        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)\n        components = Vt[:n_components].T\n    return center, components\n\n\ndef transform_video_pca(videos: np.ndarray, center, components) -> np.ndarray:\n    v = np.asarray(videos, dtype=np.float64)\n    mean = v.mean(axis=1)\n    std = v.std(axis=1)\n    feats = np.concatenate([mean, std], axis=1)\n    return (feats - center) @ components\n\n\ndef _interleaved(F, N):\n    \"\"\"F (N,4,76) -> X (N*4, 76) interleaved by (window, sensor).\"\"\"\n    return F.reshape(N * N_SENSORS, -1)\n\n\ndef build_test_features(paths: Paths, video_proj_dim: int = 64,\n                        video_transform=None, use_orientation=False):\n    \"\"\"Build test feature matrix aligned to test meta row order.\n\n    Returns (X (N, F), ids, sensor_onehot (N, S), video_X (N, V)).\n    \"\"\"\n    from .data import build_test_dataset\n    inertial, video, sensor_ids, ids, _ = build_test_dataset(paths)\n    inertial = np.nan_to_num(np.asarray(inertial, dtype=np.float64))  # (N,50,3)\n    video = np.nan_to_num(np.asarray(video, dtype=np.float64))        # (N,15,768)\n\n    F = extract_acc_features_vect(inertial)          # (N,100) single sensor\n    if use_orientation:\n        F = np.concatenate([F, extract_orientation_invariant(inertial)], axis=-1)  # (N,259)\n    if video_transform is not None:\n        center, comps = video_transform\n        V = transform_video_pca(video, center, comps)   # (N,V)\n    else:\n        V = extract_video_features_vect(video, video_proj_dim)  # (N,V)\n\n    sensor_oh = np.zeros((len(ids), N_SENSORS))\n    sensor_oh[np.arange(len(ids)), sensor_ids] = 1.0\n    # Xt is the sensor features only (matches train X); caller adds video+sensor.\n    return F, ids, sensor_oh, V",
    "run_boost": "\"\"\"Gradient-boosting classifier on engineered features (proven WEAR winner).\n\nRich time/frequency features per sensor view + pooled VideoMAE features, trained\nwith LightGBM (multi-class), then test predictions.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features,\n    build_test_features,\n    SENSOR_FEAT_DIM,\n)\nfrom .inference import write_submission\n\n\ndef _check_video_alignment(paths):\n    \"\"\"Cheap diagnostic: compare train vs test video feature scale (mean/std of values).\"\"\"\n    from .data import load_train_recording, train_stems, build_test_dataset\n    train_vals = []\n    for stem in train_stems(paths)[:4]:\n        r = load_train_recording(paths, stem)\n        v = np.asarray(r.video)\n        train_vals.append(v.ravel())\n    train_all = np.concatenate(train_vals)\n    _, test_video, _, _, _ = build_test_dataset(paths)\n    tv = np.asarray(test_video).ravel()\n    print(f\"[video-align] train mean={train_all.mean():.4f} std={train_all.std():.4f} | \"\n          f\"test mean={tv.mean():.4f} std={tv.std():.4f}\")\n    if train_all.std() > 0 and abs(train_all.std() / (tv.std() + 1e-8) - 1.0) > 0.5:\n        print(\"NOTE: train/test video std differ >50% - check alignment\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=None)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--max-depth\", type=int, default=-1)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--use-video\", type=int, default=1)\n    ap.add_argument(\"--video-pca\", type=int, default=1,\n                    help=\"1 to use PCA video features (fit on train), 0 for random projection\")\n    ap.add_argument(\"--ensemble\", type=int, default=1,\n                    help=\"number of diverse boosters to average (ensemble)\")\n    ap.add_argument(\"--seed\", type=int, default=42)\n    args = ap.parse_args()\n\n    cfg = Config()\n    device_note = \"cpu (boosting)\"\n\n    _check_video_alignment(cfg.paths)\n\n    # Fit video PCA on train (optional but recommended).\n    video_transform = None\n    if args.use_video and args.video_pca:\n        from .feature_data import fit_video_pca\n        print(\"fitting video PCA...\", flush=True)\n        video_transform = fit_video_pca(\n            cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n            seed=args.seed, n_components=args.video_proj_dim)\n        print(\"video PCA fit done\", flush=True)\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=args.seed, video_proj_dim=args.video_proj_dim,\n        video_transform=video_transform)\n    print(f\"train samples (per-sensor views) = {X.shape}, classes={np.unique(y).tolist()}\", flush=True)\n\n    if args.use_video:\n        F = np.concatenate([X, vid, sensor], axis=1)\n    else:\n        F = X\n    print(f\"feature matrix: {F.shape}\", flush=True)\n\n    import lightgbm as lgb\n    # Class weights (inverse frequency) for macro-F1 oriented training.\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n\n    # Ensemble of boosters with diverse feature subsets + seeds (boosting ensemble).\n    n_ensemble = max(1, args.ensemble)\n    import lightgbm as lgb\n    test_preds = []\n    for bi in range(n_ensemble):\n        seed = args.seed + bi\n        col_frac = 0.7 + 0.2 * (bi % 2)   # alternate 0.7 / 0.9\n        params = dict(\n            objective=\"multiclass\", num_class=N_CLASSES,\n            n_estimators=args.n_estimators, learning_rate=args.lr,\n            num_leaves=args.num_leaves, max_depth=args.max_depth,\n            subsample=0.8, colsample_bytree=col_frac, reg_lambda=1.0,\n            min_child_samples=30, n_jobs=8, random_state=seed,\n            verbose=-1,\n        )\n        model = None\n        try:\n            params[\"device_type\"] = \"gpu\"\n            model = lgb.LGBMClassifier(**params)\n            model.fit(F, y, sample_weight=sw)\n            print(f\"LightGBM[{bi}] trained on GPU\", flush=True)\n        except Exception as e:\n            print(f\"GPU LightGBM failed ({e}); using CPU\", flush=True)\n            params[\"device_type\"] = \"cpu\"\n            model = lgb.LGBMClassifier(**params)\n            model.fit(F, y, sample_weight=sw)\n            print(f\"LightGBM[{bi}] trained on CPU\", flush=True)\n\n        print(\"building test features...\", flush=True)\n        Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths,\n                                                       args.video_proj_dim,\n                                                       video_transform)\n        if args.use_video:\n            Ft = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n        else:\n            Ft = Xt\n        print(f\"test feature matrix: {Ft.shape}\", flush=True)\n        p = model.predict_proba(Ft)\n        if p.shape[1] < N_CLASSES:\n            pad = np.zeros((p.shape[0], N_CLASSES - p.shape[1]))\n            p = np.concatenate([p, pad], axis=1)\n        test_preds.append(p)\n\n    probs = np.mean(test_preds, axis=0)         # average ensemble\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "fame_model": "\"\"\"FAME-style feature-augmented multi-view neural network.\n\nAdapts the prior WEAR winner (FAME) for the current test setup:\n  * Feature augmentation: append frequency-domain features to the raw inertial\n    channels (each sensor gets raw 3 axes + magnitude + per-axis freq features).\n  * Channel-wise random sign-flipping augmentation (FAME's main driver of\n    generalization).\n  * Multi-view: a shared per-sensor encoder produces per-sensor embeddings\n    (views); a sensor-location embedding conditions the view; predictions are\n    made per sensor and the present sensor is used at test time.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import INERTIAL_WINDOW, N_CLASSES, N_SENSORS, VIDEO_DIM, VIDEO_WINDOW\n\n\ndef _append_freq_channels(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"x: (B, N_SENSORS, 3, T) -> (B, N_SENSORS, 3+2, T).\n\n    Appends 2 frequency-domain proxy features (zero-crossing rate and\n    differencing energy) as constant-over-time channels, per sensor.\n    \"\"\"\n    B, S, A, T = x.shape\n    xm = x - x.mean(dim=-1, keepdim=True)\n    zc = ((xm[:, :, :, 1:] * xm[:, :, :, :-1]) < 0).float().mean(dim=-1)   # (B,S,3)\n    d = (xm[:, :, :, 1:] - xm[:, :, :, :-1]).abs().mean(dim=-1)            # (B,S,3)\n    extra = torch.stack([zc.mean(dim=-1), d.mean(dim=-1)], dim=-1)         # (B,S,2)\n    extra = extra[:, :, :, None].expand(B, S, 2, T)                        # (B,S,2,T)\n    return torch.cat([x, extra], dim=2)                                    # (B,S,5,T)\n\n\nclass FAMEInceptionBlock(nn.Module):\n    def __init__(self, cin, out, dropout=0.1):\n        super().__init__()\n        bottleneck = max(1, out // 4)\n        self.bottleneck = nn.Conv1d(cin, bottleneck, 1, bias=False)\n        self.branches = nn.ModuleList([\n            nn.Conv1d(bottleneck, bottleneck, k, padding=k // 2, bias=False)\n            for k in (5, 11, 21)\n        ])\n        self.pool = nn.Sequential(nn.MaxPool1d(3, 1, 1),\n                                  nn.Conv1d(cin, bottleneck, 1, bias=False))\n        self.skip = (nn.Conv1d(cin, out, 1, bias=False) if cin != out else nn.Identity())\n        self.norm = nn.GroupNorm(min(8, out), out)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x):\n        b = self.bottleneck(x)\n        br = [m(b) for m in self.branches] + [self.pool(x)]\n        out = torch.cat(br, dim=1) + self.skip(x)\n        return self.drop(F.gelu(self.norm(out)))\n\n\nclass FAMEVideoEncoder(nn.Module):\n    def __init__(self, hidden=192, layers=2, heads=4):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(VIDEO_DIM), nn.Linear(VIDEO_DIM, hidden), nn.GELU())\n        pos = torch.zeros(1, VIDEO_WINDOW, hidden); nn.init.trunc_normal_(pos, std=0.02)\n        self.pos = nn.Parameter(pos)\n        self.enc = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(hidden, heads, hidden * 4, 0.15, \"gelu\",\n                                       batch_first=True, norm_first=True),\n            layers, enable_nested_tensor=False)\n        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2))\n\n    def forward(self, x):  # (B,15,768)\n        h = self.proj(x) + self.pos\n        h = self.enc(h)\n        att = h.mean(1)\n        return self.head(torch.cat([att, h.mean(1)], dim=-1))\n\n\nclass FAMEViewModel(nn.Module):\n    \"\"\"Shared per-sensor encoder + view-specific branches + sensor conditioning.\"\"\"\n\n    def __init__(self, n_classes=N_CLASSES, hidden=192, channels=128,\n                 video_layers=2, sensor_dim=16, dropout=0.2):\n        super().__init__()\n        self.inertial_encoder = nn.Sequential(\n            FAMEInceptionBlock(6, channels),\n            FAMEInceptionBlock(channels, channels),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(channels * 2, hidden), nn.LayerNorm(hidden),\n            nn.GELU(), nn.Dropout(dropout))\n        self.video_encoder = FAMEVideoEncoder(hidden=hidden, layers=video_layers)\n        self.sensor_embed = nn.Embedding(N_SENSORS, sensor_dim)\n        gate_in = hidden + hidden + sensor_dim\n        self.gate = nn.Sequential(nn.Linear(gate_in, hidden), nn.Sigmoid())\n        self.classifier = nn.Sequential(\n            nn.Linear(hidden * 2 + sensor_dim, hidden),\n            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, n_classes))\n        self.sensor_dropout = 0.2\n        self.sign_flip = True\n\n    def forward(self, inertial, video):\n        \"\"\"inertial (B,50,4,3), video (B,15,768) -> (B,4,n_classes).\"\"\"\n        B = inertial.shape[0]\n        inert = inertial.permute(0, 2, 3, 1)            # (B,4,50,3)\n        if self.training and self.sensor_dropout > 0:\n            keep = torch.rand(B, N_SENSORS, device=inert.device) >= self.sensor_dropout\n            inert = inert * keep[:, :, None, None].float()\n        # per-sensor: (B*4, 3, 50) -> add magnitude -> (B*4,4,50)\n        x = inert.reshape(B * N_SENSORS, 3, INERTIAL_WINDOW)\n        mag = torch.linalg.vector_norm(x, dim=1, keepdim=True)\n        x = torch.cat([x, mag], dim=1)                  # (B*4,4,50)\n        # sign-flip augmentation\n        if self.training and self.sign_flip:\n            flip = (torch.rand(x.shape[0], 1, 1, device=x.device) < 0.5).float() * 2 - 1\n            x = x * flip\n        # append frequency features to the 3 raw axes -> (B*4, 5, 50)\n        x3 = x[:, :3]                                   # (B*4,3,50)\n        xf = _append_freq_channels(\n            x3.reshape(B, N_SENSORS, 3, INERTIAL_WINDOW)\n        ).reshape(B * N_SENSORS, 5, INERTIAL_WINDOW)\n        # combine: 3 axes + 2 freq + magnitude = 6 channels\n        x = torch.cat([xf, x[:, 3:4]], dim=1)           # (B*4,6,50)\n        h = self.inertial_encoder(x)\n        pooled = torch.cat([h.mean(2), h.amax(2)], dim=1)\n        i_feat = self.head(pooled).reshape(B, N_SENSORS, -1)\n\n        v_feat = self.video_encoder(video)[:, None].expand(B, N_SENSORS, -1)\n        pos = self.sensor_embed.weight[None].expand(B, N_SENSORS, -1)\n        gate = torch.sigmoid(self.gate(torch.cat([i_feat, v_feat, pos], dim=-1)))\n        fused = gate * i_feat + (1 - gate) * v_feat\n        inter = i_feat * v_feat\n        logits = self.classifier(torch.cat([fused, inter, pos], dim=-1))\n        return logits",
    "run_fame": "\"\"\"Train the FAME-style feature-augmented multi-view model and submit.\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import (\n    InertialNormalizer,\n    WearTrainDataset,\n    build_dataset_windows,\n    subsample_windows,\n)\nfrom .inference import write_submission\nfrom .fame_model import FAMEViewModel\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag=\"\", seed=0,\n           label_smooth=0.05, warmup_epochs=1, grad_norm=5.0):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warmup = warmup_epochs * len(dl)\n\n    def ll(step):\n        if step < warmup:\n            return step / max(1, warmup)\n        p = (step - warmup) / max(1, total - warmup)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r = 0.0\n        nb = 0\n        for batch in dl:\n            inert = batch[\"inertial\"].to(device)\n            vid = batch[\"video\"].to(device)\n            tgt = batch[\"target\"].to(device)\n            valid = batch[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = criterion(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            if grad_norm:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item()\n            nb += 1\n        print(f\"[{tag}] epoch {ep+1}/{epochs} loss={r/max(1,nb):.4f} \"\n              f\"lr={opt.param_groups[0]['lr']:.2e}\")\n\n\ndef _predict_test(model, paths, device, use_amp, normalizer):\n    from . import inference as inf\n    ds = inf.WearTestDataset(paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                    collate_fn=inf._collate)\n    model.eval()\n    all_probs, all_ids = [], []\n    with torch.inference_mode():\n        for inert, vid, sensor_id, ids in dl:\n            inert = inert.to(device); vid = vid.to(device)\n            sensor_id = sensor_id.to(device)\n            inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)\n            present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n            present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n            inert = inert * present[:, None, :, None]\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid)\n            probs = torch.softmax(logits.float(), dim=-1)\n            probs = probs[torch.arange(probs.shape[0], device=device),\n                          sensor_id, :]\n            all_probs.append(probs.cpu().numpy())\n            all_ids.append(ids.numpy())\n    probs = np.concatenate(all_probs, axis=0)\n    ids = np.concatenate(all_ids, axis=0)\n    o = np.argsort(ids)\n    return ids[o], probs[o]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=10)\n    ap.add_argument(\"--lr\", type=float, default=7e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=4)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n    out_dir = cfg.paths.output_dir / \"fame\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"building windows...\")\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\")\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True,\n                    num_workers=4, pin_memory=True,\n                    worker_init_fn=lambda w: np.random.seed(0 + w),\n                    persistent_workers=True)\n\n    acc = None\n    ref_ids = None\n    for seed in range(args.seeds):\n        model = FAMEViewModel().to(device)\n        nparam = sum(p.numel() for p in model.parameters())\n        print(f\"seed {seed}: params={nparam/1e6:.2f}M\")\n        _train(model, dl, args.epochs, args.lr, 1e-3, device, use_amp,\n               tag=f\"fame-{seed}\", seed=seed)\n        ids, probs = _predict_test(model, cfg.paths, device, use_amp, normalizer)\n        ref_ids = ids\n        acc = probs if acc is None else acc + probs\n\n    blended = acc / args.seeds\n    blended = blended.copy()\n    blended[:, 0] *= np.exp(args.null_bias)\n    blended = blended / blended.sum(1, keepdims=True)\n\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ref_ids, blended, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\")\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_ensemble": "\"\"\"Fast diverse-model ensemble + Test-Time Augmentation (GPU).\n\nTrains 3 diverse models that all run on GPU (fast):\n  * FAME-style multi-view neural (freq channels + sign-flip + video)\n  * Strong fusion neural (deep video transformer + lean inertial + video)\n  * Small MLP on engineered features (sub-window/freq + pooled video)\n\nApplies Test-Time Augmentation (noise + feature flip) and averages predictions.\nDesigned to complete in ~30-45 min on a single GPU.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .data import InertialNormalizer, WearTrainDataset, build_dataset_windows, subsample_windows\nfrom .fame_model import FAMEViewModel\nfrom .strong_model import StrongFusionModel\nfrom .inference import WearTestDataset, write_submission\n\n\ndef _collate_loader(ds, bs, shuffle, worker_seed):\n    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=4,\n                      pin_memory=True,\n                      worker_init_fn=lambda w: np.random.seed(worker_seed + w),\n                      persistent_workers=True)\n\n\ndef _train(model, dl, epochs, lr, wd, device, use_amp, tag, seed):\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    crit = nn.CrossEntropyLoss(label_smoothing=0.05)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)\n    total = epochs * len(dl)\n    warm = 1 * len(dl)\n\n    def ll(step):\n        if step < warm:\n            return step / max(1, warm)\n        p = (step - warm) / max(1, total - warm)\n        return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n    model.train()\n    for ep in range(epochs):\n        r, nb = 0.0, 0\n        for b in dl:\n            inert = b[\"inertial\"].to(device)\n            vid = b[\"video\"].to(device)\n            tgt = b[\"target\"].to(device)\n            valid = b[\"valid\"].to(device)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                logits = model(inert, vid).reshape(-1, N_CLASSES)\n                tt = tgt[:, None].expand(-1, N_SENSORS).reshape(-1)\n                vv = valid.reshape(-1)\n                loss = crit(logits[vv], tt[vv]) if vv.any() else torch.tensor(0.0, device=device)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            r += loss.item(); nb += 1\n        print(f\"[{tag}] ep {ep+1}/{epochs} loss={r/max(1,nb):.4f}\", flush=True)\n    return model\n\n\ndef _predict_tta(model, paths, device, use_amp, normalizer, n_tta=3):\n    \"\"\"Predict test with TTA (noise + sign-flip on inertial, averaged).\"\"\"\n    model.eval()\n    all_probs = None\n    all_ids = None\n    for t in range(n_tta):\n        ds = WearTestDataset(paths, normalizer=normalizer)\n        dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0,\n                        collate_fn=_test_collate)\n        probs = []\n        ids_out = None\n        with torch.inference_mode():\n            for inert, vid, sensor_id, ids in dl:\n                inert = inert.to(device); vid = vid.to(device); sensor_id = sensor_id.to(device)\n                inert = inert.unsqueeze(2).expand(-1, -1, N_SENSORS, -1)  # (B,50,4,3)\n                present = torch.zeros(inert.shape[0], N_SENSORS, device=device)\n                present.scatter_(1, sensor_id.unsqueeze(1), 1.0)\n                inert = inert * present[:, None, :, None]\n                if t > 0:\n                    # TTA: small noise + channel sign flip (on present sensors)\n                    noise = torch.randn_like(inert) * 0.05\n                    flip = (torch.rand(inert.shape[0], 1, N_SENSORS, 1, device=device) < 0.1).float() * 2 - 1\n                    inert = inert * flip + noise\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = model(inert, vid)\n                p = torch.softmax(logits.float(), dim=-1)\n                p = p[torch.arange(p.shape[0], device=device), sensor_id, :].cpu().numpy()\n                probs.append(p)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        p = np.concatenate(probs)\n        o = np.argsort(ids_out)\n        p = p[o]\n        all_probs = p if all_probs is None else all_probs + p\n        all_ids = ids_out if all_ids is None else np.sort(ids_out)\n    return all_ids, all_probs / n_tta\n\n\ndef _test_collate(batch):\n    import torch\n    inert = torch.as_tensor(np.stack([b[\"inertial\"] for b in batch]))\n    vid = torch.as_tensor(np.stack([b[\"video\"] for b in batch]))\n    sid = torch.as_tensor([b[\"sensor_id\"] for b in batch])\n    ids = torch.as_tensor([b[\"id\"] for b in batch])\n    return inert, vid, sid, ids\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=150)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=6)\n    ap.add_argument(\"--fame-seeds\", type=int, default=2)\n    ap.add_argument(\"--strong-seeds\", type=int, default=2)\n    ap.add_argument(\"--tta\", type=int, default=3)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.data.window_stride = args.stride\n    cfg.data.max_windows_per_class_per_subject = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, cfg.data.window_stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.data.seed)\n    print(f\"windows = {len(windows)}\", flush=True)\n    normalizer = InertialNormalizer.fit(cfg.paths)\n\n    ds = WearTrainDataset(windows, cfg.paths, normalizer=normalizer)\n    dl = _collate_loader(ds, 256, True, cfg.data.seed)\n\n    # ---- Train diverse models ----\n    ensemble_probs = None\n    ids = None\n\n    for s in range(args.fame_seeds):\n        m = FAMEViewModel().to(device)\n        _train(m, dl, args.epochs, 7e-4, 1e-3, device, use_amp, f\"fame{s}\", cfg.data.seed + s)\n        i, p = _predict_tta(m, cfg.paths, device, use_amp, normalizer, args.tta)\n        ids = i\n        ensemble_probs = p if ensemble_probs is None else ensemble_probs + p\n        del m; torch.cuda.empty_cache()\n\n    for s in range(args.strong_seeds):\n        m = StrongFusionModel(video_layers=2, video_hidden=192).to(device)\n        _train(m, dl, args.epochs, 6e-4, 1e-3, device, use_amp, f\"strong{s}\", cfg.data.seed + s)\n        i, p = _predict_tta(m, cfg.paths, device, use_amp, normalizer, args.tta)\n        ids = i\n        ensemble_probs = ensemble_probs + p\n        del m; torch.cuda.empty_cache()\n\n    n_models = args.fame_seeds + args.strong_seeds\n    ensemble_probs = ensemble_probs / n_models\n    ensemble_probs = ensemble_probs.copy()\n    ensemble_probs[:, 0] *= np.exp(args.null_bias)\n    ensemble_probs = ensemble_probs / ensemble_probs.sum(1, keepdims=True)\n\n    from pathlib import Path\n    sub_path = Path(\"/kaggle/working/submission.csv\")\n    sub_path.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, ensemble_probs, cfg.paths.sample_submission, sub_path)\n    print(f\"submission written -> {sub_path}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "video_only": "\"\"\"Video-only temporal baseline (the critical diagnostic).\n\nThe hypothesis (from analysis): mean-pooling the 15x768 VideoMAE sequence\ndestroys the temporal information that distinguishes the 19 activities.  This\nmodel keeps the full 15-frame sequence, models it temporally (Transformer),\nand classifies without pooling to a single mean vector.\n\nPer ChatGPT's spec: small (2 layers, 4 heads, hidden 256, ~1M params).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import N_CLASSES\n\n\nclass VideoOnlyTransformer(nn.Module):\n    \"\"\"(B,15,768) -> (B,19). Temporal Transformer, attention pooling.\"\"\"\n\n    def __init__(self, hidden: int = 256, layers: int = 2, heads: int = 4,\n                 n_classes: int = N_CLASSES, dropout: float = 0.2):\n        super().__init__()\n        self.proj = nn.Sequential(\n            nn.LayerNorm(768), nn.Linear(768, hidden), nn.GELU())\n        pos = torch.zeros(1, 15, hidden)\n        nn.init.trunc_normal_(pos, std=0.02)\n        self.pos = nn.Parameter(pos)\n        self.enc = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(hidden, heads, hidden * 4, dropout,\n                                       \"gelu\", batch_first=True, norm_first=True),\n            layers, enable_nested_tensor=False)\n        # attention pooling\n        self.attn = nn.Sequential(nn.Linear(hidden, 64), nn.Tanh(), nn.Linear(64, 1))\n        self.head = nn.Sequential(\n            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, n_classes))\n\n    def forward(self, video):\n        \"\"\"video: (B,15,768) -> (B,19).\"\"\"\n        h = self.proj(video) + self.pos\n        h = self.enc(h)                       # (B,15,hidden)\n        scores = self.attn(h).squeeze(-1)\n        w = torch.softmax(scores, dim=1).unsqueeze(-1)\n        attended = (h * w).sum(dim=1)\n        pooled = torch.cat([attended, h.mean(dim=1)], dim=-1)\n        return self.head(pooled)",
    "run_video": "\"\"\"Train + submit the video-only temporal baseline.\n\nDiagnostic #1: how much signal does the full 15-frame VideoMAE sequence carry?\nIf this scores ~0.75-0.85, video is the key and our 0.657 was a feature bottleneck.\nIf ~0.55-0.65, the winning teams exploit something else.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .config import Config, N_CLASSES\nfrom .data import (\n    build_dataset_windows, subsample_windows, load_train_recording,\n    VIDEO_WINDOW, VIDEO_FRAME_OFFSET,\n)\nfrom .video_only import VideoOnlyTransformer\n\n\nclass VideoWinDataset(Dataset):\n    \"\"\"Preloaded per-window video (vectorized per recording) -> (15,768), target.\"\"\"\n\n    def __init__(self, windows, paths, augment=False, seed=0):\n        self.windows = windows.reset_index(drop=True)\n        self.augment = augment\n        self.rng = np.random.RandomState(seed)\n        vids, ys = [], []\n        for rec in windows[\"rec\"].unique():\n            r = load_train_recording(paths, rec)\n            sub = windows[windows[\"rec\"] == rec]\n            vstarts = sub[\"video_start\"].to_numpy()\n            vidx = vstarts[:, None] + np.arange(VIDEO_WINDOW)[None, :]\n            v = np.asarray(r.video[vidx], dtype=np.float32)\n            vids.append(np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0))\n            ys.append(sub[\"target\"].to_numpy())\n        self.vids = np.concatenate(vids) if vids else np.zeros((0, VIDEO_WINDOW, 768), np.float32)\n        self.ys = np.concatenate(ys) if ys else np.zeros(0, np.int64)\n\n    def __len__(self):\n        return len(self.windows)\n\n    def __getitem__(self, idx):\n        v = self.vids[idx]\n        if self.augment:\n            # mild frame noise + feature masking\n            v = v + self.rng.randn(*v.shape).astype(np.float32) * 0.02\n        return v, int(self.ys[idx])\n\n\ndef _collate(batch):\n    vids = np.stack([b[0] for b in batch])\n    tgt = np.array([b[1] for b in batch])\n    return torch.as_tensor(vids), torch.as_tensor(tgt, dtype=torch.long)\n\n\nclass VideoTestDataset(Dataset):\n    def __init__(self, paths):\n        from .data import build_test_dataset\n        _, video, _, ids, _ = build_test_dataset(paths)\n        self.video = np.nan_to_num(np.asarray(video, dtype=np.float32))\n        self.ids = ids\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, idx):\n        return self.video[idx], self.ids[idx]\n\n\ndef _collate_test(batch):\n    vids = torch.as_tensor(np.stack([b[0] for b in batch]))\n    ids = torch.as_tensor([b[1] for b in batch])\n    return vids, ids\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=300)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--epochs\", type=int, default=8)\n    ap.add_argument(\"--lr\", type=float, default=5e-4)\n    ap.add_argument(\"--batch-size\", type=int, default=256)\n    ap.add_argument(\"--seeds\", type=int, default=3)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n    cfg.reservoir.max_per_class = args.max_per_class\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    use_amp = device.type == \"cuda\"\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.reservoir.seed)\n    print(f\"windows = {len(windows)}\", flush=True)\n\n    ds = VideoWinDataset(windows, cfg.paths, augment=True, seed=cfg.reservoir.seed)\n    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=0,\n                    collate_fn=_collate)\n    tds = VideoTestDataset(cfg.paths)\n    tdl = DataLoader(tds, batch_size=args.batch_size, shuffle=False, num_workers=0,\n                     collate_fn=_collate_test)\n\n    crit = nn.CrossEntropyLoss(label_smoothing=0.05)\n    acc_probs = None\n    test_ids = None\n    for seed in range(args.seeds):\n        torch.manual_seed(seed)\n        model = VideoOnlyTransformer().to(device)\n        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n        total = args.epochs * len(dl)\n        warm = 1 * len(dl)\n\n        def ll(step):\n            if step < warm:\n                return step / max(1, warm)\n            p = (step - warm) / max(1, total - warm)\n            return 0.5 * (1 + float(np.cos(p * np.pi)))\n\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n        model.train()\n        for ep in range(args.epochs):\n            r, nb = 0.0, 0\n            for video, tgt in dl:\n                video = video.to(device)\n                tgt = tgt.to(device)\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    loss = crit(model(video), tgt)\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                scaler.step(opt)\n                scaler.update()\n                sched.step()\n                r += loss.item(); nb += 1\n            print(f\"[video] seed{seed} ep {ep+1}/{args.epochs} loss={r/max(1,nb):.4f}\", flush=True)\n\n        # predict test\n        model.eval()\n        probs = []\n        ids_out = None\n        with torch.inference_mode():\n            for video, ids in tdl:\n                video = video.to(device)\n                with torch.autocast(\"cuda\", enabled=use_amp, dtype=torch.float16):\n                    logits = model(video)\n                p = torch.softmax(logits.float(), dim=-1).cpu().numpy()\n                probs.append(p)\n                ids_out = ids.numpy() if ids_out is None else np.concatenate([ids_out, ids.numpy()])\n        p = np.concatenate(probs)\n        o = np.argsort(ids_out)\n        p = p[o]\n        test_ids = np.sort(ids_out)\n        acc_probs = p if acc_probs is None else acc_probs + p\n\n    probs = acc_probs / args.seeds\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(test_ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_video_knn": "\"\"\"Video kNN / retrieval baseline (ChatGPT's #1 bet, tested cheaply).\n\nEven though a video classifier generalizes poorly (0.45), retrieval matches each\ntest window to the most similar TRAINING windows, which might transfer better if\ntest windows visually resemble train windows of the same class.\n\nNo neural training needed - just extract video embeddings (mean of 15 frames,\noptionally + PCA) and find nearest training windows by cosine distance.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config\nfrom .data import (\n    build_dataset_windows, subsample_windows, load_train_recording,\n    build_test_dataset, VIDEO_WINDOW,\n)\n\n\ndef _video_embedding(v):\n    \"\"\"v: (15,768) -> embedding (mean frame, and mean of frame norms).\"\"\"\n    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)\n    mean = v.mean(axis=0)                 # (768,)\n    std = v.std(axis=0)\n    return np.concatenate([mean, std])    # (1536,)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=300)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--k\", type=int, default=10)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--distance\", type=str, default=\"cosine\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building windows...\", flush=True)\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    windows = subsample_windows(windows, args.max_per_class, cfg.reservoir.seed)\n    print(f\"train windows = {len(windows)}\", flush=True)\n\n    # Extract train video embeddings + labels\n    print(\"extracting train video embeddings...\", flush=True)\n    train_feats, train_y = [], []\n    for rec in windows[\"rec\"].unique():\n        r = load_train_recording(cfg.paths, rec)\n        sub = windows[windows[\"rec\"] == rec]\n        vstarts = sub[\"video_start\"].to_numpy()\n        ys = sub[\"target\"].to_numpy()\n        for v0, y in zip(vstarts, ys):\n            v = np.asarray(r.video[v0:v0 + VIDEO_WINDOW], dtype=np.float32)\n            train_feats.append(_video_embedding(v))\n            train_y.append(int(y))\n    X_tr = np.stack(train_feats)          # (T,1536)\n    y_tr = np.array(train_y)\n    # L2-normalize for cosine\n    X_tr = X_tr / (np.linalg.norm(X_tr, axis=1, keepdims=True) + 1e-9)\n    print(f\"train embeddings: {X_tr.shape}\", flush=True)\n\n    # Test embeddings\n    print(\"extracting test video embeddings...\", flush=True)\n    _, video, _, ids, _ = build_test_dataset(cfg.paths)\n    test_feats = np.stack([_video_embedding(np.asarray(video[i], np.float32)) for i in range(len(ids))])\n    X_te = test_feats / (np.linalg.norm(test_feats, axis=1, keepdims=True) + 1e-9)\n    print(f\"test embeddings: {X_te.shape}\", flush=True)\n\n    # kNN by cosine (dot product of L2-normalized vectors)\n    print(f\"computing {args.k}-NN...\", flush=True)\n    sim = X_te @ X_tr.T                 # (N, T)\n    k = min(args.k, X_tr.shape[0])\n    top_idx = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]\n    # gather labels of top-k, weight by similarity^2\n    n_classes = 19\n    probs = np.zeros((len(X_te), n_classes))\n    for i in range(len(X_te)):\n        idxs = top_idx[i]\n        s = sim[i, idxs]\n        w = np.clip(s, 0, None) ** 2 + 1e-9\n        labels = y_tr[idxs]\n        for j, lab in enumerate(labels):\n            probs[i, lab] += w[j]\n        probs[i] /= probs[i].sum()\n\n    # null bias\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    o = np.argsort(ids)\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids[o], probs[o], cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_sensor_spec": "\"\"\"Priority 1: 4 separate sensor-specialist LightGBMs (Submission A).\n\nEach sensor gets its OWN model using the champion features. At test, a window\nis routed to its sensor's specialist. This tests whether sensor-specific\nmodeling (vs one shared LightGBM + one-hot) improves generalization.\n\nExpected: +0.01 to +0.06 if sensor physics differ meaningfully.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features, build_test_features,\n)\nfrom .features import extract_acc_features_vect, extract_video_features_vect\n\n\ndef _fit_one(X, y, seed, n_est, lr, leaves):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(X, y, sample_weight=sw)\n    return m\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--use-video\", type=int, default=1)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n\n    # X is per-sensor-view flattened: (N*4, 100). Ordering is\n    # (w0s0,w0s1,w0s2,w0s3, w1s0,...). sensor one-hot tells which.\n    N_views = X.shape[0]\n    # split by sensor\n    sens_id = sensor.argmax(axis=1)   # (N*4,)\n    feat_vid = np.concatenate([X, vid, sensor], axis=1) if args.use_video else X\n    print(f\"total views={N_views}\", flush=True)\n\n    models = {}\n    for s in range(N_SENSORS):\n        mask = sens_id == s\n        Xs = feat_vid[mask]\n        ys = y[mask]\n        print(f\"sensor {s}: {Xs.shape[0]} views\", flush=True)\n        models[s] = _fit_one(Xs, ys, cfg.reservoir.seed + s,\n                             args.n_estimators, args.lr, args.num_leaves)\n        print(f\"sensor {s} model trained\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    if args.use_video:\n        Ft = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    else:\n        Ft = Xt\n    N = len(ids)\n    test_sens = sensor_t.argmax(axis=1)   # (N,)\n\n    probs = np.zeros((N, N_CLASSES))\n    for s in range(N_SENSORS):\n        mask = test_sens == s\n        if not mask.any():\n            continue\n        p = models[s].predict_proba(Ft[mask])\n        if p.shape[1] < N_CLASSES:\n            pad = np.zeros((p.shape[0], N_CLASSES - p.shape[1]))\n            p = np.concatenate([p, pad], axis=1)\n        probs[mask] = p\n\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_domain_weight": "\"\"\"Priority 3: domain-density importance weighting (Submission 3).\n\nMotivation: \"more training data hurts\" (0.657 -> 0.586). Some train windows are\nharmful for the test domain. We train a domain classifier (train=0, test=1) on\nchampion features, then weight each train window by P(test|x)/P(train|x) so\ntest-like examples get higher weight, and train the champion LightGBM with\nthose sample weights (per-sensor domain model, but SHARED LightGBM).\n\nDe-risked: ChatGPT notes to check whether the method \"consistently survives\ndomain changes\" on synthetic train-splits, but the champion pipeline + weights\nis the executable version.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import build_train_features, build_test_features\n\n\ndef _domain_weight(X_src, X_tgt, seed):\n    \"\"\"Return importance weight per source sample = P(tgt|x)/P(src|x) via a\n    train/test binary classifier's odds.\"\"\"\n    from sklearn.linear_model import LogisticRegression\n    n_src = len(X_src)\n    X = np.concatenate([X_src, X_tgt], axis=0)\n    y = np.concatenate([np.zeros(n_src), np.ones(len(X_tgt))])\n    clf = LogisticRegression(max_iter=1000, C=1.0)\n    clf.fit(X, y)\n    p = clf.predict_proba(X_src)[:, 1]          # P(test|x)\n    p = np.clip(p, 1e-4, 1 - 1e-4)\n    w = p / (1 - p)                              # odds = P(test)/P(train)\n    # normalize to mean 1\n    w = w / w.mean()\n    return w\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    print(f\"test views={Fte.shape}\", flush=True)\n\n    # Domain weights per sensor (shared model, per-sensor domain fit)\n    weights = np.ones(len(Ftr))\n    sens_id = sensor.argmax(axis=1)\n    test_sens = sensor_t.argmax(axis=1)\n    for s in range(N_SENSORS):\n        src = Ftr[sens_id == s]\n        tgt = Fte[test_sens == s]\n        if len(tgt) < 50:\n            continue\n        w = _domain_weight(src, tgt, cfg.reservoir.seed + s)\n        weights[sens_id == s] = w\n        print(f\"sensor {s}: src={len(src)} tgt={len(tgt)} weight mean={w.mean():.3f} \"\n              f\"std={w.std():.3f}\", flush=True)\n    # clip extreme weights\n    lo, hi = np.percentile(weights, [1, 99])\n    weights = np.clip(weights, lo, hi)\n    weights = weights / weights.mean()\n\n    # Champion LightGBM with domain weights\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    cls_w = 1.0 / counts[y]\n    cls_w = cls_w / cls_w.mean()\n    sw = weights * cls_w\n    sw = sw / sw.mean()\n\n    model = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=args.n_estimators, learning_rate=args.lr,\n        num_leaves=args.num_leaves, subsample=0.8, colsample_bytree=0.8,\n        reg_lambda=1.0, min_child_samples=30, n_jobs=8,\n        random_state=cfg.reservoir.seed, verbose=-1)\n    model.fit(Ftr, y, sample_weight=sw)\n    print(\"domain-weighted LightGBM trained\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose_duplicates": "\"\"\"Diagnostic #2: detect cross-sensor duplicate/temporal-group windows in TEST.\n\nThe test has ~3050 windows per sensor (4 sensors -> ~12,200 ~= 12,234). If the\ntest comes from ~3050 distinct temporal moments, each captured by multiple\nsensors, then the VIDEO at those moments is near-identical (same camera). This\ndiagnostic finds near-duplicate video windows and reports group structure.\n\nNo training, no submission. Cheap read-only analysis.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport pandas as pd\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--sample\", type=int, default=20000,\n                    help=\"max windows to embed for duplicate detection\")\n    ap.add_argument(\"--threshold\", type=float, default=0.02,\n                    help=\"L2 distance threshold on normalized embedding for 'duplicate'\")\n    args = ap.parse_args()\n\n    from .config import Config\n    from .data import build_test_dataset\n    cfg = Config()\n\n    print(\"loading test data...\", flush=True)\n    inertial, video, sensor_ids, ids, sbj = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n    print(f\"test: {len(ids)} windows, subjects={np.unique(sbj).tolist()}\", flush=True)\n\n    # Embed each window's video as mean frame + std (compressed).\n    v = np.asarray(video, dtype=np.float64)\n    n = min(len(v), args.sample)\n    vid = v[:n]\n    mean = vid.mean(axis=1)          # (n,768)\n    std = vid.std(axis=1)\n    emb = np.concatenate([mean, std], axis=1)   # (n,1536)\n    # L2 normalize\n    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    print(f\"embedded {n} video windows\", flush=True)\n\n    # Fast near-duplicate detection via sorted L2 (brute force on sample is fine for n~12k)\n    # Reduce memory: only compute for a subsample if needed.\n    n2 = len(emb)\n    print(\"computing pairwise distances (subsample)...\", flush=True)\n    sub = np.random.RandomState(0).choice(n2, min(3000, n2), replace=False)\n    D = emb[sub] @ emb.T            # cosine similarity (n_sub, n2)\n    sim = D                          # higher = more similar\n    # For each sampled window, count how many OTHERS are above threshold similarity\n    near_dup_counts = (sim > (1 - args.threshold)).sum(axis=1) - 1  # exclude self\n    frac_has_dup = (near_dup_counts > 0).mean()\n    print(f\"sampled {len(sub)} windows; frac with >=1 near-duplicate video: {frac_has_dup:.3f}\")\n    print(f\"max near-duplicates for one window: {near_dup_counts.max()}\")\n\n    # Are near-duplicates from DIFFERENT sensors (cross-sensor) or same?\n    cross = 0\n    same_sensor = 0\n    for i, si in enumerate(sub):\n        sims = sim[i]\n        near = np.where(sims > (1 - args.threshold))[0]\n        near = near[near != si]\n        if len(near) == 0:\n            continue\n        for j in near:\n            if sensor_names[j] != sensor_names[si]:\n                cross += 1\n            else:\n                same_sensor += 1\n    print(f\"near-duplicate pairs: cross-sensor={cross} same-sensor={same_sensor}\")\n    if cross > same_sensor:\n        print(\"STRONG SIGNAL: cross-sensor duplicate windows exist -> shared temporal moments!\")\n    else:\n        print(\"Mostly same-sensor or no strong cross-sensor duplicate structure.\")\n\n    # Save a readable report (this notebook is a DIAGNOSTIC - do NOT submit it).\n    report = {\n        \"n_test\": len(ids),\n        \"subjects\": np.unique(sbj).tolist(),\n        \"sensor_counts\": pd.Series(sensor_names).value_counts().to_dict(),\n        \"frac_with_dup\": float(frac_has_dup),\n        \"max_dups\": int(near_dup_counts.max()),\n        \"cross_sensor_pairs\": int(cross),\n        \"same_sensor_pairs\": int(same_sensor),\n    }\n    import json\n    from pathlib import Path\n    rep = Path(\"/kaggle/working/dup_report.json\")\n    rep.write_text(json.dumps(report, indent=2))\n    print(\"report -> /kaggle/working/dup_report.json\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_propagate": "\"\"\"Cross-sensor prediction propagation (multi-threshold).\n\nThe test contains ~243 cross-sensor duplicate moments (shared temporal moments\nacross sensors). We train the champion LightGBM, predict all test windows, find\ncross-sensor near-duplicate groups via video similarity, and propagate the\nbest prediction within each group. Supports evaluating MULTIPLE thresholds in\none run (writes each as a named candidate + primary to submission.csv).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES\nfrom .feature_data import build_train_features, build_test_features\n\n\ndef _train_champion(Ftr, y, cfg, n_est, lr, leaves, seed):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(Ftr, y, sample_weight=sw)\n    return m\n\n\ndef _propagate(probs, sim, sensor_names, threshold, agg):\n    \"\"\"Apply propagation for one threshold. Returns propagated probs copy.\"\"\"\n    n = sim.shape[0]\n    adj = sim > (1 - threshold)\n    np.fill_diagonal(adj, False)\n    visited = np.zeros(n, dtype=bool)\n    p_prop = probs.copy()\n    propagated = 0\n    for i in range(n):\n        if visited[i]:\n            continue\n        comp = [i]\n        visited[i] = True\n        stack = [i]\n        while stack:\n            u = stack.pop()\n            nb = np.where(adj[u])[0]\n            for vv in nb:\n                if not visited[vv]:\n                    visited[vv] = True\n                    comp.append(vv)\n                    stack.append(vv)\n        if len(comp) > 1 and len(set(sensor_names[comp])) >= 2:\n            p = p_prop[comp]                       # (g,19)\n            if agg == \"softvote\":\n                best = int(p.sum(axis=0).argmax())\n            else:  # maxconf\n                preds = p.argmax(axis=1)\n                conf = p.max(axis=1)\n                best = preds[int(conf.argmax())]\n            p_prop[comp] = 0.0\n            p_prop[comp, best] = 1.0\n            propagated += len(comp)\n    return p_prop, propagated\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--dup-threshold\", type=float, default=0.02)\n    ap.add_argument(\"--agg\", type=str, default=\"maxconf\",\n                    choices=[\"maxconf\", \"softvote\"])\n    ap.add_argument(\"--thresholds\", type=str, default=None,\n                    help=\"comma-separated thresholds to evaluate in one run\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n\n    model = _train_champion(Ftr, y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, cfg.reservoir.seed)\n    print(\"champion trained\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = np.clip(probs, 1e-9, None)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    # Align probs to id order (build_test_features returns meta order == id order).\n    order = np.argsort(ids)\n    probs = probs[order]\n    sorted_ids = np.sort(ids)\n\n    from .data import build_test_dataset\n    _, video, _, _, _ = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n\n    # Precompute video cosine similarity matrix once (shared across thresholds).\n    v = np.asarray(video, dtype=np.float64)\n    mean = v.mean(axis=1)\n    std = v.std(axis=1)\n    emb = np.concatenate([mean, std], axis=1)\n    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    sim = emb @ emb.T\n    print(f\"similarity matrix: {sim.shape}\", flush=True)\n\n    thresh_list = [float(t) for t in args.thresholds.split(\",\")] if args.thresholds else [args.dup_threshold]\n    from .inference import write_submission\n    out_dir = Path(\"/kaggle/working\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    for thr in thresh_list:\n        p_prop, propagated = _propagate(probs, sim, sensor_names, thr, args.agg)\n        print(f\"thr={thr}: propagated {propagated} windows\", flush=True)\n        final = p_prop.copy()\n        final[:, 0] *= np.exp(args.null_bias)\n        final = final / final.sum(1, keepdims=True)\n        fname = \"submission.csv\" if abs(thr - args.dup_threshold) < 1e-9 else f\"submission_prop_{thr:.3f}.csv\"\n        sub = out_dir / fname\n        write_submission(sorted_ids, final, cfg.paths.sample_submission, sub)\n        print(f\"wrote -> {sub}\", flush=True)\n\n    print(\"done\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "diagnose_groups": "\"\"\"Group-coverage diagnostic: can we recover the ~3,050 latent 4-sensor moments?\n\nUses the FULL 15x768 video as a join key (ChatGPT's #1 priority). Reports the\n2/3/4-sensor group size distribution and coverage. NO submission - read-only.\nThis decides whether building full group-fusion is worth it.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _video_fingerprint(v, pca=None):\n    \"\"\"Full-sequence fingerprint of a (15,768) window -> compact vector.\n\n    mean, std, first, last frame + frame-difference mean + optional PCA on\n    the flattened 15*768 (compressed by a random projection).\n    \"\"\"\n    v = np.asarray(v, dtype=np.float64)\n    mean = v.mean(axis=0)                  # (768,)\n    std = v.std(axis=0)\n    first = v[0]\n    last = v[-1]\n    d = np.diff(v, axis=0).mean(axis=0)    # (768,)\n    feats = np.concatenate([mean, std, first, last, d])  # (3840,)\n    if pca is not None:\n        feats = (feats - pca[0]) @ pca[1]\n    return feats\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--threshold\", type=float, default=0.03)\n    ap.add_argument(\"--subsample\", type=int, default=12234)\n    ap.add_argument(\"--pca-components\", type=int, default=128)\n    args = ap.parse_args()\n\n    from .config import Config\n    from .data import build_test_dataset\n    cfg = Config()\n\n    print(\"loading test data...\", flush=True)\n    _, video, sensor_ids, ids, sbj = build_test_dataset(cfg.paths)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    sensor_names = meta[\"sensor_location\"].to_numpy()\n    subj = meta[\"sbj_id\"].to_numpy()\n    N = len(ids)\n    print(f\"test: {N} windows, subjects={np.unique(subj).tolist()}\", flush=True)\n\n    v = np.asarray(video, dtype=np.float64)[:args.subsample]\n\n    # Build fingerprints (raw 3840-dim) then fit a random projection for compactness.\n    print(\"computing video fingerprints...\", flush=True)\n    feats_raw = np.stack([_video_fingerprint(v[i]) for i in range(len(v))])\n    # center + random projection (deterministic)\n    center = feats_raw.mean(axis=0)\n    rng = np.random.RandomState(0)\n    proj = rng.randn(feats_raw.shape[1], args.pca_components) / np.sqrt(feats_raw.shape[1])\n    feats = (feats_raw - center) @ proj\n    feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-9)\n    print(f\"fingerprints: {feats.shape}\", flush=True)\n\n    # Nearest-neighbor cross-sensor matching within same subject.\n    # For each window, find windows of a DIFFERENT sensor, same subject, that are\n    # near-identical (candidate same-moment views).\n    print(\"matching cross-sensor windows (same subject)...\", flush=True)\n    sim = feats @ feats.T          # (N,N)\n    n = len(feats)\n    groups = []\n    assigned = np.zeros(n, dtype=bool)\n    for i in range(n):\n        if assigned[i]:\n            continue\n        # candidate same-moment: same subject, diff sensor, above threshold\n        cand = np.where(\n            (subj == subj[i]) & (sensor_names != sensor_names[i]) & (sim[i] > (1 - args.threshold))\n        )[0]\n        cand = cand[cand != i]\n        if len(cand) == 0:\n            continue\n        # build group: i + all candidates (union, then dedupe by scanning)\n        member = {int(i)}\n        for c in cand:\n            member.add(int(c))\n        # expand: any member's own matches add to group\n        changed = True\n        while changed:\n            changed = False\n            for m in list(member):\n                mcand = np.where(\n                    (subj == subj[m]) & (sensor_names != sensor_names[m]) & (sim[m] > (1 - args.threshold))\n                )[0]\n                for c in mcand:\n                    if c not in member:\n                        member.add(int(c)); changed = True\n        member = sorted(member)\n        if len(member) >= 2:\n            for m in member:\n                assigned[m] = True\n            groups.append(member)\n\n    # Stats\n    sizes = [len(g) for g in groups]\n    sensors_per_group = [len(set(sensor_names[g])) for g in groups]\n    print(f\"\\n=== GROUP RECOVERY (threshold={args.threshold}) ===\")\n    print(f\"total groups: {len(groups)}\")\n    print(f\"windows in groups: {sum(sizes)} / {n}\")\n    print(f\"coverage: {sum(sizes)/n:.3f}\")\n    print(f\"group sizes: 2-sensor={sum(1 for s in sizes if s==2)}, \"\n          f\"3-sensor={sum(1 for s in sizes if s==3)}, \"\n          f\"4-sensor={sum(1 for s in sizes if s>=4)}\")\n    # distinct sensors per group\n    print(f\"groups with >=2 distinct sensors: {sum(1 for sp in sensors_per_group if sp>=2)}\")\n    print(f\"groups with 3 distinct sensors: {sum(1 for sp in sensors_per_group if sp==3)}\")\n    print(f\"groups with 4 distinct sensors: {sum(1 for sp in sensors_per_group if sp>=4)}\")\n\n    import json\n    from pathlib import Path\n    rep = {\n        \"threshold\": args.threshold,\n        \"total_groups\": len(groups),\n        \"windows_in_groups\": int(sum(sizes)),\n        \"coverage\": float(sum(sizes)/n),\n        \"size2\": int(sum(1 for s in sizes if s==2)),\n        \"size3\": int(sum(1 for s in sizes if s==3)),\n        \"size4plus\": int(sum(1 for s in sizes if s>=4)),\n        \"sens3\": int(sum(1 for sp in sensors_per_group if sp==3)),\n        \"sens4\": int(sum(1 for sp in sensors_per_group if sp>=4)),\n    }\n    Path(\"/kaggle/working/group_report.json\").write_text(json.dumps(rep, indent=2))\n    print(\"report -> /kaggle/working/group_report.json\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_recording_select": "\"\"\"Recording-level domain selection (explains 'more data hurts' + CV-uselessness).\n\nHypothesis: the test domain corresponds to a specific subset of training\nRECORDINGS (session/location conditions). Adding more training recordings that\nare OFF-domain hurts (0.657 -> 0.586). We select the train recordings whose\nfeature distribution best matches each test subject, then train the champion on\nonly those recordings.\n\nSteps:\n1. Build per-window features, grouped by recording.\n2. Build test features, grouped by subject.\n3. For each test subject, find the K closest train recordings (mean feature dist).\n4. Train champion on the union of selected recordings.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import (\n    build_train_features, build_test_features,\n)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--k-recordings\", type=int, default=14,\n                    help=\"number of closest train recordings to keep per test subject\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features (per-sensor views)...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=None,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    n_view = Ftr.shape[0]\n    print(f\"all train views={n_view}\", flush=True)\n\n    # Build per-recording feature means. We need recording labels per view.\n    from .data import build_dataset_windows, load_train_recording, InertialNormalizer\n    windows = build_dataset_windows(cfg.paths, args.stride)\n    normalizer = InertialNormalizer.fit(cfg.paths)\n    grp, rec_names = _recording_groups(windows, cfg.paths, normalizer)\n    rec_id = {r: i for i, r in enumerate(rec_names)}\n\n    # Per-recording mean of full feature vector (per sensor view).\n    rec_means = {}\n    for r in rec_names:\n        idx = np.where(grp == rec_id[r])[0]\n        rec_means[r] = Ftr[idx].mean(axis=0)\n    rec_list = list(rec_names)\n\n    # Test features grouped by subject.\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n    meta = pd.read_csv(cfg.paths.test_meta)\n    test_subj = meta[\"sbj_id\"].to_numpy()\n\n    # For each test subject, find K closest train recordings (mean feature distance).\n    selected_recs = set()\n    for subj in np.unique(test_subj):\n        subj_idx = np.where(test_subj == subj)[0]\n        subj_mean = Fte[subj_idx].mean(axis=0)\n        # distance to each recording mean\n        dists = []\n        for r in rec_list:\n            d = np.linalg.norm(rec_means[r] - subj_mean)\n            dists.append((d, r))\n        dists.sort()\n        for _, r in dists[:args.k_recordings]:\n            selected_recs.add(r)\n    print(f\"selected {len(selected_recs)}/{len(rec_list)} recordings for training\", flush=True)\n\n    # Train on selected recordings only.\n    sel_idx = np.where(np.isin(grp, [rec_id[r] for r in selected_recs]))[0]\n    Fs = Ftr[sel_idx]\n    ys = y[sel_idx]\n    print(f\"training views after selection: {Fs.shape}\", flush=True)\n\n    import lightgbm as lgb\n    counts = np.bincount(ys, minlength=N_CLASSES).astype(float) + 1\n    sw = 1.0 / counts[ys]\n    sw = sw / sw.mean()\n    model = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=N_CLASSES,\n        n_estimators=args.n_estimators, learning_rate=args.lr,\n        num_leaves=args.num_leaves, subsample=0.8, colsample_bytree=0.8,\n        reg_lambda=1.0, min_child_samples=30, n_jobs=8,\n        random_state=cfg.reservoir.seed, verbose=-1)\n    model.fit(Fs, ys, sample_weight=sw)\n    print(\"champion trained on selected recordings\", flush=True)\n\n    probs = model.predict_proba(Fte)\n    if probs.shape[1] < N_CLASSES:\n        probs = np.concatenate([probs, np.zeros((len(probs), N_CLASSES - probs.shape[1]))], axis=1)\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\ndef _recording_groups(windows, paths, normalizer):\n    \"\"\"Recording-index per per-sensor view, matching build_train_features order.\n\n    Order: for each recording (in unique order), for each window, for each valid\n    sensor s in 0..3 -> append recording index. Returns (grp (M,), rec_names).\n    \"\"\"\n    from .config import N_SENSORS\n    from .data import load_train_recording, INERTIAL_WINDOW\n    rec_names = list(windows[\"rec\"].unique())\n    rec_id = {r: i for i, r in enumerate(rec_names)}\n    grp = []\n    for rec in rec_names:\n        r = load_train_recording(paths, rec)\n        sub = windows[windows[\"rec\"] == rec]\n        starts = sub[\"inertial_start\"].to_numpy()\n        idx = starts[:, None] + np.arange(INERTIAL_WINDOW)[None, :]\n        w = np.asarray(r.inertial[idx])\n        valid = np.isfinite(w).all(axis=(1, 3))   # (M,4)\n        gid = rec_id[rec]\n        for s in range(N_SENSORS):\n            keep = valid[:, s]\n            grp.extend([gid] * int(keep.sum()))\n    return np.array(grp, dtype=np.int64), rec_names\n\n\nif __name__ == \"__main__\":\n    main()",
    "run_hierarchical": "\"\"\"Hierarchical classification (family -> variant) targeting macro-F1.\n\nMacro-F1 loses points distinguishing similar VARIANTS (jogging vs jogging-arm-\nrotation, push-ups vs complex, sit-ups vs complex, lunges vs complex). We split\nthe 19 classes into 8 families and train:\n  * Family model (8-way) on champion features + video.\n  * Per-family variant models (on features + video) to distinguish within family.\nAt test, predict family, then variant within that family.\n\nThis forces the model to spend capacity on the hard within-family distinctions\nthat drive macro-F1 down.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .config import Config, N_CLASSES, N_SENSORS\nfrom .feature_data import build_train_features, build_test_features\n\n# Family -> list of class ids (0-18)\nFAMILIES = [\n    [0],                 # null\n    [1, 2, 3, 4, 5],     # jogging + variants\n    [6, 7, 8, 9, 10],    # stretching + variants\n    [11, 12],            # push-ups + complex\n    [13, 14],            # sit-ups + complex\n    [16, 17],            # lunges + complex\n    [15],                # burpees\n    [18],                # bench-dips\n]\n# class -> family index\nCLASS_TO_FAM = np.zeros(N_CLASSES, dtype=np.int64)\nfor fi, classes in enumerate(FAMILIES):\n    for c in classes:\n        CLASS_TO_FAM[c] = fi\nN_FAM = len(FAMILIES)\n\n\ndef _train_lgbm(X, y, cfg, n_est, lr, leaves, n_class, seed):\n    import lightgbm as lgb\n    counts = np.bincount(y, minlength=n_class).astype(float) + 1\n    sw = 1.0 / counts[y]\n    sw = sw / sw.mean()\n    m = lgb.LGBMClassifier(\n        objective=\"multiclass\", num_class=n_class,\n        n_estimators=n_est, learning_rate=lr, num_leaves=leaves,\n        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,\n        min_child_samples=30, n_jobs=8, random_state=seed, verbose=-1)\n    m.fit(X, y, sample_weight=sw)\n    return m\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--stride\", type=int, default=25)\n    ap.add_argument(\"--max-per-class\", type=int, default=600)\n    ap.add_argument(\"--n-estimators\", type=int, default=800)\n    ap.add_argument(\"--lr\", type=float, default=0.07)\n    ap.add_argument(\"--num-leaves\", type=int, default=63)\n    ap.add_argument(\"--video-proj-dim\", type=int, default=64)\n    ap.add_argument(\"--null-bias\", type=float, default=0.75)\n    ap.add_argument(\"--video-weight\", type=float, default=0.3,\n                    help=\"weight on video branch probabilities in final blend\")\n    args = ap.parse_args()\n\n    cfg = Config()\n\n    print(\"building train features...\", flush=True)\n    X, y, vid, sensor = build_train_features(\n        cfg.paths, stride=args.stride, max_per_class=args.max_per_class,\n        seed=cfg.reservoir.seed, video_proj_dim=args.video_proj_dim)\n    Ftr = np.concatenate([X, vid, sensor], axis=1)\n    print(f\"train views={Ftr.shape}\", flush=True)\n\n    print(\"building test features...\", flush=True)\n    Xt, ids, sensor_t, vid_t = build_test_features(cfg.paths, args.video_proj_dim)\n    Fte = np.concatenate([Xt, vid_t, sensor_t], axis=1)\n\n    # --- Family model ---\n    fam_y = CLASS_TO_FAM[y]\n    print(\"training family model...\", flush=True)\n    fam_model = _train_lgbm(Ftr, fam_y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, N_FAM, cfg.reservoir.seed)\n    fam_probs = fam_model.predict_proba(Fte)          # (N, N_FAM)\n    if fam_probs.shape[1] < N_FAM:\n        fam_probs = np.concatenate([fam_probs, np.zeros((len(fam_probs), N_FAM - fam_probs.shape[1]))], axis=1)\n\n    # --- Per-family variant models ---\n    # Base class probs: start with family prob spread across family members.\n    N = len(Fte)\n    probs = np.zeros((N, N_CLASSES))\n    for fi, classes in enumerate(FAMILIES):\n        mask = fam_y == fi\n        # only classes actually present in this family's training set\n        present = sorted(set(y[mask].tolist()) & set(classes))\n        if mask.sum() < 100 or len(present) < 2:\n            # fall back to uniform within the family\n            for c in present:\n                probs[:, c] += fam_probs[:, fi] / len(present)\n            continue\n        Xf = Ftr[mask]\n        yf = y[mask]\n        n_var = len(present)\n        local_map = {c: j for j, c in enumerate(present)}\n        yf_local = np.array([local_map[c] for c in yf])\n        vm = _train_lgbm(Xf, yf_local, cfg, args.n_estimators, args.lr,\n                         args.num_leaves, n_var, cfg.reservoir.seed + fi + 1)\n        vp = vm.predict_proba(Fte)\n        if vp.shape[1] < n_var:\n            vp = np.concatenate([vp, np.zeros((N, n_var - vp.shape[1]))], axis=1)\n        # combine: family prob * variant prob within family\n        for j, c in enumerate(present):\n            probs[:, c] += fam_probs[:, fi] * vp[:, j]\n\n    # Ensure rows sum to 1\n    probs = probs / probs.sum(1, keepdims=True)\n\n    # Optional: blend with a plain 19-class champion (video-weighted).\n    if args.video_weight > 0:\n        print(\"training 19-class champion for blend...\", flush=True)\n        champ = _train_lgbm(Ftr, y, cfg, args.n_estimators, args.lr,\n                            args.num_leaves, N_CLASSES, cfg.reservoir.seed + 99)\n        champ_probs = champ.predict_proba(Fte)\n        if champ_probs.shape[1] < N_CLASSES:\n            champ_probs = np.concatenate([champ_probs, np.zeros((N, N_CLASSES - champ_probs.shape[1]))], axis=1)\n        probs = probs * (1 - args.video_weight) + champ_probs * args.video_weight\n        probs = probs / probs.sum(1, keepdims=True)\n\n    probs = probs.copy()\n    probs[:, 0] *= np.exp(args.null_bias)\n    probs = probs / probs.sum(1, keepdims=True)\n\n    from .inference import write_submission\n    sub = Path(\"/kaggle/working/submission.csv\")\n    sub.parent.mkdir(parents=True, exist_ok=True)\n    write_submission(ids, probs, cfg.paths.sample_submission, sub)\n    print(f\"submission written -> {sub}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()",
}
for _m, _c in SRC.items():
    (WROOT / f'{_m}.py').write_text(_c, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
for _m in list(sys.modules):
    if _m == 'wearfusion' or _m.startswith('wearfusion.'):
        del sys.modules[_m]
from wearfusion import *  # noqa
import wearfusion
# ---- Hierarchical classification (family -> variant) ----
# Family model (8-way) + per-family variant models on champion features + video.
# Targets macro-F1 weakness: distinguishing similar variants. Blends with a
# 19-class champion. Run on CPU (~3-4h). Writes submission.csv.
import sys, subprocess
try:
    import lightgbm
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
from wearfusion.run_hierarchical import main as hier_main
sys.argv = ["run_hierarchical", "--stride", "25", "--max-per-class", "600",
            "--n-estimators", "800", "--lr", "0.07", "--num-leaves", "63",
            "--video-weight", "0.3", "--null-bias", "0.75"]
hier_main()
